# Risk Signal Quality (RSQ)

In [2]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ================================================================
# RSQ CODING
# INPUT: Themes.xlsx
# ================================================================

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)

# ================================================================
# 1. FIND PARTICIPANT SHEETS
# ================================================================

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ================================================================
# 2. EXTRACT RSQ RESPONSES
# ================================================================

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        # Find RSQ regardless of capitalization
        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "RSQ":

                construct_position = position
                break

        if construct_position is None:
            continue

        # Everything after RSQ is response/theme information
        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "RSQ",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ================================================================
# 3. CHECK RSQ RESPONSES
# ================================================================

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No RSQ responses were found.

        Check that the construct is written as RSQ
        somewhere in the participant sheets.
        """
    )

print("\nRSQ original responses:")
print(len(original_df))


# ================================================================
# 4. SPLIT INTO RAW THEMES
# ================================================================

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    # Standardize separators
    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "RSQ",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ================================================================
# 5. CLEAN THEMES
# ================================================================

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

# Remove empty
clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

# Remove duplicate theme within participant
clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)


print("\nRSQ cleaned theme observations:")
print(len(clean_df))


# ================================================================
# 6. DISPLAY ALL UNIQUE RSQ RAW THEMES
# ================================================================

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(drop=True)
)

print("\n==============================")
print("RSQ RAW THEMES")
print("==============================")

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ================================================================
# 7. RSQ NORMALIZATION DICTIONARY
# ================================================================
#
# IMPORTANT:
# Do NOT invent conceptual mergers before seeing your actual
# RSQ themes.
#
# First, exact duplicates / capitalization variants are handled
# automatically.
#
# Add conceptual mappings below AFTER reviewing the raw-theme list.
#
# Format:
#
# "raw theme": "Normalized Theme"
#
# ================================================================

RSQ_NORMALIZATION = {

    # ------------------------------------------------------------
    # ADD YOUR RSQ NORMALIZATION RULES HERE
    # ------------------------------------------------------------
    
    # Example only:
    #
    # "rapid sensing": "Rapid risk sensing",
    # "fast risk sensing": "Rapid risk sensing",
    # "early risk identification": "Early risk identification",

}


# ================================================================
# 8. APPLY NORMALIZATION
# ================================================================

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    # ------------------------------------------------------------
    # If researcher has supplied a mapping
    # ------------------------------------------------------------

    if theme_key in RSQ_NORMALIZATION:

        normalized = RSQ_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        # --------------------------------------------------------
        # If no mapping supplied, keep original theme temporarily
        # --------------------------------------------------------

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "No conceptual merger specified; "
            "retained pending researcher review."
        )


# ================================================================
# 9. MAP NORMALIZED THEMES BACK TO PARTICIPANTS
# ================================================================

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ================================================================
# 10. PARTICIPANT × RSQ THEME MATRIX
# ================================================================

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source[
        "Normalized_Theme"
    ],
    matrix_source[
        "Participant"
    ]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    /
    len(participants)
    *
    100
).round(1)


# ================================================================
# 11. RSQ THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ================================================================
# 12. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Count"
    )
)


# ================================================================
# 13. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    coded_df
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "RSQ_Theme_Count"
]


# ================================================================
# 14. SAVE EXCEL
# ================================================================

output_file = Path(
    "RSQ_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_RSQu",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ================================================================
# 15. FINAL OUTPUT
# ================================================================

print("\n")
print("=" * 60)
print("RSQ CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique RSQ raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized RSQ themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")
print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")
print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

RSQ original responses:
26

RSQ cleaned theme observations:
106

RSQ RAW THEMES
               Raw_Theme              Theme_Clean                Theme_Key
                Accuracy                 Accuracy                 accuracy
       action assessment        action assessment        action assessment
           actionability            actionability            actionability
  Anomaly discrimination   Anomaly discrimination   anomaly discrimination
attention prioritization attention prioritization attention prioritization
        Change detection         Change detection         change detection
                 clarity                  clarity                  clarity
                 Clarity                  

In [13]:
# ================================================================
# COMPLETE RSQ QUALITATIVE ANALYSIS
# ================================================================
#
# INPUT:
#     RSQ_Coding_20260825_131519.xlsx
#
# OUTPUT:
#     RSQ_FINAL_QUALITATIVE_ANALYSIS_20260825_134232.xlsx
#
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

input_file = Path("RSQ_Coding_20260825_131519.xlsx")

if not input_file.exists():

    possible_files = list(Path(".").glob("*RSQ*.xlsx"))

    if len(possible_files) == 0:
        raise FileNotFoundError(
            "\nNo RSQ Excel file was found.\n"
            "Please put RSQ_Complete_Qualitative_Coding.xlsx "
            "in the same folder as your notebook."
        )

    elif len(possible_files) == 1:
        input_file = possible_files[0]

    else:
        print("Multiple RSQ files were found:")
        for i, f in enumerate(possible_files):
            print(f"{i}: {f.name}")

        raise ValueError(
            "\nMore than one RSQ Excel file was found. "
            "Keep only the correct completed RSQ workbook."
        )


print("=" * 70)
print("RSQ QUALITATIVE ANALYSIS")
print("=" * 70)

print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")
for sheet in excel.sheet_names:
    print(" -", sheet)


# ================================================================
# 3. FIND REQUIRED SHEETS
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    # More flexible matching
    for sheet in excel.sheet_names:

        clean_sheet = (
            sheet.lower()
            .replace(" ", "")
            .replace("_", "")
        )

        for name in possible_names:

            clean_name = (
                name.lower()
                .replace(" ", "")
                .replace("_", "")
            )

            if clean_name in clean_sheet:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_RSQ",
    "01_Original",
    "Original_RSQ",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_RSQ",
    "02_Raw",
    "Raw_RSQ",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_RSQ",
    "03_Cleaned",
    "Cleaned_RSQ",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")

print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


if any(
    x is None
    for x in [
        original_sheet,
        raw_sheet,
        clean_sheet,
        coding_sheet
    ]
):

    raise ValueError(
        "\nCould not identify all required sheets.\n"
        "Check the sheet names printed above."
    )


# ================================================================
# 4. READ DATA
# ================================================================

original_df = pd.read_excel(
    input_file,
    sheet_name=original_sheet
)

raw_df = pd.read_excel(
    input_file,
    sheet_name=raw_sheet
)

clean_df = pd.read_excel(
    input_file,
    sheet_name=clean_sheet
)

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    df.columns = [
        str(c).strip()
        for c in df.columns
    ]


# ================================================================
# 6. CHECK REQUIRED CODING COLUMNS
# ================================================================

required_columns = [
    "Theme_Key",
    "Raw_Theme",
    "Normalized_Theme",
    "Decision"
]

missing = [
    c
    for c in required_columns
    if c not in coding_df.columns
]

if missing:

    raise ValueError(
        "\nThe following columns are missing from the "
        "Normalized Coding sheet:\n"
        + str(missing)
        + "\n\nAvailable columns are:\n"
        + str(list(coding_df.columns))
    )


# ================================================================
# 7. IDENTIFY PARTICIPANT COLUMN
# ================================================================

participant_candidates = [
    "Participant",
    "participant",
    "Participant_ID",
    "Participant ID"
]

participant_column = None

for c in participant_candidates:

    if c in clean_df.columns:
        participant_column = c
        break

if participant_column is None:

    raise ValueError(
        "\nParticipant column was not found in the "
        "Cleaned RSQ sheet.\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


# Standardize to Participant

if participant_column != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_column:
            "Participant"
        }
    )


# ================================================================
# 8. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        ["nan", "None", ""],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 9. CLEAN CLEANED-THEME DATA
# ================================================================

if "Theme_Key" not in clean_df.columns:

    raise ValueError(
        "\nTheme_Key is missing from the Cleaned RSQ sheet.\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# ================================================================
# 10. IDENTIFY PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df["Participant"]
    .dropna()
    .astype(str)
    .unique()
)

print("\nParticipants identified:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 11. MERGE CLEANED THEMES WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 12. CHECK UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df["Normalized_Theme"].isna()
].copy()


if len(missing_mapping) > 0:

    print("\nWARNING:")
    print(
        "Unmapped RSQ themes:",
        len(missing_mapping)
    )

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ].to_string(index=False)
    )

else:

    print(
        "\nAll RSQ cleaned themes have normalized mappings."
    )


# ================================================================
# 13. PARTICIPANT × NORMALIZED THEME DATA
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
    .copy()
)


# ================================================================
# 14. CREATE PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme["Normalized_Theme"],
    participant_theme["Participant"]
)


# Try to preserve P01-P26 ordering

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing_participants = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other_participants = [
    p
    for p in matrix.columns
    if p not in existing_participants
]

matrix = matrix.reindex(
    columns=existing_participants + other_participants,
    fill_value=0
)


matrix = matrix.reset_index()


# ================================================================
# 15. CALCULATE FREQUENCY
# ================================================================

matrix["Frequency"] = matrix[
    existing_participants + other_participants
].sum(axis=1)


total_participants = len(participants)

matrix["Percentage"] = (
    matrix["Frequency"]
    / total_participants
    * 100
).round(1)


# ================================================================
# 16. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    by=[
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


matrix.insert(
    0,
    "Rank",
    range(1, len(matrix) + 1)
)


# ================================================================
# 17. THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_RSQ_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 18. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ].apply(prevalence_category)
)


# ================================================================
# 19. CODING AUDIT
# ================================================================

coding_audit = coding_df.copy()


theme_participant_counts = (
    participant_theme
    .groupby("Normalized_Theme")
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_audit.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit["Experts_Mentioning"] = (
    coding_audit["Experts_Mentioning"]
    .fillna(0)
    .astype(int)
)


coding_audit["Percentage_of_Experts"] = (
    coding_audit["Experts_Mentioning"]
    / total_participants
    * 100
).round(1)


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 20. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


decision_summary["Percentage"] = (
    decision_summary[
        "Number_of_Raw_Themes"
    ]
    / len(coding_df)
    * 100
).round(1)


# ================================================================
# 21. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=[
            "Normalized_Theme"
        ]
    )
    .groupby("Normalized_Theme")
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Percentage_of_Experts"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 22. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_Themes"
]


# ================================================================
# 23. CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": ["RSQ"],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df[
            "Theme_Key"
        ].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 24. FINAL RSQ EVIDENCE TABLE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_RSQ_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 25. QUALITY CHECKS
# ================================================================

quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme combinations",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        total_participants,

        len(original_df),

        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        (
            len(coded_df)
            -
            len(participant_theme)
        ),

        (
            coding_df["Decision"]
            == "Keep"
        ).sum(),

        (
            coding_df["Decision"]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants >= 1
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"
    ]
})


# ================================================================
# 26. SAVE OUTPUT
# ================================================================

output_file = Path(
    "RSQ_FINAL_QUALITATIVE_ANALYSIS_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_RSQ_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 27. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("RSQ ANALYSIS COMPLETED")
print("=" * 70)

print("\nParticipants:", total_participants)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df[
        "Theme_Key"
    ].nunique()
)

print(
    "Final normalized RSQ themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df["Decision"]
        == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df["Decision"]
        == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 28. DISPLAY FINAL RSQ THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL RSQ THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_RSQ_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(output_file.resolve())

print(
    "\nComplete RSQ qualitative analysis workbook created successfully."
)

RSQ QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\RSQ_Coding_20260825_131519.xlsx

Sheets found:
 - 01_Original_RSQu
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_RSQu
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Participants identified:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

All RSQ cleaned themes have normalized mappings.


RSQ ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 106
Cleaned theme observations: 106
Unique raw themes: 53
Final normalized RSQ themes: 62
Keep decisions: 62
Merge decisions:

In [28]:
# ================================================================
# RSQ — COMPLETE QUALITATIVE CODING WORKFLOW
# ================================================================
# INPUT:
#   RSQ_FINAL_QUALITATIVE_ANALYSIS_20260821_095315.xlsx
#
# OUTPUT:
#   RSQ_COMPLETE_QUALITATIVE_CODING_YYYYMMDD_HHMMSS.xlsx
#
# Creates:
#   01_Original_RSQ
#   02_Raw_Themes
#   03_Cleaned_Themes
#   04_Coding_Audit
#   05_Participant_Matrix
#   06_Theme_Summary
#   07_Raw_Theme_Summary
#   08_Decision_Summary
#   09_Participant_Coverage
#   10_Unmapped_Check
#   11_Decision_Check
#   12_Quality_Checks
#   13_Final_RSQ_Evidence
#   14_RSQ_Dimension_Summary
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

target_name = "RSQ_FINAL_QUALITATIVE_ANALYSIS_20260825_134232"

possible_files = []

for ext in [".xlsx", ".xlsm", ".xls"]:
    possible_files.extend(
        Path(".").glob(target_name + ext)
    )

if len(possible_files) == 0:

    # Also search current directory recursively
    for ext in [".xlsx", ".xlsm", ".xls"]:
        possible_files.extend(
            Path(".").rglob(target_name + ext)
        )

if len(possible_files) == 0:

    raise FileNotFoundError(
        "\nCould not find:\n"
        f"{target_name}.xlsx\n\n"
        "Put the RSQ Excel file in the same folder as "
        "your Python/Jupyter working directory."
    )

INPUT_FILE = possible_files[0]

print("=" * 75)
print("RSQ COMPLETE QUALITATIVE CODING")
print("=" * 75)

print("\nInput file found:")
print(INPUT_FILE.resolve())


# ================================================================
# 2. READ EXCEL WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. HELPER FUNCTIONS
# ================================================================

def normalize_col_name(x):

    x = str(x).strip()

    x = x.replace(" ", "_")
    x = x.replace("-", "_")

    return x


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def find_column(df, candidates):

    normalized = {
        normalize_col_name(c).lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        key = normalize_col_name(candidate).lower()

        if key in normalized:

            return normalized[key]

    # Fuzzy fallback
    for col in df.columns:

        col_clean = normalize_col_name(col).lower()

        for candidate in candidates:

            candidate_clean = (
                normalize_col_name(candidate)
                .lower()
            )

            if (
                candidate_clean in col_clean
                or col_clean in candidate_clean
            ):

                return col

    return None


def safe_sheet_name(name):

    # Excel sheet names cannot exceed 31 characters
    return name[:31]


# ================================================================
# 4. FIND RAW THEME SHEET
# ================================================================

raw_sheet = None

for s in xls.sheet_names:

    sl = s.lower()

    if "raw" in sl and "theme" in sl:
        raw_sheet = s
        break


if raw_sheet is None:

    # fallback
    for s in xls.sheet_names:

        if "raw" in s.lower():
            raw_sheet = s
            break


if raw_sheet is None:

    raise ValueError(
        "Could not find the Raw Themes sheet."
    )


print("\nRaw theme sheet:")
print(raw_sheet)


# ================================================================
# 5. READ RAW THEMES
# ================================================================

raw = pd.read_excel(
    INPUT_FILE,
    sheet_name=raw_sheet
)

raw.columns = [
    normalize_col_name(c)
    for c in raw.columns
]


print("\nRaw theme columns:")
print(list(raw.columns))


# ================================================================
# 6. IDENTIFY REQUIRED COLUMNS
# ================================================================

participant_col = find_column(
    raw,
    [
        "Participant",
        "Participant_ID",
        "ParticipantID",
        "Expert",
        "Expert_ID"
    ]
)

construct_col = find_column(
    raw,
    [
        "Construct",
        "Construct_Name"
    ]
)

theme_col = find_column(
    raw,
    [
        "Raw_Theme",
        "RawTheme",
        "Theme",
        "Raw_Theme_Name"
    ]
)


if participant_col is None:

    raise ValueError(
        "\nParticipant column not found.\n"
        f"Available columns: {list(raw.columns)}"
    )


if construct_col is None:

    raise ValueError(
        "\nConstruct column not found.\n"
        f"Available columns: {list(raw.columns)}"
    )


if theme_col is None:

    raise ValueError(
        "\nRaw Theme column not found.\n"
        f"Available columns: {list(raw.columns)}"
    )


# ================================================================
# 7. EXTRACT RSQ
# ================================================================

raw["__Construct_Clean"] = (
    raw[construct_col]
    .astype(str)
    .str.strip()
    .str.upper()
)

rsq = raw[
    raw["__Construct_Clean"] == "RSQ"
].copy()


if len(rsq) == 0:

    raise ValueError(
        "\nNo RSQ rows were found.\n\n"
        "Check the Construct column and confirm "
        "that RSQ is used as the construct name."
    )


rsq = rsq.rename(
    columns={
        participant_col: "Participant",
        construct_col: "Construct",
        theme_col: "Raw_Theme"
    }
)


rsq["Participant"] = (
    rsq["Participant"]
    .apply(clean_text)
)


rsq["Construct"] = "RSQ"


rsq["Raw_Theme"] = (
    rsq["Raw_Theme"]
    .apply(clean_text)
)


# Remove empty themes
rsq = rsq[
    rsq["Raw_Theme"] != ""
].copy()


# ================================================================
# 8. STANDARDIZE PARTICIPANT IDS
# ================================================================

def standardize_participant(x):

    x = str(x).strip()

    match = re.search(
        r"(\d+)",
        x
    )

    if match:

        number = int(match.group(1))

        return f"P{number:02d}"

    return x


rsq["Participant"] = (
    rsq["Participant"]
    .apply(standardize_participant)
)


# ================================================================
# 9. CREATE THEME KEY
# ================================================================

rsq["Theme_Key"] = (

    rsq["Raw_Theme"]

    .str.lower()

    .str.strip()

)


# ================================================================
# 10. FIND CODING AUDIT
# ================================================================

audit_sheet = None

for s in xls.sheet_names:

    sl = s.lower()

    if "audit" in sl:

        audit_sheet = s
        break


if audit_sheet is None:

    raise ValueError(
        "Could not find the Coding Audit sheet."
    )


print("\nCoding audit sheet:")
print(audit_sheet)


audit = pd.read_excel(
    INPUT_FILE,
    sheet_name=audit_sheet
)


audit.columns = [
    normalize_col_name(c)
    for c in audit.columns
]


print("\nCoding audit columns:")
print(list(audit.columns))


# ================================================================
# 11. FIND AUDIT COLUMNS
# ================================================================

audit_raw = find_column(
    audit,
    [
        "Raw_Theme",
        "RawTheme",
        "Theme"
    ]
)

audit_normalized = find_column(
    audit,
    [
        "Normalized_Theme",
        "NormalizedTheme",
        "Normalized_Theme_Name",
        "Normalized Theme"
    ]
)

audit_decision = find_column(
    audit,
    [
        "Decision",
        "Coding_Decision"
    ]
)

audit_reason = find_column(
    audit,
    [
        "Reason",
        "Decision_Reason",
        "Rationale"
    ]
)


if audit_raw is None:

    raise ValueError(
        "Raw Theme column not found in Coding Audit."
    )


if audit_normalized is None:

    raise ValueError(
        "Normalized Theme column not found in Coding Audit."
    )


if audit_decision is None:

    raise ValueError(
        "Decision column not found in Coding Audit."
    )


# ================================================================
# 12. STANDARDIZE AUDIT
# ================================================================

audit2 = pd.DataFrame()

audit2["Raw_Theme"] = (
    audit[audit_raw]
    .apply(clean_text)
)

audit2["Theme_Key"] = (
    audit2["Raw_Theme"]
    .str.lower()
    .str.strip()
)

audit2["Normalized_Theme"] = (
    audit[audit_normalized]
    .apply(clean_text)
)

audit2["Decision"] = (
    audit[audit_decision]
    .apply(clean_text)
    .str.title()
)

if audit_reason is not None:

    audit2["Reason"] = (
        audit[audit_reason]
        .apply(clean_text)
    )

else:

    audit2["Reason"] = ""


# ================================================================
# 13. REMOVE DUPLICATE AUDIT ENTRIES
# ================================================================

audit2 = (
    audit2
    .drop_duplicates(
        subset=["Theme_Key"],
        keep="last"
    )
)


# ================================================================
# 14. MERGE CODING AUDIT WITH RSQ
# ================================================================

coded = rsq.merge(

    audit2[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision",
            "Reason"
        ]
    ],

    on="Theme_Key",

    how="left"

)


# ================================================================
# 15. CHECK UNMAPPED THEMES
# ================================================================

coded["Normalized_Theme"] = (
    coded["Normalized_Theme"]
    .apply(clean_text)
)


coded["Decision"] = (
    coded["Decision"]
    .apply(clean_text)
)


unmapped = coded[
    coded["Normalized_Theme"] == ""
].copy()


missing_decision = coded[
    coded["Decision"] == ""
].copy()


print("\n")
print("-" * 75)
print("CODING CHECK")
print("-" * 75)

print(
    "Total RSQ raw-theme observations:",
    len(coded)
)

print(
    "Unmapped observations:",
    len(unmapped)
)

print(
    "Missing decisions:",
    len(missing_decision)
)


# ================================================================
# 16. CREATE CLEANED THEMES
# ================================================================

cleaned_themes = coded[
    [
        "Participant",
        "Construct",
        "Raw_Theme",
        "Theme_Key"
    ]
].copy()


# ================================================================
# 17. CODING AUDIT
# ================================================================

coding_audit = coded[
    [
        "Participant",
        "Construct",
        "Raw_Theme",
        "Theme_Key",
        "Normalized_Theme",
        "Decision",
        "Reason"
    ]
].copy()


# ================================================================
# 18. PARTICIPANT LIST
# ================================================================

participants_detected = sorted(
    coded["Participant"]
    .dropna()
    .unique()
)


# ================================================================
# 19. PARTICIPANT × NORMALIZED THEME MATRIX
# ================================================================

valid_coded = coded[
    coded["Normalized_Theme"] != ""
].copy()


# Remove duplicate mention of the same normalized theme
# by the same participant

valid_coded_unique = (
    valid_coded
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)


participant_matrix = pd.crosstab(

    valid_coded_unique[
        "Normalized_Theme"
    ],

    valid_coded_unique[
        "Participant"
    ]

)


participant_matrix = (
    participant_matrix
    .reindex(
        columns=participants_detected,
        fill_value=0
    )
)


# ================================================================
# 20. FREQUENCY AND PREVALENCE
# ================================================================

participant_matrix[
    "Experts_Mentioning"
] = (
    participant_matrix[
        participants_detected
    ]
    .sum(axis=1)
)


n_participants = len(
    participants_detected
)


if n_participants > 0:

    participant_matrix[
        "Expert_Prevalence_%"
    ] = (

        participant_matrix[
            "Experts_Mentioning"
        ]

        /

        n_participants

        *

        100

    ).round(1)

else:

    participant_matrix[
        "Expert_Prevalence_%"
    ] = 0


# ================================================================
# 21. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(x):

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


participant_matrix[
    "Prevalence_Category"
] = (
    participant_matrix[
        "Expert_Prevalence_%"
    ]
    .apply(prevalence_category)
)


# ================================================================
# 22. THEME SUMMARY
# ================================================================

theme_summary = (
    participant_matrix
    .reset_index()
)


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "RSQ_Theme"
    }
)


theme_summary = theme_summary.sort_values(
    [
        "Experts_Mentioning",
        "RSQ_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


theme_summary.insert(
    0,
    "Rank",
    range(
        1,
        len(theme_summary) + 1
    )
)


# ================================================================
# 23. RAW THEME SUMMARY
# ================================================================

raw_theme_summary = (

    coded

    .groupby(
        [
            "Raw_Theme",
            "Normalized_Theme",
            "Decision"
        ],
        dropna=False
    )

    .agg(
        Experts_Mentioning=(
            "Participant",
            "nunique"
        )
    )

    .reset_index()

)


raw_theme_summary[
    "Expert_Prevalence_%"
] = (

    raw_theme_summary[
        "Experts_Mentioning"
    ]

    /

    n_participants

    *

    100

).round(1)


# ================================================================
# 24. DECISION SUMMARY
# ================================================================

decision_summary = (

    coded

    .groupby(
        [
            "Normalized_Theme",
            "Decision"
        ],
        dropna=False
    )

    .agg(
        Number_of_Raw_Themes=(
            "Raw_Theme",
            "nunique"
        )
    )

    .reset_index()

)


# ================================================================
# 25. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (

    valid_coded_unique

    .groupby(
        "Participant"
    )

    ["Normalized_Theme"]

    .nunique()

    .reset_index()

)


participant_coverage = (
    participant_coverage
    .rename(
        columns={
            "Normalized_Theme":
                "Number_of_Normalized_Themes"
        }
    )
)


number_normalized_themes = (
    valid_coded_unique[
        "Normalized_Theme"
    ].nunique()
)


if number_normalized_themes > 0:

    participant_coverage[
        "Coverage_Percentage"
    ] = (

        participant_coverage[
            "Number_of_Normalized_Themes"
        ]

        /

        number_normalized_themes

        *

        100

    ).round(1)

else:

    participant_coverage[
        "Coverage_Percentage"
    ] = 0


# ================================================================
# 26. FINAL RSQ EVIDENCE
# ================================================================

final_rsq_evidence = theme_summary.copy()


final_rsq_evidence = final_rsq_evidence[
    [
        "Rank",
        "RSQ_Theme",
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category"
    ]
]


# ================================================================
# 27. RSQ DIMENSION MAPPING
# ================================================================
#
# IMPORTANT:
# These dimensions follow the RSQ construct definition:
#
# Timeliness
# Accuracy
# Reliability/Credibility
# Clarity/Interpretability
# Actionability
#
# The code attempts to map normalized themes using their wording.
# Themes that cannot be confidently mapped are marked:
# "Review Required"
#
# This prevents the code from silently inventing dimensions.
# ================================================================

def map_rsq_dimension(theme):

    t = str(theme).lower()

    # ------------------------------------------------------------
    # TIMELINESS
    # ------------------------------------------------------------

    if any(
        k in t
        for k in [
            "timely",
            "timeliness",
            "real-time",
            "realtime",
            "speed",
            "rapid",
            "quick",
            "immediate",
            "early",
            "prompt",
            "up-to-date",
            "current"
        ]
    ):
        return "Timeliness"


    # ------------------------------------------------------------
    # ACCURACY
    # ------------------------------------------------------------

    if any(
        k in t
        for k in [
            "accuracy",
            "accurate",
            "correct",
            "precision",
            "error-free",
            "validity",
            "valid"
        ]
    ):
        return "Accuracy"


    # ------------------------------------------------------------
    # RELIABILITY / CREDIBILITY
    # ------------------------------------------------------------

    if any(
        k in t
        for k in [
            "reliab",
            "trust",
            "credible",
            "credibility",
            "consistent",
            "consistency",
            "integrity",
            "verified",
            "verification",
            "authentic"
        ]
    ):
        return "Reliability / Credibility"


    # ------------------------------------------------------------
    # CLARITY / INTERPRETABILITY
    # ------------------------------------------------------------

    if any(
        k in t
        for k in [
            "clear",
            "clarity",
            "understand",
            "interpret",
            "interpretability",
            "transparent",
            "transparency",
            "visibility"
        ]
    ):
        return "Clarity / Interpretability"


    # ------------------------------------------------------------
    # ACTIONABILITY
    # ------------------------------------------------------------

    if any(
        k in t
        for k in [
            "action",
            "actionable",
            "decision",
            "response",
            "respond",
            "useful",
            "usable",
            "operational",
            "support",
            "warning"
        ]
    ):
        return "Actionability"


    # ------------------------------------------------------------
    # UNRESOLVED
    # ------------------------------------------------------------

    return "Review Required"


theme_summary[
    "RSQ_Dimension"
] = (
    theme_summary[
        "RSQ_Theme"
    ]
    .apply(map_rsq_dimension)
)


# ================================================================
# 28. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    theme_summary
    .groupby("RSQ_Dimension")
):

    included_themes = (
        group[
            "RSQ_Theme"
        ]
        .astype(str)
        .tolist()
    )


    experts = int(
        group[
            "Experts_Mentioning"
        ].max()
    )


    # More robust dimension-level prevalence:
    # calculate unique participants mentioning ANY theme
    # belonging to the dimension.

    dimension_themes = set(
        included_themes
    )


    dimension_data = valid_coded_unique[
        valid_coded_unique[
            "Normalized_Theme"
        ].isin(
            dimension_themes
        )
    ]


    dimension_experts = (
        dimension_data[
            "Participant"
        ]
        .nunique()
    )


    dimension_prevalence = (

        dimension_experts

        /

        n_participants

        *

        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "RSQ_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(included_themes),

        "Experts_Mentioning":
            dimension_experts,

        "Expert_Prevalence_%":
            round(
                dimension_prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                dimension_prevalence
            ),

        "Included_RSQ_Themes":
            "; ".join(
                included_themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 29. UNMAPPED CHECK
# ================================================================

if len(unmapped) > 0:

    unmapped_check = unmapped[
        [
            "Participant",
            "Raw_Theme",
            "Theme_Key"
        ]
    ].drop_duplicates()

else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All RSQ raw themes were mapped."
        ]

    })


# ================================================================
# 30. DECISION CHECK
# ================================================================

if len(missing_decision) > 0:

    decision_check = missing_decision[
        [
            "Participant",
            "Raw_Theme",
            "Theme_Key",
            "Normalized_Theme"
        ]
    ].drop_duplicates()

else:

    decision_check = pd.DataFrame({

        "Status": [
            "PASS — All RSQ themes have a coding decision."
        ]

    })


# ================================================================
# 31. QUALITY CHECKS
# ================================================================

quality_checks = pd.DataFrame({

    "Quality_Check": [

        "Input file found",

        "RSQ observations extracted",

        "Participants detected",

        "Raw themes identified",

        "Normalized themes identified",

        "Unmapped themes",

        "Missing decisions",

        "Participant matrix created",

        "Dimension mapping completed"

    ],

    "Result": [

        "PASS",

        len(coded),

        n_participants,

        coded[
            "Raw_Theme"
        ].nunique(),

        coded[
            "Normalized_Theme"
        ].replace(
            "",
            np.nan
        )
        .nunique(),

        len(unmapped),

        len(missing_decision),

        "PASS",

        len(
            dimension_summary
        )

    ],

    "Status": [

        "OK",

        "OK" if len(coded) > 0 else "CHECK",

        "OK" if n_participants > 0 else "CHECK",

        "OK",

        "OK",

        "PASS" if len(unmapped) == 0 else "CHECK",

        "PASS" if len(missing_decision) == 0 else "CHECK",

        "OK",

        "OK"

    ]

})


# ================================================================
# 32. OUTPUT FILE
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

OUTPUT_FILE = Path(
    f"RSQ_COMPLETE_QUALITATIVE_CODING_{timestamp}.xlsx"
)


# ================================================================
# 33. WRITE EXCEL
# ================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:


    # ------------------------------------------------------------
    # 01
    # ------------------------------------------------------------

    rsq.to_excel(
        writer,
        sheet_name="01_Original_RSQ",
        index=False
    )


    # ------------------------------------------------------------
    # 02
    # ------------------------------------------------------------

    rsq[
        [
            "Participant",
            "Construct",
            "Raw_Theme",
            "Theme_Key"
        ]
    ].to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # 03
    # ------------------------------------------------------------

    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # 04
    # ------------------------------------------------------------

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )


    # ------------------------------------------------------------
    # 05
    # ------------------------------------------------------------

    participant_matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix"
    )


    # ------------------------------------------------------------
    # 06
    # ------------------------------------------------------------

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 07
    # ------------------------------------------------------------

    raw_theme_summary.to_excel(
        writer,
        sheet_name="07_Raw_Theme_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 08
    # ------------------------------------------------------------

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 09
    # ------------------------------------------------------------

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    # ------------------------------------------------------------
    # 10
    # ------------------------------------------------------------

    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    # ------------------------------------------------------------
    # 11
    # ------------------------------------------------------------

    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    # ------------------------------------------------------------
    # 12
    # ------------------------------------------------------------

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    # ------------------------------------------------------------
    # 13
    # ------------------------------------------------------------

    final_rsq_evidence.to_excel(
        writer,
        sheet_name="13_Final_RSQ_Evidence",
        index=False
    )


    # ------------------------------------------------------------
    # 14
    # ------------------------------------------------------------

    dimension_summary.to_excel(
        writer,
        sheet_name="14_RSQ_Dimension_Summary",
        index=False
    )


# ================================================================
# 34. FINAL REPORT
# ================================================================

print("\n")
print("=" * 75)
print("RSQ CODING COMPLETED SUCCESSFULLY")
print("=" * 75)

print(
    "\nInput:",
    INPUT_FILE.name
)

print(
    "Output:",
    OUTPUT_FILE.resolve()
)

print("\nRSQ statistics:")

print(
    "Raw observations:",
    len(coded)
)

print(
    "Unique raw themes:",
    coded["Raw_Theme"].nunique()
)

print(
    "Normalized themes:",
    coded[
        "Normalized_Theme"
    ]
    .replace(
        "",
        np.nan
    )
    .nunique()
)

print(
    "Participants:",
    n_participants
)

print(
    "Unmapped themes:",
    len(unmapped)
)

print(
    "Missing decisions:",
    len(missing_decision)
)

print(
    "RSQ dimensions:",
    len(dimension_summary)
)


print("\n")
print("=" * 75)
print("RSQ DIMENSION SUMMARY")
print("=" * 75)

if len(dimension_summary) > 0:

    print(
        dimension_summary.to_string(
            index=False
        )
    )


print("\n")
print("=" * 75)
print("OUTPUT CREATED")
print("=" * 75)

print(
    OUTPUT_FILE.resolve()
)

print("\nYou can now open the output Excel workbook.")

RSQ COMPLETE QUALITATIVE CODING

Input file found:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\RSQ_FINAL_QUALITATIVE_ANALYSIS_20260825_134232.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_RSQ_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Raw theme sheet:
02_Raw_Themes

Raw theme columns:
['Participant', 'Construct', 'Raw_Theme', 'Source_Sheet', 'Source_Row']

Coding audit sheet:
04_Coding_Audit

Coding audit columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason', 'Experts_Mentioning', 'Percentage_of_Experts']


---------------------------------------------------------------------------
CODING CHECK
--------------------------------------------------------

# SCCE

In [3]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ================================================================
# SCCE CODING
# INPUT FILE: Themes.xlsx
# ================================================================

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ================================================================
# 1. FIND PARTICIPANT SHEETS
# ================================================================

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ================================================================
# 2. EXTRACT SCCE RESPONSES
# ================================================================

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        # Find SCCE regardless of capitalization
        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "SCCE":

                construct_position = position
                break

        if construct_position is None:
            continue

        # Everything after SCCE
        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "SCCE",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ================================================================
# 3. CHECK SCCE RESPONSES
# ================================================================

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No SCCE responses were found.

        Check that the construct is written as SCCE
        in the participant sheets.
        """
    )

print("\nSCCE original responses:")
print(len(original_df))


# ================================================================
# 4. SPLIT INTO RAW THEMES
# ================================================================

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "SCCE",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ================================================================
# 5. CLEAN THEMES
# ================================================================

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)


print("\nSCCE cleaned theme observations:")
print(len(clean_df))


# ================================================================
# 6. DISPLAY UNIQUE RAW SCCE THEMES
# ================================================================

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("SCCE RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ================================================================
# 7. SCCE NORMALIZATION DICTIONARY
# ================================================================
#
# IMPORTANT:
# Do not automatically merge themes simply because their words
# look similar.
#
# Add researcher-approved conceptual mappings here after seeing
# the actual SCCE raw themes.
#
# Example format:
#
# "rapid corrective action":
#     "Rapid corrective action",
#
# "quick corrective response":
#     "Rapid corrective action",
#
# ================================================================

SCCE_NORMALIZATION = {

    # ------------------------------------------------------------
    # ADD ACTUAL SCCE MAPPINGS HERE
    # ------------------------------------------------------------

}


# ================================================================
# 8. APPLY NORMALIZATION
# ================================================================

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in SCCE_NORMALIZATION:

        normalized = SCCE_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        # Until conceptual mapping is approved,
        # retain the raw theme rather than inventing
        # a merger.

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ================================================================
# 9. MAP BACK TO PARTICIPANTS
# ================================================================

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ================================================================
# 10. PARTICIPANT × SCCE THEME MATRIX
# ================================================================

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source[
        "Normalized_Theme"
    ],
    matrix_source[
        "Participant"
    ]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ================================================================
# 11. SCCE THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ================================================================
# 12. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Count"
    )
)


# ================================================================
# 13. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    coded_df
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "SCCE_Theme_Count"
]


# ================================================================
# 14. SAVE EXCEL
# ================================================================

output_file = Path(
    "SCCE_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ================================================================
# 15. FINAL REPORT
# ================================================================

print("\n")
print("=" * 60)
print("SCCE CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique SCCE raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized SCCE themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

SCCE original responses:
26

SCCE cleaned theme observations:
94


SCCE RAW THEMES
                Raw_Theme               Theme_Clean                 Theme_Key
        Agreed procedures         Agreed procedures         agreed procedures
          Agreed triggers           Agreed triggers           agreed triggers
       approval reduction        approval reduction        approval reduction
     automated initiation      automated initiation      automated initiation
         automatic action          automatic action          automatic action
      automatic execution       automatic execution       automatic execution
     automatic initiation      automatic initiation      automatic initiation
               

In [14]:
# ================================================================
# COMPLETE SCCE QUALITATIVE ANALYSIS
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path("SCCE_Coding_20260825_131658.xlsx")

if not input_file.exists():

    possible_files = list(Path(".").glob("SCCE_Coding_*.xlsx"))

    if len(possible_files) == 0:
        raise FileNotFoundError(
            "\nSCCE input file was not found.\n"
            "Make sure the Excel file is in the same folder "
            "as your Python notebook."
        )

    elif len(possible_files) == 1:
        input_file = possible_files[0]

    else:
        print("SCCE files found:")
        for f in possible_files:
            print(" -", f.name)

        raise ValueError(
            "\nMore than one SCCE coding file was found. "
            "Please keep the intended file."
        )


print("=" * 70)
print("SCCE QUALITATIVE CODING ANALYSIS")
print("=" * 70)

print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")

for sheet in excel.sheet_names:
    print(" -", sheet)


# ================================================================
# 3. FIND SHEETS FLEXIBLY
# ================================================================

def find_sheet(possible_names):

    # Exact match first
    for name in possible_names:
        if name in excel.sheet_names:
            return name

    # Flexible match
    for sheet in excel.sheet_names:

        clean_sheet = (
            sheet.lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            clean_name = (
                name.lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if clean_name in clean_sheet:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_SCCE",
    "01_Original",
    "Original_SCCE",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_SCCE",
    "02_Raw",
    "Raw_SCCE",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_SCCE",
    "03_Cleaned",
    "Cleaned_SCCE",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 4. READ THE DATA
# ================================================================

if original_sheet:
    original_df = pd.read_excel(
        input_file,
        sheet_name=original_sheet
    )
else:
    original_df = pd.DataFrame()


if raw_sheet:
    raw_df = pd.read_excel(
        input_file,
        sheet_name=raw_sheet
    )
else:
    raw_df = pd.DataFrame()


if clean_sheet:
    clean_df = pd.read_excel(
        input_file,
        sheet_name=clean_sheet
    )
else:
    clean_df = pd.DataFrame()


if coding_sheet:
    coding_df = pd.read_excel(
        input_file,
        sheet_name=coding_sheet
    )
else:
    raise ValueError(
        "\nNormalized Coding sheet could not be found."
    )


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:

        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 6. IDENTIFY CODING COLUMNS
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:

        if candidate in df.columns:
            return candidate

    # Flexible search
    for column in df.columns:

        clean_column = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            clean_candidate = (
                candidate
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if clean_column == clean_candidate:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

raw_theme_col = find_column(
    coding_df,
    [
        "Raw_Theme",
        "Raw Theme",
        "RawTheme"
    ]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    [
        "Decision"
    ]
)


if theme_key_col is None:
    raise ValueError(
        "\nTheme_Key column not found."
    )

if raw_theme_col is None:
    raise ValueError(
        "\nRaw_Theme column not found."
    )

if normalized_col is None:
    raise ValueError(
        "\nNormalized_Theme column not found."
    )

if decision_col is None:
    raise ValueError(
        "\nDecision column not found."
    )


# Rename to standard names

coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. IDENTIFY PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)


if participant_col is None:

    raise ValueError(
        "\nParticipant column was not found in the "
        "Cleaned SCCE sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col:
            "Participant"
        }
    )


# ================================================================
# 8. IDENTIFY THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)


if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key was not found in the Cleaned SCCE sheet."
    )


if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key:
            "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        ["nan", "None", "", "NaN"],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN CLEANED DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df[
        "Participant"
    ]
    .dropna()
    .unique()
)


print("\nParticipants identified:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 12. MERGE CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df[
        "Normalized_Theme"
    ].isna()
].copy()


print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)


if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED THEME
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme[
        "Normalized_Theme"
    ],
    participant_theme[
        "Participant"
    ]
)


# P01-P26 ordering

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)


matrix = matrix.reset_index()


# ================================================================
# 16. FREQUENCY AND PERCENTAGE
# ================================================================

participant_columns = (
    existing + other
)


matrix["Frequency"] = matrix[
    participant_columns
].sum(axis=1)


total_participants = len(participants)


matrix["Percentage"] = (
    matrix["Frequency"]
    / total_participants
    * 100
).round(1)


# ================================================================
# 17. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    by=[
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 18. THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_SCCE_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 19. PREVALENCE
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 20. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby(
        "Normalized_Theme"
    )
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit[
    "Experts_Mentioning"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


coding_audit[
    "Percentage_of_Experts"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 21. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


decision_summary[
    "Percentage"
] = (
    decision_summary[
        "Number_of_Raw_Themes"
    ]
    / len(coding_df)
    * 100
).round(1)


# ================================================================
# 22. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=[
            "Normalized_Theme"
        ]
    )
    .groupby(
        "Normalized_Theme"
    )
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


normalization_summary[
    "Percentage_of_Experts"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 23. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_Themes"
]


# ================================================================
# 24. CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": ["SCCE"],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df[
            "Theme_Key"
        ].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 25. FINAL SCCE EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_SCCE_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 26. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)


quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Final normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme records",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        total_participants,

        len(original_df),

        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum(),

        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"
    ]
})


# ================================================================
# 27. SAVE OUTPUT
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"SCCE_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_SCCE_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 28. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("SCCE ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nParticipants:",
    total_participants
)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df[
        "Theme_Key"
    ].nunique()
)

print(
    "Final normalized SCCE themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df[
            "Decision"
        ] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df[
            "Decision"
        ] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 29. DISPLAY FINAL SCCE THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL SCCE THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_SCCE_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 30. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(
    output_file.resolve()
)

print(
    "\nComplete SCCE qualitative analysis workbook "
    "created successfully."
)

SCCE QUALITATIVE CODING ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\SCCE_Coding_20260825_131658.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Participants identified:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

Number of unmapped themes: 0


SCCE ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 94
Cleaned theme observa

In [32]:
# ================================================================
# SCCE — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
#   SCCE_FINAL_QUALITATIVE_ANALYSIS_20260821_095518.xlsx
#
# OUTPUT:
#   SCCE_FINAL_EVIDENCE_AND_DIMENSIONS_YYYYMMDD_HHMMSS.xlsx
#
# This code uses the completed SCCE qualitative coding workbook.
#
# It creates:
#
# 01_Original_Responses
# 02_Raw_Themes
# 03_Cleaned_Themes
# 04_Coding_Audit
# 05_Participant_Matrix
# 06_Theme_Summary
# 07_Normalization_Summary
# 08_Decision_Summary
# 09_Participant_Coverage
# 10_Unmapped_Check
# 11_Decision_Check
# 12_Quality_Checks
# 13_Final_SCCE_Evidence
# 14_SCCE_Dimension_Summary
# 15_SCCE_Dimension_Themes
#
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. FIND SCCE INPUT FILE
# ================================================================

TARGET = "SCCE_FINAL_QUALITATIVE_ANALYSIS_20260825_135013"

possible_files = []

# Search current directory and subdirectories
for ext in [".xlsx", ".xlsm", ".xls"]:

    possible_files.extend(
        Path(".").rglob(TARGET + ext)
    )

if len(possible_files) == 0:

    raise FileNotFoundError(
        "\n\nSCCE input file was not found.\n"
        f"Expected file:\n{TARGET}.xlsx\n\n"
        "Place the file in the same folder as your notebook/script."
    )


INPUT_FILE = possible_files[0]

print("=" * 80)
print("SCCE FINAL QUALITATIVE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. HELPER FUNCTION
# ================================================================

def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


# ================================================================
# 4. READ ALL EXISTING SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 5. IDENTIFY IMPORTANT SHEETS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():

            return s

    return None


raw_sheet = find_sheet("02_Raw_Themes")
cleaned_sheet = find_sheet("03_Cleaned_Themes")
audit_sheet = find_sheet("04_Coding_Audit")
matrix_sheet = find_sheet("05_Participant_Matrix")
theme_sheet = find_sheet("06_Theme_Summary")
normalization_sheet = find_sheet("07_Normalization_Summary")
decision_sheet = find_sheet("08_Decision_Summary")
coverage_sheet = find_sheet("09_Participant_Coverage")
quality_sheet = find_sheet("12_Quality_Checks")
final_sheet = find_sheet("11_Final_SCCE_Evidence")
unmapped_sheet = find_sheet("13_Unmapped_Themes")


print("\nDetected sheets:")

print("Raw themes:",
      raw_sheet)

print("Coding audit:",
      audit_sheet)

print("Theme summary:",
      theme_sheet)

print("Final SCCE evidence:",
      final_sheet)


# ================================================================
# 6. READ FINAL SCCE EVIDENCE
# ================================================================

if final_sheet is None:

    raise ValueError(
        "11_Final_SCCE_Evidence sheet was not found."
    )


final_evidence = sheets[
    final_sheet
].copy()


print("\nFinal SCCE evidence columns:")

print(
    list(
        final_evidence.columns
    )
)


# ================================================================
# 7. STANDARDIZE FINAL EVIDENCE COLUMNS
# ================================================================

final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# Find theme column

theme_col = None

for c in final_evidence.columns:

    if "Final_SCCE_Theme" in str(c):

        theme_col = c
        break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify SCCE theme column."
    )


final_evidence[
    "Final_SCCE_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE EXPERT PREVALENCE
# ================================================================

if "Experts_Mentioning" not in final_evidence.columns:

    if "Experts Mentioning" in final_evidence.columns:

        final_evidence[
            "Experts_Mentioning"
        ] = final_evidence[
            "Experts Mentioning"
        ]


if "Expert_Prevalence_%" not in final_evidence.columns:

    if "Percentage_of_Experts" in final_evidence.columns:

        final_evidence[
            "Expert_Prevalence_%"
        ] = final_evidence[
            "Percentage_of_Experts"
        ]


# ================================================================
# 9. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(x):

    try:
        x = float(x)
    except:
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(prevalence_category)
    )


# ================================================================
# 10. REMOVE COMPLETELY EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_SCCE_Theme"
    ] != ""
].copy()


# ================================================================
# 11. READ PARTICIPANT MATRIX
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


# ================================================================
# 12. READ CODING AUDIT
# ================================================================

coding_audit = None

if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


# ================================================================
# 13. CREATE PARTICIPANT INFORMATION
# ================================================================

participants = []

if participant_matrix is not None:

    # Participant matrix normally has participants as columns.
    # Remove obvious non-participant columns.

    for c in participant_matrix.columns:

        cstr = str(c)

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(cstr)


# If matrix did not identify participants,
# infer participant count from prevalence.

if len(participants) == 0:

    n_participants = 26

else:

    n_participants = len(participants)


print(
    "\nNumber of participants:",
    n_participants
)


# ================================================================
# 14. CREATE FINAL SCCE EVIDENCE TABLE
# ================================================================

evidence_columns = []

for c in [
    "Final_SCCE_Theme",
    "Number_of_Raw_Themes",
    "Experts_Mentioning",
    "Expert_Prevalence_%",
    "Raw_Themes_Kept",
    "Raw_Themes_Merged",
    "Prevalence_Category"
]:

    if c in final_evidence.columns:

        evidence_columns.append(c)


final_scc_evidence = final_evidence[
    evidence_columns
].copy()


# ================================================================
# 15. SORT FINAL EVIDENCE
# ================================================================

if "Experts_Mentioning" in final_scc_evidence.columns:

    final_scc_evidence = (
        final_scc_evidence
        .sort_values(
            [
                "Experts_Mentioning",
                "Final_SCCE_Theme"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )


final_scc_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_scc_evidence) + 1
    )
)


# ================================================================
# 16. SCCE DIMENSION MAPPING
# ================================================================
#
# The mapping is based specifically on the themes actually present
# in your SCCE qualitative evidence.
#
# Dimensions:
#
# 1. Rule & Trigger Specification
# 2. Automated Execution
# 3. Predefined Response & Procedural Standardization
# 4. Response Speed & Reduced Administrative Delay
# 5. Exception Handling & Human Oversight
#
# "Review Required" is used where the theme does not clearly fit.
#
# ================================================================

def map_scce_dimension(theme):

    t = str(theme).lower().strip()


    # ------------------------------------------------------------
    # 1. RULE & TRIGGER SPECIFICATION
    # ------------------------------------------------------------

    trigger_words = [

        "trigger",
        "triggers",
        "threshold",
        "thresholds",
        "condition",
        "conditions",
        "rule",
        "rules",
        "objective condition",
        "measurable condition",
        "measurable trigger",
        "objective trigger",
        "verified trigger",
        "contractual trigger",
        "agreed trigger",
        "clear condition",
        "clear rule",
        "defined rule",
        "defined trigger",
        "predefined condition",
        "predefined rule",
        "predefined event",
        "condition activation",
        "threshold activation",
        "threshold trigger"

    ]

    if any(
        word in t
        for word in trigger_words
    ):

        return "Rule & Trigger Specification"


    # ------------------------------------------------------------
    # 2. AUTOMATED EXECUTION
    # ------------------------------------------------------------

    automation_words = [

        "automation",
        "automated",
        "automatic",
        "automatically",
        "automatic action",
        "automatic execution",
        "automatic initiation",
        "automated initiation",
        "trigger execution",
        "condition-based execution",
        "routine automation",
        "repetitive automation",
        "rule-based automation"

    ]

    if any(
        word in t
        for word in automation_words
    ):

        return "Automated Execution"


    # ------------------------------------------------------------
    # 3. PREDEFINED RESPONSE & PROCEDURAL STANDARDIZATION
    # ------------------------------------------------------------

    procedure_words = [

        "predefined procedure",
        "predefined procedures",
        "predefined response",
        "predefined responses",
        "predefined action",
        "predefined actions",
        "predefined process",
        "predefined processes",
        "agreed procedure",
        "agreed procedures",
        "predictable procedure",
        "predictable procedures",
        "procedural reliability",
        "defined procedure",
        "defined procedures",
        "rules"

    ]

    if any(
        word in t
        for word in procedure_words
    ):

        return "Predefined Response & Procedural Standardization"


    # ------------------------------------------------------------
    # 4. RESPONSE SPEED & REDUCED ADMINISTRATIVE DELAY
    # ------------------------------------------------------------

    speed_words = [

        "speed",
        "response speed",
        "immediate action",
        "immediate execution",
        "reduced waiting",
        "reduced approval",
        "reduced approvals",
        "approval reduction",
        "reduced manual steps",
        "rapid",
        "quick",
        "fast",
        "delay"

    ]

    if any(
        word in t
        for word in speed_words
    ):

        return "Response Speed & Reduced Administrative Delay"


    # ------------------------------------------------------------
    # 5. EXCEPTION HANDLING & HUMAN OVERSIGHT
    # ------------------------------------------------------------

    human_words = [

        "exception",
        "exceptions",
        "exception handling",
        "exception review",
        "human assessment",
        "human intervention",
        "human judgment",
        "human oversight",
        "managerial intervention",
        "managerial judgment",
        "managerial review",
        "judgment boundary",
        "negotiation boundary",
        "automation boundaries",
        "escalation"

    ]

    if any(
        word in t
        for word in human_words
    ):

        return "Exception Handling & Human Oversight"


    # ------------------------------------------------------------
    # UNRESOLVED
    # ------------------------------------------------------------

    return "Review Required"


final_scc_evidence[
    "SCCE_Dimension"
] = (
    final_scc_evidence[
        "Final_SCCE_Theme"
    ]
    .apply(
        map_scce_dimension
    )
)


# ================================================================
# 17. CREATE DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_scc_evidence
    .groupby(
        "SCCE_Dimension"
    )
):

    themes = (
        group[
            "Final_SCCE_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # Calculate dimension-level expert prevalence.
    #
    # We use the maximum expert count among the themes when the
    # participant matrix is unavailable.
    #

    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        # Identify normalized theme index column

        index_col = (
            participant_matrix.columns[0]
        )

        temp = participant_matrix.copy()

        temp[
            "Theme_TEMP"
        ] = temp[
            index_col
        ].astype(str)


        matched = temp[
            temp[
                "Theme_TEMP"
            ].isin(
                themes
            )
        ]


        if len(matched) > 0:

            participant_values = (
                matched[
                    participants
                ]
                .fillna(0)
                .astype(float)
            )


            experts_mentioning = int(
                (
                    participant_values
                    .sum(axis=0) > 0
                )
                .sum()
            )

        else:

            experts_mentioning = 0

    else:

        experts_mentioning = int(
            group[
                "Experts_Mentioning"
            ].max()
        )


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0
    )


    dimension_rows.append({

        "SCCE_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_SCCE_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 18. SORT DIMENSIONS
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 19. DIMENSION × THEME TABLE
# ================================================================

dimension_themes = final_scc_evidence[
    [
        "SCCE_Dimension",
        "Final_SCCE_Theme",
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category"
    ]
].copy()


dimension_themes = (
    dimension_themes
    .sort_values(
        [
            "SCCE_Dimension",
            "Experts_Mentioning"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ================================================================
# 20. CREATE UNMAPPED CHECK
# ================================================================

unmapped_dimension = final_scc_evidence[
    final_scc_evidence[
        "SCCE_Dimension"
    ] == "Review Required"
].copy()


if len(unmapped_dimension) > 0:

    unmapped_check = unmapped_dimension[
        [
            "Final_SCCE_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ]
    ].copy()

    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )

else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All SCCE themes mapped to a dimension."
        ]

    })


# ================================================================
# 21. DECISION CHECK
# ================================================================

if coding_audit is not None:

    audit_cols = [
        c for c in coding_audit.columns
        if "Decision" in str(c)
    ]

    if len(audit_cols) > 0:

        decision_col = audit_cols[0]

        missing_decisions = coding_audit[
            coding_audit[
                decision_col
            ]
            .isna()
        ]

        if len(missing_decisions) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All SCCE coding decisions are present."
                ]

            })

        else:

            decision_check = (
                missing_decisions
                .copy()
            )

    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })

else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit sheet unavailable."
        ]

    })


# ================================================================
# 22. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final SCCE evidence available",

    "Result":
        "PASS"
        if len(final_scc_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_scc_evidence)} normalized themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Number of participants used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "Normalized SCCE themes",

    "Result":
        len(final_scc_evidence),

    "Details":
        "Themes available for measurement development"

})


quality_rows.append({

    "Quality_Check":
        "Dimension mapping",

    "Result":
        len(dimension_summary),

    "Details":
        "SCCE dimensions generated"

})


quality_rows.append({

    "Quality_Check":
        "Themes requiring manual review",

    "Result":
        len(unmapped_dimension),

    "Details":
        "Review Required dimension"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 23. CREATE NORMALIZATION SUMMARY
# ================================================================

if normalization_sheet is not None:

    normalization_summary = sheets[
        normalization_sheet
    ].copy()

else:

    normalization_summary = pd.DataFrame({

        "Status": [
            "Normalization summary was not available."
        ]

    })


# ================================================================
# 24. CREATE DECISION SUMMARY
# ================================================================

if decision_sheet is not None:

    decision_summary = sheets[
        decision_sheet
    ].copy()

else:

    decision_summary = pd.DataFrame({

        "Status": [
            "Decision summary was not available."
        ]

    })


# ================================================================
# 25. PARTICIPANT COVERAGE
# ================================================================

if coverage_sheet is not None:

    participant_coverage = sheets[
        coverage_sheet
    ].copy()

else:

    participant_coverage = pd.DataFrame({

        "Status": [
            "Participant coverage sheet was not available."
        ]

    })


# ================================================================
# 26. ORIGINAL / RAW / CLEANED SHEETS
# ================================================================

def get_or_empty(sheet_name):

    if sheet_name is not None:

        return sheets[
            sheet_name
        ].copy()

    return pd.DataFrame()


original_responses = get_or_empty(
    find_sheet("01_Original_Responses")
)

raw_themes = get_or_empty(
    raw_sheet
)

cleaned_themes = get_or_empty(
    cleaned_sheet
)


# ================================================================
# 27. WRITE OUTPUT
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"SCCE_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:


    # ------------------------------------------------------------
    # 01
    # ------------------------------------------------------------

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    # ------------------------------------------------------------
    # 02
    # ------------------------------------------------------------

    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # 03
    # ------------------------------------------------------------

    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # 04
    # ------------------------------------------------------------

    if coding_audit is not None:

        coding_audit.to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )


    # ------------------------------------------------------------
    # 05
    # ------------------------------------------------------------

    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    # ------------------------------------------------------------
    # 06
    # ------------------------------------------------------------

    theme_summary_existing = get_or_empty(
        theme_sheet
    )

    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 07
    # ------------------------------------------------------------

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 08
    # ------------------------------------------------------------

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 09
    # ------------------------------------------------------------

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    # ------------------------------------------------------------
    # 10
    # ------------------------------------------------------------

    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    # ------------------------------------------------------------
    # 11
    # ------------------------------------------------------------

    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    # ------------------------------------------------------------
    # 12
    # ------------------------------------------------------------

    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    # ------------------------------------------------------------
    # 13
    # ------------------------------------------------------------

    final_scc_evidence.to_excel(
        writer,
        sheet_name="13_Final_SCCE_Evidence",
        index=False
    )


    # ------------------------------------------------------------
    # 14
    # ------------------------------------------------------------

    dimension_summary.to_excel(
        writer,
        sheet_name="14_SCCE_Dimension_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 15
    # ------------------------------------------------------------

    dimension_themes.to_excel(
        writer,
        sheet_name="15_SCCE_Dimension_Themes",
        index=False
    )


# ================================================================
# 28. FINAL REPORT
# ================================================================

print("\n")
print("=" * 80)
print("SCCE PROCESS COMPLETED")
print("=" * 80)

print(
    "\nOutput file:"
)

print(
    OUTPUT_FILE.resolve()
)


print("\n")
print("-" * 80)
print("SCCE FINAL EVIDENCE")
print("-" * 80)

print(
    final_scc_evidence.to_string(
        index=False
    )
)


print("\n")
print("-" * 80)
print("SCCE DIMENSION SUMMARY")
print("-" * 80)

print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("FILES / SHEETS CREATED")
print("=" * 80)

print("""
01_Original_Responses
02_Raw_Themes
03_Cleaned_Themes
04_Coding_Audit
05_Participant_Matrix
06_Theme_Summary
07_Normalization_Summary
08_Decision_Summary
09_Participant_Coverage
10_Unmapped_Check
11_Decision_Check
12_Quality_Checks
13_Final_SCCE_Evidence
14_SCCE_Dimension_Summary
15_SCCE_Dimension_Themes
""")


print("=" * 80)
print("IMPORTANT")
print("=" * 80)

print("""
The main new output for questionnaire development is:

14_SCCE_Dimension_Summary

and the detailed supporting table is:

15_SCCE_Dimension_Themes

These should be used for the next stage:
developing candidate SCCE questionnaire items from the
expert-derived themes.
""")

SCCE FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\SCCE_FINAL_QUALITATIVE_ANALYSIS_20260825_135013.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_SCCE_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Detected sheets:
Raw themes: 02_Raw_Themes
Coding audit: 04_Coding_Audit
Theme summary: 06_Theme_Summary
Final SCCE evidence: 11_Final_SCCE_Evidence

Final SCCE evidence columns:
['Final_SCCE_Theme', 'Number_of_Raw_Themes', 'Experts_Mentioning', 'Expert_Prevalence_%', 'Raw_Themes_Kept', 'Raw_Themes_Merged', 'Prevalence_Category']

Number of participants: 26


SCCE PROCESS COMPLETED

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode

## DIC

In [4]:
# ================================================================
# DIC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT DIC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "DIC":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "DIC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No DIC responses were found.

        Check that DIC appears in the participant sheets.
        """
    )

print("\nDIC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "DIC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)


print("\nDIC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW DIC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("DIC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. DIC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# Add the researcher-approved DIC mappings here.
#
# Example:
#
# "data interpretation":
#     "Data interpretation",
#
# "information interpretation":
#     "Data interpretation",
#
# ------------------------------------------------

DIC_NORMALIZATION = {

    # ADD DIC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in DIC_NORMALIZATION:

        normalized = DIC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × DIC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "DIC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "DIC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("DIC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique DIC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized DIC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

DIC original responses:
26

DIC cleaned theme observations:
82


DIC RAW THEMES
                             Raw_Theme                            Theme_Clean                              Theme_Key
                alternative comparison                 alternative comparison                 alternative comparison
                 alternative selection                  alternative selection                  alternative selection
                          alternatives                           alternatives                           alternatives
                          Alternatives                           Alternatives                           alternatives
                 business consequences                  b

In [15]:
# ================================================================
# COMPLETE DIC QUALITATIVE CODING ANALYSIS
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. FIND DIC INPUT FILE
# ================================================================

files = list(Path(".").glob("DIC_Coding_20260825_131815*.xlsx"))

if len(files) == 0:
    raise FileNotFoundError(
        "\nNo file beginning with 'DIC_Coding_' was found.\n"
        "Make sure your DIC Excel file is in the same folder as "
        "your Python notebook."
    )

if len(files) > 1:
    print("DIC files found:")
    for f in files:
        print(" -", f.name)

    # Use most recently modified file
    input_file = max(files, key=lambda x: x.stat().st_mtime)
    print("\nUsing the most recently modified DIC file:")
else:
    input_file = files[0]

print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 3. FLEXIBLE SHEET FINDER
# ================================================================

def find_sheet(possible_names):

    # Exact
    for name in possible_names:
        if name in excel.sheet_names:
            return name

    # Flexible
    for sheet in excel.sheet_names:

        a = (
            str(sheet).lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name).lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_DIC",
    "01_Original",
    "Original_DIC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_DIC",
    "02_Raw",
    "Raw_DIC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_DIC",
    "03_Cleaned",
    "Cleaned_DIC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])

print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 4. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(input_file, sheet_name=original_sheet)
    if original_sheet else pd.DataFrame()
)

raw_df = (
    pd.read_excel(input_file, sheet_name=raw_sheet)
    if raw_sheet else pd.DataFrame()
)

clean_df = (
    pd.read_excel(input_file, sheet_name=clean_sheet)
    if clean_sheet else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the DIC Coding/Normalized Coding sheet."
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 6. COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    ["Theme_Key", "Theme Key", "ThemeKey"]
)

raw_theme_col = find_column(
    coding_df,
    ["Raw_Theme", "Raw Theme", "RawTheme"]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    ["Decision"]
)


if theme_key_col is None:
    raise ValueError("Theme_Key column not found.")

if raw_theme_col is None:
    raise ValueError("Raw_Theme column not found.")

if normalized_col is None:
    raise ValueError(
        "Normalized_Theme column not found."
    )

if decision_col is None:
    raise ValueError(
        "Decision column not found."
    )


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:
    raise ValueError(
        "\nParticipant column not found in Cleaned DIC sheet.\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )

if participant_col != "Participant":
    clean_df = clean_df.rename(
        columns={
            participant_col: "Participant"
        }
    )


# ================================================================
# 8. THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:
    raise ValueError(
        "\nTheme_Key not found in Cleaned DIC sheet."
    )

if clean_theme_key != "Theme_Key":
    clean_df = clean_df.rename(
        columns={
            clean_theme_key: "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        ["nan", "None", "", "NaN"],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df["Participant"]
    .dropna()
    .unique()
)

print("\nParticipants:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 12. MERGE CLEANED THEMES WITH CODING
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df["Normalized_Theme"].isna()
].copy()

print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)

if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED THEME
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme["Normalized_Theme"],
    participant_theme["Participant"]
)


# P01–P26 first

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p for p in expected_participants
    if p in matrix.columns
]

other = [
    p for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()


# ================================================================
# 16. FREQUENCY
# ================================================================

participant_columns = existing + other

matrix["Frequency"] = (
    matrix[participant_columns]
    .sum(axis=1)
)


# ================================================================
# 17. PERCENTAGE
# ================================================================

total_participants = len(participants)

matrix["Percentage"] = (
    matrix["Frequency"]
    / total_participants
    * 100
).round(1)


# ================================================================
# 18. RANK
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)

matrix.insert(
    0,
    "Rank",
    range(1, len(matrix) + 1)
)


# ================================================================
# 19. THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()

theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_DIC_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 20. PREVALENCE
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 21. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby(
        "Normalized_Theme"
    )
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit[
    "Experts_Mentioning"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


coding_audit[
    "Percentage_of_Experts"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 22. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)

decision_summary[
    "Percentage"
] = (
    decision_summary[
        "Number_of_Raw_Themes"
    ]
    / len(coding_df)
    * 100
).round(1)


# ================================================================
# 23. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=["Normalized_Theme"]
    )
    .groupby(
        "Normalized_Theme"
    )
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


normalization_summary[
    "Percentage_of_Experts"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 24. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_Themes"
]


# ================================================================
# 25. DIC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": ["DIC"],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df["Theme_Key"].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 26. FINAL DIC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_DIC_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 27. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)


quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Final normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme records",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        total_participants,

        len(original_df),

        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df["Decision"]
            == "Keep"
        ).sum(),

        (
            coding_df["Decision"]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"

    ]
})


# ================================================================
# 28. SAVE COMPLETE DIC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"DIC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


# Avoid PermissionError if file already exists/open
if output_file.exists():

    output_file = Path(
        f"DIC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}_NEW.xlsx"
    )


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_DIC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 29. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("DIC ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nParticipants:",
    total_participants
)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df[
        "Theme_Key"
    ].nunique()
)

print(
    "Final normalized DIC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df[
            "Decision"
        ] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df[
            "Decision"
        ] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 30. DISPLAY FINAL DIC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL DIC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_DIC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 31. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(
    output_file.resolve()
)

print(
    "\nComplete DIC qualitative analysis workbook "
    "created successfully."
)

C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\DIC_Coding_20260825_131815.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Participants:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

Number of unmapped themes: 0


DIC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 82
Cleaned theme observations: 82
Unique raw themes: 58
Final normalized DIC themes

In [33]:
# ================================================================
# DIC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# DIC_FINAL_QUALITATIVE_ANALYSIS_20260821_095737.xlsx
#
# MAIN OUTPUT:
# DIC_FINAL_EVIDENCE_AND_DIMENSIONS_YYYYMMDD_HHMMSS.xlsx
#
# PURPOSE:
# 1. Preserve the completed DIC qualitative coding
# 2. Extract final DIC evidence
# 3. Organize themes into conceptually meaningful DIC dimensions
# 4. Calculate expert prevalence
# 5. Identify themes requiring manual review
# 6. Produce dimension-level evidence for questionnaire development
#
# IMPORTANT:
# The dimensions generated here are QUALITATIVE DIMENSIONS.
# They are NOT yet statistically validated measurement dimensions.
# ================================================================


import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. INPUT FILE
# ================================================================

TARGET = "DIC_FINAL_QUALITATIVE_ANALYSIS_20260825_135244"

possible_files = []

for ext in [".xlsx", ".xlsm", ".xls"]:
    possible_files.extend(
        Path(".").rglob(TARGET + ext)
    )

if len(possible_files) == 0:
    raise FileNotFoundError(
        "\nDIC input file was not found.\n\n"
        f"Expected:\n{TARGET}.xlsx\n\n"
        "Put the file in the same folder as the notebook."
    )

INPUT_FILE = possible_files[0]

print("=" * 80)
print("DIC FINAL QUALITATIVE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:
        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )
    except Exception as e:
        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():
            return s

    return None


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:
        x = float(x)
    except:
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


# ================================================================
# 5. IDENTIFY EXISTING SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_DIC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


print("\nDetected final evidence sheet:")
print(final_sheet)


# ================================================================
# 6. READ FINAL DIC EVIDENCE
# ================================================================

if final_sheet is None:

    raise ValueError(
        "11_Final_DIC_Evidence was not found."
    )

final_evidence = sheets[
    final_sheet
].copy()

final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


print("\nFinal DIC evidence columns:")
print(
    list(
        final_evidence.columns
    )
)


# ================================================================
# 7. IDENTIFY DIC THEME COLUMN
# ================================================================

theme_col = None

for c in final_evidence.columns:

    if str(c).strip() == "Final_DIC_Theme":

        theme_col = c
        break


if theme_col is None:

    for c in final_evidence.columns:

        if (
            "DIC" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify Final DIC Theme column."
    )


final_evidence[
    "Final_DIC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE PREVALENCE COLUMNS
# ================================================================

if (
    "Experts_Mentioning"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if (
    "Expert_Prevalence_%"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_DIC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. DETERMINE NUMBER OF PARTICIPANTS
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []

if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(
                cstr
            )


if len(participants) > 0:

    n_participants = len(
        participants
    )

else:

    # Your DIC study uses 26 experts.
    # This is used only if the participant matrix
    # does not expose P01...P26 columns.

    n_participants = 26


print(
    "\nNumber of participants:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if (
    "Expert_Prevalence_%"
    in final_evidence.columns
):

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 12. SORT FINAL DIC EVIDENCE
# ================================================================

sort_columns = []

if "Experts_Mentioning" in final_evidence.columns:
    sort_columns.append(
        "Experts_Mentioning"
    )

sort_columns.append(
    "Final_DIC_Theme"
)


final_evidence = (
    final_evidence
    .sort_values(
        sort_columns,
        ascending=[
            False
            if c == "Experts_Mentioning"
            else True
            for c in sort_columns
        ]
    )
    .reset_index(drop=True)
)


# ================================================================
# 13. FINAL DIC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_DIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]

evidence_columns = [
    c
    for c in preferred_columns
    if c in final_evidence.columns
]


final_dic_evidence = final_evidence[
    evidence_columns
].copy()


# IMPORTANT:
# Avoid "Rank already exists" errors.
# Remove any previous Rank column first.

if "Rank" in final_dic_evidence.columns:

    final_dic_evidence = (
        final_dic_evidence
        .drop(
            columns=["Rank"]
        )
    )


final_dic_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_dic_evidence) + 1
    )
)


# ================================================================
# 14. DIC DIMENSION MAPPING
# ================================================================
#
# The dimensions below are derived from the actual themes found
# in your DIC qualitative evidence.
#
# Dimension 1:
# Information Integration & Interpretation
#
# Dimension 2:
# Evidence-Based Decision Framing
#
# Dimension 3:
# Alternative & Option Evaluation
#
# Dimension 4:
# Consequence & Scenario Analysis
#
# Dimension 5:
# Multi-Criteria & Multi-Perspective Assessment
#
# Dimension 6:
# Decision-Relevant Information & Knowledge Integration
#
# Themes that cannot be assigned confidently are marked
# "Review Required".
#
# ================================================================


def map_dic_dimension(theme):

    t = str(theme).lower().strip()


    # ------------------------------------------------------------
    # 1. INFORMATION INTEGRATION & INTERPRETATION
    # ------------------------------------------------------------

    information_words = [

        "multi-source information",
        "multi source information",
        "information integration",
        "evidence integration",
        "interpretation",
        "conflicting information",
        "relevant information",
        "filtering",
        "multiple indicators",
        "cross-functional information",
        "fact–assumption distinction",
        "fact-assumption distinction"

    ]

    if any(
        word in t
        for word in information_words
    ):

        return (
            "Information Integration & Interpretation"
        )


    # ------------------------------------------------------------
    # 2. EVIDENCE-BASED DECISION FRAMING
    # ------------------------------------------------------------

    decision_framing_words = [

        "decision framing",
        "decision focus",
        "decision-oriented information",
        "decision oriented information",
        "informed decisions",
        "evidence + experience",
        "data + experience",
        "data + expertise",
        "data + knowledge",
        "data + operational knowledge",
        "operational knowledge",
        "managerial knowledge",
        "experience",
        "capacity",
        "suitability"

    ]

    if any(
        word in t
        for word in decision_framing_words
    ):

        return (
            "Evidence-Based Decision Framing"
        )


    # ------------------------------------------------------------
    # 3. ALTERNATIVE & OPTION EVALUATION
    # ------------------------------------------------------------

    alternative_words = [

        "alternatives",
        "alternative comparison",
        "alternative selection",
        "option comparison",
        "option selection",
        "options",
        "systematic comparison",
        "suitability"

    ]

    if any(
        word in t
        for word in alternative_words
    ):

        return (
            "Alternative & Option Evaluation"
        )


    # ------------------------------------------------------------
    # 4. CONSEQUENCE & SCENARIO ANALYSIS
    # ------------------------------------------------------------

    consequence_words = [

        "consequence",
        "consequences",
        "consequence analysis",
        "consequence assessment",
        "business consequences",
        "downstream consequences",
        "knock-on effects",
        "immediate/long-term consequences",
        "immediate and long-term effects",
        "temporal consequences",
        "scenario",
        "scenarios",
        "scenario comparison",
        "scenario consideration",
        "scenario testing",
        "forward-looking assessment",
        "future continuity"

    ]

    if any(
        word in t
        for word in consequence_words
    ):

        return (
            "Consequence & Scenario Analysis"
        )


    # ------------------------------------------------------------
    # 5. MULTI-CRITERIA & MULTI-PERSPECTIVE ASSESSMENT
    # ------------------------------------------------------------

    multi_criteria_words = [

        "multi-criteria evaluation",
        "multi criteria evaluation",
        "multi-criteria decisions",
        "multi criteria decisions",
        "multi-criteria assessment",
        "multi criteria assessment",
        "multi-perspective assessment",
        "multi perspective assessment",
        "multi-outcome assessment",
        "multi-dimensional consequences",
        "trade-offs",
        "financial/customer assessment",
        "financial/sustainability effects",
        "operational/financial/customer effects",
        "overall outcomes",
        "systemic thinking",
        "interdependencies",
        "risk tolerance"

    ]

    if any(
        word in t
        for word in multi_criteria_words
    ):

        return (
            "Multi-Criteria & Multi-Perspective Assessment"
        )


    # ------------------------------------------------------------
    # 6. DECISION-RELEVANT INFORMATION & KNOWLEDGE INTEGRATION
    # ------------------------------------------------------------

    knowledge_words = [

        "data + knowledge",
        "data + operational knowledge",
        "data + experience",
        "data + expertise",
        "operational knowledge",
        "managerial knowledge",
        "cross-functional information",
        "evidence + experience",
        "relevant information",
        "information integration"

    ]

    if any(
        word in t
        for word in knowledge_words
    ):

        return (
            "Decision-Relevant Information & Knowledge Integration"
        )


    # ------------------------------------------------------------
    # 7. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_dic_evidence[
    "DIC_Dimension"
] = (
    final_dic_evidence[
        "Final_DIC_Theme"
    ]
    .apply(
        map_dic_dimension
    )
)


# ================================================================
# 15. MANUALLY RESOLVE SPECIFIC AMBIGUITIES
# ================================================================
#
# These explicit mappings prevent generic words such as
# "Alternatives", "capacity", or "suitability" from being
# incorrectly grouped.
#
# ================================================================

manual_mapping = {

    "alternatives":
        "Alternative & Option Evaluation",

    "Alternatives":
        "Alternative & Option Evaluation",

    "alternative comparison":
        "Alternative & Option Evaluation",

    "alternative selection":
        "Alternative & Option Evaluation",

    "option comparison":
        "Alternative & Option Evaluation",

    "Option comparison":
        "Alternative & Option Evaluation",

    "systematic comparison":
        "Alternative & Option Evaluation",

    "consequences":
        "Consequence & Scenario Analysis",

    "Consequence analysis":
        "Consequence & Scenario Analysis",

    "consequence analysis":
        "Consequence & Scenario Analysis",

    "Consequence assessment":
        "Consequence & Scenario Analysis",

    "business consequences":
        "Consequence & Scenario Analysis",

    "downstream consequences":
        "Consequence & Scenario Analysis",

    "knock-on effects":
        "Consequence & Scenario Analysis",

    "immediate/long-term consequences":
        "Consequence & Scenario Analysis",

    "immediate and long-term effects":
        "Consequence & Scenario Analysis",

    "temporal consequences":
        "Consequence & Scenario Analysis",

    "scenario consideration":
        "Consequence & Scenario Analysis",

    "scenario testing":
        "Consequence & Scenario Analysis",

    "scenario comparison":
        "Consequence & Scenario Analysis",

    "scenarios":
        "Consequence & Scenario Analysis",

    "Multi-criteria evaluation":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-criteria evaluation":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-criteria assessment":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-criteria decisions":
        "Multi-Criteria & Multi-Perspective Assessment",

    "trade-offs":
        "Multi-Criteria & Multi-Perspective Assessment",

    "Systemic thinking":
        "Multi-Criteria & Multi-Perspective Assessment",

    "interdependencies":
        "Multi-Criteria & Multi-Perspective Assessment",

    "risk tolerance":
        "Multi-Criteria & Multi-Perspective Assessment",

    "Multi-perspective assessment":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-outcome assessment":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-dimensional consequences":
        "Multi-Criteria & Multi-Perspective Assessment",

    "financial/customer assessment":
        "Multi-Criteria & Multi-Perspective Assessment",

    "financial/sustainability effects":
        "Multi-Criteria & Multi-Perspective Assessment",

    "operational/financial/customer effects":
        "Multi-Criteria & Multi-Perspective Assessment",

    "overall outcomes":
        "Multi-Criteria & Multi-Perspective Assessment",

    "future continuity":
        "Consequence & Scenario Analysis",

    "Forward-looking assessment":
        "Consequence & Scenario Analysis",

    "Decision framing":
        "Evidence-Based Decision Framing",

    "Decision focus":
        "Evidence-Based Decision Framing",

    "Decision-oriented information":
        "Evidence-Based Decision Framing",

    "informed decisions":
        "Evidence-Based Decision Framing",

    "Evidence + experience":
        "Evidence-Based Decision Framing",

    "data + experience":
        "Evidence-Based Decision Framing",

    "data + expertise":
        "Evidence-Based Decision Framing",

    "data + knowledge":
        "Decision-Relevant Information & Knowledge Integration",

    "data + operational knowledge":
        "Decision-Relevant Information & Knowledge Integration",

    "operational knowledge":
        "Decision-Relevant Information & Knowledge Integration",

    "managerial knowledge":
        "Decision-Relevant Information & Knowledge Integration",

    "cross-functional information":
        "Decision-Relevant Information & Knowledge Integration",

    "experience":
        "Evidence-Based Decision Framing",

    "capacity":
        "Evidence-Based Decision Framing",

    "suitability":
        "Evidence-Based Decision Framing",

    "Multi-source information":
        "Information Integration & Interpretation",

    "multi-source information":
        "Information Integration & Interpretation",

    "Evidence integration":
        "Information Integration & Interpretation",

    "information integration":
        "Information Integration & Interpretation",

    "Interpretation":
        "Information Integration & Interpretation",

    "filtering":
        "Information Integration & Interpretation",

    "Multiple indicators":
        "Information Integration & Interpretation",

    "Conflicting information":
        "Information Integration & Interpretation",

    "relevant information":
        "Information Integration & Interpretation",

    "Fact–assumption distinction":
        "Information Integration & Interpretation"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_dic_evidence[
            "Final_DIC_Theme"
        ]
        .astype(str)
        .str.strip()
        .eq(theme)
    )

    final_dic_evidence.loc[
        mask,
        "DIC_Dimension"
    ] = dimension


# ================================================================
# 16. CREATE DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_dic_evidence
    .groupby(
        "DIC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_DIC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # ------------------------------------------------------------
    # Calculate experts mentioning dimension
    # ------------------------------------------------------------

    experts_mentioning = 0


    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        temp = participant_matrix.copy()

        first_col = temp.columns[0]

        temp[
            "_Theme"
        ] = (
            temp[
                first_col
            ]
            .astype(str)
            .str.strip()
        )


        # Case-insensitive matching
        theme_lower = {
            str(x).strip().lower()
            for x in themes
        }


        matched = temp[
            temp[
                "_Theme"
            ]
            .str.lower()
            .isin(
                theme_lower
            )
        ]


        if len(matched) > 0:

            available_participants = [
                p
                for p in participants
                if p in matched.columns
            ]


            if len(
                available_participants
            ) > 0:

                vals = (
                    matched[
                        available_participants
                    ]
                    .apply(
                        pd.to_numeric,
                        errors="coerce"
                    )
                    .fillna(0)
                )


                experts_mentioning = int(
                    (
                        vals.sum(axis=0) > 0
                    ).sum()
                )


    # ------------------------------------------------------------
    # Fallback if participant matrix cannot be used
    # ------------------------------------------------------------

    if experts_mentioning == 0:

        if (
            "Experts_Mentioning"
            in group.columns
        ):

            experts_mentioning = int(
                group[
                    "Experts_Mentioning"
                ]
                .max()
            )


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0
    )


    dimension_rows.append({

        "DIC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_DIC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 17. SORT DIMENSIONS
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(
                columns=["Rank"]
            )
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 18. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "DIC_Dimension",

    "Final_DIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [
    c
    for c in dimension_theme_columns
    if c in final_dic_evidence.columns
]


dimension_themes = final_dic_evidence[
    dimension_theme_columns
].copy()


dimension_themes = (
    dimension_themes
    .sort_values(
        [
            "DIC_Dimension",
            "Experts_Mentioning"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ================================================================
# 19. REVIEW-REQUIRED THEMES
# ================================================================

unmapped_check = final_dic_evidence[
    final_dic_evidence[
        "DIC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    unmapped_check = unmapped_check[
        [
            "Final_DIC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ]
    ].copy()

    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )

else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All DIC themes mapped to a dimension."
        ]

    })


# ================================================================
# 20. DECISION CHECK
# ================================================================

coding_audit = None

if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


if coding_audit is not None:

    decision_columns = [

        c
        for c in coding_audit.columns
        if "Decision" in str(c)

    ]


    if len(decision_columns) > 0:

        decision_col = decision_columns[0]

        missing = coding_audit[
            coding_audit[
                decision_col
            ].isna()
        ]


        if len(missing) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All DIC coding decisions are present."
                ]

            })

        else:

            decision_check = (
                missing.copy()
            )


    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })

else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit unavailable."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final DIC evidence available",

    "Result":
        "PASS"
        if len(final_dic_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_dic_evidence)} final DIC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "DIC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative dimensions"

})


quality_rows.append({

    "Quality_Check":
        "Themes requiring review",

    "Result":
        len(unmapped_check)
        if "Final_DIC_Theme"
        in unmapped_check.columns
        else 0,

    "Details":
        "Themes assigned to Review Required"

})


quality_rows.append({

    "Quality_Check":
        "Duplicate theme labels",

    "Result":
        int(
            final_dic_evidence[
                "Final_DIC_Theme"
            ]
            .str.lower()
            .duplicated()
            .sum()
        ),

    "Details":
        "Case-insensitive duplicate labels detected"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD ORIGINAL SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:

        return sheets[
            sheet
        ].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. WRITE FINAL EXCEL FILE
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"DIC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:


    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )


    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    # ------------------------------------------------------------
    # MAIN DIC EVIDENCE
    # ------------------------------------------------------------

    final_dic_evidence.to_excel(
        writer,
        sheet_name="13_Final_DIC_Evidence",
        index=False
    )


    # ------------------------------------------------------------
    # DIC DIMENSION SUMMARY
    # ------------------------------------------------------------

    dimension_summary.to_excel(
        writer,
        sheet_name="14_DIC_Dimension_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # DIC DIMENSION × THEME
    # ------------------------------------------------------------

    dimension_themes.to_excel(
        writer,
        sheet_name="15_DIC_Dimension_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # ORIGINAL CONSTRUCT STATISTICS
    # ------------------------------------------------------------

    construct_statistics.to_excel(
        writer,
        sheet_name="16_Construct_Statistics",
        index=False
    )


# ================================================================
# 24. PRINT RESULTS
# ================================================================

print("\n")
print("=" * 80)
print("DIC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 80)


print("\nOutput file:")

print(
    OUTPUT_FILE.resolve()
)


print("\n")
print("-" * 80)
print("FINAL DIC EVIDENCE")
print("-" * 80)


print(
    final_dic_evidence.to_string(
        index=False
    )
)


print("\n")
print("-" * 80)
print("DIC DIMENSION SUMMARY")
print("-" * 80)


print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("OUTPUT SHEETS")
print("=" * 80)

print("""
01_Original_Responses
02_Raw_Themes
03_Cleaned_Themes
04_Coding_Audit
05_Participant_Matrix
06_Theme_Summary
07_Normalization_Summary
08_Decision_Summary
09_Participant_Coverage
10_Unmapped_Check
11_Decision_Check
12_Quality_Checks
13_Final_DIC_Evidence
14_DIC_Dimension_Summary
15_DIC_Dimension_Themes
16_Construct_Statistics
""")


print("\n")
print("=" * 80)
print("NEXT STAGE")
print("=" * 80)

print("""
The two most important sheets for questionnaire development are:

14_DIC_Dimension_Summary
15_DIC_Dimension_Themes

These provide the bridge from expert qualitative evidence
to candidate DIC questionnaire items.

DO NOT treat these dimensions as statistically validated
subdimensions yet. They are evidence-based qualitative
groupings that will be used to formulate candidate items.
""")

DIC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\DIC_FINAL_QUALITATIVE_ANALYSIS_20260825_135244.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_DIC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Detected final evidence sheet:
11_Final_DIC_Evidence

Final DIC evidence columns:
['Final_DIC_Theme', 'Number_of_Raw_Themes', 'Experts_Mentioning', 'Expert_Prevalence_%', 'Raw_Themes_Kept', 'Raw_Themes_Merged', 'Prevalence_Category']

Number of participants: 26


DIC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\DIC_FINAL_EVIDENCE_AND_DIMENSIONS_20260

## BGC

In [16]:
# ================================================================
# BGC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT BGC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "BGC":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "BGC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No BGC responses were found.

        Check that BGC appears in the participant sheets.
        """
    )

print("\nBGC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "BGC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)


print("\nBGC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW BGC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("BGC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. BGC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# Add researcher-approved BGC mappings here after reviewing
# the actual BGC raw themes.
#
# Example:
#
# "shared governance":
#     "Collaborative blockchain governance",
#
# "joint governance":
#     "Collaborative blockchain governance",
#
# ------------------------------------------------

BGC_NORMALIZATION = {

    # ADD BGC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in BGC_NORMALIZATION:

        normalized = BGC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × BGC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "BGC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "BGC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("BGC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique BGC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized BGC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

BGC original responses:
26

BGC cleaned theme observations:
100


BGC RAW THEMES
                    Raw_Theme                   Theme_Clean                     Theme_Key
                       Access                        Access                        access
                       access                        access                        access
                access rights                 access rights                 access rights
               accountability                accountability                accountability
               Accountability                Accountability                accountability
                   compliance                    compliance                    compliance
        com

In [18]:
# ================================================================
# COMPLETE BGC QUALITATIVE CODING ANALYSIS
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path("BGC_Coding_20260825_135502.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as "
        "your Python notebook."
    )

print("=" * 70)
print("BGC QUALITATIVE CODING ANALYSIS")
print("=" * 70)

print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 3. FLEXIBLE SHEET FINDER
# ================================================================

def find_sheet(possible_names):

    # Exact match
    for name in possible_names:
        if name in excel.sheet_names:
            return name

    # Flexible match
    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_BGC",
    "01_Original",
    "Original_BGC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_BGC",
    "02_Raw",
    "Raw_BGC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_BGC",
    "03_Cleaned",
    "Cleaned_BGC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 4. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(
        input_file,
        sheet_name=original_sheet
    )
    if original_sheet
    else pd.DataFrame()
)

raw_df = (
    pd.read_excel(
        input_file,
        sheet_name=raw_sheet
    )
    if raw_sheet
    else pd.DataFrame()
)

clean_df = (
    pd.read_excel(
        input_file,
        sheet_name=clean_sheet
    )
    if clean_sheet
    else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the BGC Coding/Normalized Coding sheet."
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 6. COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    # Exact match
    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    # Flexible match
    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

raw_theme_col = find_column(
    coding_df,
    [
        "Raw_Theme",
        "Raw Theme",
        "RawTheme"
    ]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    [
        "Decision"
    ]
)


if theme_key_col is None:
    raise ValueError(
        "\nTheme_Key column not found."
    )

if raw_theme_col is None:
    raise ValueError(
        "\nRaw_Theme column not found."
    )

if normalized_col is None:
    raise ValueError(
        "\nNormalized_Theme column not found."
    )

if decision_col is None:
    raise ValueError(
        "\nDecision column not found."
    )


# Rename to standard names

coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:

    raise ValueError(
        "\nParticipant column not found in Cleaned BGC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col:
            "Participant"
        }
    )


# ================================================================
# 8. THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key not found in Cleaned BGC sheet."
    )


if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key:
            "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        [
            "nan",
            "None",
            "",
            "NaN"
        ],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df[
        "Participant"
    ]
    .dropna()
    .unique()
)

print("\nParticipants:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 12. MERGE CLEANED DATA WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. CHECK UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df[
        "Normalized_Theme"
    ].isna()
].copy()


print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)


if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED THEME
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme[
        "Normalized_Theme"
    ],
    participant_theme[
        "Participant"
    ]
)


# P01–P26 ordering

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()


# ================================================================
# 16. FREQUENCY
# ================================================================

participant_columns = existing + other

matrix["Frequency"] = (
    matrix[
        participant_columns
    ]
    .sum(axis=1)
)


# ================================================================
# 17. PERCENTAGE
# ================================================================

total_participants = len(participants)

matrix["Percentage"] = (
    matrix["Frequency"]
    / total_participants
    * 100
).round(1)


# ================================================================
# 18. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)

matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 19. THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()

theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_BGC_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 20. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 21. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby(
        "Normalized_Theme"
    )
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit[
    "Experts_Mentioning"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


coding_audit[
    "Percentage_of_Experts"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 22. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


decision_summary[
    "Percentage"
] = (
    decision_summary[
        "Number_of_Raw_Themes"
    ]
    / len(coding_df)
    * 100
).round(1)


# ================================================================
# 23. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=[
            "Normalized_Theme"
        ]
    )
    .groupby(
        "Normalized_Theme"
    )
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


normalization_summary[
    "Percentage_of_Experts"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 24. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_Themes"
]


# ================================================================
# 25. BGC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": [
        "BGC"
    ],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df[
            "Theme_Key"
        ].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 26. FINAL BGC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_BGC_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 27. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)


quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Final normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme records",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        total_participants,

        len(original_df),

        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum(),

        (
            coding_df[
                "Decision"
            ] == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"

    ]
})


# ================================================================
# 28. SAVE COMPLETE BGC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"BGC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


# Prevent accidental overwrite

counter = 1

base_output = output_file

while output_file.exists():

    output_file = Path(
        f"BGC_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_BGC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 29. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("BGC ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nParticipants:",
    total_participants
)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df[
        "Theme_Key"
    ].nunique()
)

print(
    "Final normalized BGC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df[
            "Decision"
        ] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df[
            "Decision"
        ] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 30. DISPLAY FINAL BGC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL BGC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_BGC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 31. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(
    output_file.resolve()
)

print(
    "\nComplete BGC qualitative analysis workbook "
    "created successfully."
)

BGC QUALITATIVE CODING ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\BGC_Coding_20260825_135502.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Participants:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

Number of unmapped themes: 0


BGC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 100
Cleaned theme observations: 100
Un

In [34]:
# ================================================================
# BGC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# BGC_FINAL_QUALITATIVE_ANALYSIS_20260821_100628.xlsx
#
# OUTPUT:
# BGC_FINAL_EVIDENCE_AND_DIMENSIONS_YYYYMMDD_HHMMSS.xlsx
#
# PURPOSE:
# 1. Preserve the completed BGC qualitative coding
# 2. Extract final BGC evidence
# 3. Organize final themes into evidence-based BGC dimensions
# 4. Calculate expert prevalence
# 5. Identify themes requiring review
# 6. Create dimension × theme evidence tables
#
# NOTE:
# These are QUALITATIVE dimensions derived from expert evidence.
# They are NOT yet statistically validated subdimensions.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

TARGET = "BGC_FINAL_QUALITATIVE_ANALYSIS_20260825_135606"

possible_files = []

for ext in [".xlsx", ".xlsm", ".xls"]:
    possible_files.extend(
        Path(".").rglob(TARGET + ext)
    )

if len(possible_files) == 0:
    raise FileNotFoundError(
        "\nBGC input file was not found.\n\n"
        f"Expected file:\n{TARGET}.xlsx\n\n"
        "Put the Excel file in the same folder as your notebook."
    )

INPUT_FILE = possible_files[0]

print("=" * 80)
print("BGC FINAL QUALITATIVE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:
    try:
        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )
    except Exception as e:
        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():
            return s

    return None


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:
        x = float(x)
    except:
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


# ================================================================
# 5. IDENTIFY SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_BGC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


if final_sheet is None:

    raise ValueError(
        "11_Final_BGC_Evidence was not found."
    )


print(
    "\nFinal BGC evidence sheet:",
    final_sheet
)


# ================================================================
# 6. READ FINAL BGC EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()

final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# ================================================================
# 7. IDENTIFY THEME COLUMN
# ================================================================

theme_col = None

for c in final_evidence.columns:

    if str(c).strip() == "Final_BGC_Theme":

        theme_col = c
        break


if theme_col is None:

    for c in final_evidence.columns:

        if (
            "BGC" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify Final BGC Theme column."
    )


final_evidence[
    "Final_BGC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE PREVALENCE COLUMNS
# ================================================================

if (
    "Experts_Mentioning"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if (
    "Expert_Prevalence_%"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_BGC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. PARTICIPANT COUNT
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []

if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(
                cstr
            )


# Your study has 26 experts.
# The participant matrix is used when available.

if len(participants) > 0:

    n_participants = len(
        participants
    )

else:

    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if (
    "Expert_Prevalence_%"
    in final_evidence.columns
):

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 12. SORT FINAL EVIDENCE
# ================================================================

sort_columns = []

if "Experts_Mentioning" in final_evidence.columns:

    sort_columns.append(
        "Experts_Mentioning"
    )

sort_columns.append(
    "Final_BGC_Theme"
)


final_evidence = (
    final_evidence
    .sort_values(
        sort_columns,
        ascending=[
            False
            if c == "Experts_Mentioning"
            else True
            for c in sort_columns
        ]
    )
    .reset_index(drop=True)
)


# ================================================================
# 13. FINAL BGC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_BGC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]


evidence_columns = [
    c
    for c in preferred_columns
    if c in final_evidence.columns
]


final_bgc_evidence = final_evidence[
    evidence_columns
].copy()


# Prevent Rank duplication

if "Rank" in final_bgc_evidence.columns:

    final_bgc_evidence = (
        final_bgc_evidence
        .drop(columns=["Rank"])
    )


final_bgc_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_bgc_evidence) + 1
    )
)


# ================================================================
# 14. BGC DIMENSION MAPPING
# ================================================================
#
# Dimensions are derived from the actual BGC themes in the
# attached workbook.
#
# ================================================================


def map_bgc_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. ROLES, RESPONSIBILITIES & ACCOUNTABILITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "accountability",
            "responsibility",
            "responsibilities",
            "roles",
            "ownership",
            "participant responsibility",
            "participant obligations",
            "partner obligations",
            "network responsibility",
            "data responsibility",
            "governance responsibilities"

        ]
    ):

        return (
            "Roles, Responsibilities & Accountability"
        )


    # ------------------------------------------------------------
    # 2. PARTICIPATION, ACCESS & AUTHORITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "access",
            "access rights",
            "participation",
            "participation rules",
            "participant admission",
            "information authority",
            "contribution authority",
            "data-entry authority",
            "modification rights",
            "contribution"

        ]
    ):

        return (
            "Participation, Access & Authority"
        )


    # ------------------------------------------------------------
    # 3. GOVERNANCE RULES, STANDARDS & COMPLIANCE
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "compliance",
            "compliance monitoring",
            "standards",
            "data standards",
            "data-quality standards",
            "predefined rules",
            "governance rules",
            "governance",
            "procedures",
            "mandatory information",
            "information requirements",
            "prior agreements"

        ]
    ):

        return (
            "Governance Rules, Standards & Compliance"
        )


    # ------------------------------------------------------------
    # 4. DATA QUALITY, VALIDATION & INFORMATION CONTROL
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "verification",
            "validation",
            "data quality",
            "information quality",
            "inaccurate records",
            "correction",
            "correction procedures",
            "corrective action",
            "error resolution",
            "error handling"

        ]
    ):

        return (
            "Data Quality, Validation & Information Control"
        )


    # ------------------------------------------------------------
    # 5. DISPUTE, EXCEPTION & PROBLEM MANAGEMENT
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "dispute",
            "disputes",
            "dispute resolution",
            "dispute handling",
            "dispute management",
            "disagreement management",
            "disagreement resolution",
            "exception handling",
            "failure procedures",
            "problem resolution"

        ]
    ):

        return (
            "Dispute, Exception & Problem Management"
        )


    # ------------------------------------------------------------
    # 6. INTERORGANIZATIONAL GOVERNANCE ALIGNMENT & TRUST
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "interorganizational agreement",
            "interorganizational trust",
            "governance alignment"

        ]
    ):

        return (
            "Interorganizational Governance Alignment & Trust"
        )


    # ------------------------------------------------------------
    # 7. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_bgc_evidence[
    "BGC_Dimension"
] = (
    final_bgc_evidence[
        "Final_BGC_Theme"
    ]
    .apply(
        map_bgc_dimension
    )
)


# ================================================================
# 15. EXPLICIT MAPPING FOR AMBIGUOUS THEMES
# ================================================================
#
# This resolves capitalization and short generic labels without
# altering the original qualitative evidence.
# ================================================================

manual_mapping = {

    # Roles / responsibility

    "Accountability":
        "Roles, Responsibilities & Accountability",

    "accountability":
        "Roles, Responsibilities & Accountability",

    "Responsibility":
        "Roles, Responsibilities & Accountability",

    "responsibility":
        "Roles, Responsibilities & Accountability",

    "Responsibilities":
        "Roles, Responsibilities & Accountability",

    "responsibilities":
        "Roles, Responsibilities & Accountability",

    "Roles":
        "Roles, Responsibilities & Accountability",

    "Ownership":
        "Roles, Responsibilities & Accountability",

    "Network responsibility":
        "Roles, Responsibilities & Accountability",

    "Data responsibility":
        "Roles, Responsibilities & Accountability",

    "data responsibility":
        "Roles, Responsibilities & Accountability",

    "Participant responsibility":
        "Roles, Responsibilities & Accountability",

    "Participant obligations":
        "Roles, Responsibilities & Accountability",

    "Partner obligations":
        "Roles, Responsibilities & Accountability",

    "Governance responsibilities":
        "Roles, Responsibilities & Accountability",


    # Participation / access / authority

    "access":
        "Participation, Access & Authority",

    "Access":
        "Participation, Access & Authority",

    "access rights":
        "Participation, Access & Authority",

    "Participation":
        "Participation, Access & Authority",

    "participation":
        "Participation, Access & Authority",

    "Participation rules":
        "Participation, Access & Authority",

    "participant admission":
        "Participation, Access & Authority",

    "Information authority":
        "Participation, Access & Authority",

    "Contribution authority":
        "Participation, Access & Authority",

    "contribution":
        "Participation, Access & Authority",

    "Data-entry authority":
        "Participation, Access & Authority",

    "modification rights":
        "Participation, Access & Authority",


    # Rules / standards / compliance

    "compliance":
        "Governance Rules, Standards & Compliance",

    "compliance monitoring":
        "Governance Rules, Standards & Compliance",

    "standards":
        "Governance Rules, Standards & Compliance",

    "Data standards":
        "Governance Rules, Standards & Compliance",

    "data-quality standards":
        "Governance Rules, Standards & Compliance",

    "Predefined rules":
        "Governance Rules, Standards & Compliance",

    "Governance rules":
        "Governance Rules, Standards & Compliance",

    "Governance":
        "Governance Rules, Standards & Compliance",

    "Procedures":
        "Governance Rules, Standards & Compliance",

    "information requirements":
        "Governance Rules, Standards & Compliance",

    "mandatory information":
        "Governance Rules, Standards & Compliance",

    "prior agreements":
        "Governance Rules, Standards & Compliance",


    # Data quality / validation

    "verification":
        "Data Quality, Validation & Information Control",

    "validation":
        "Data Quality, Validation & Information Control",

    "data quality":
        "Data Quality, Validation & Information Control",

    "information quality":
        "Data Quality, Validation & Information Control",

    "inaccurate records":
        "Data Quality, Validation & Information Control",

    "correction":
        "Data Quality, Validation & Information Control",

    "correction procedures":
        "Data Quality, Validation & Information Control",

    "corrective action":
        "Data Quality, Validation & Information Control",

    "error resolution":
        "Data Quality, Validation & Information Control",

    "error handling":
        "Data Quality, Validation & Information Control",


    # Dispute / exception

    "dispute resolution":
        "Dispute, Exception & Problem Management",

    "disputes":
        "Dispute, Exception & Problem Management",

    "dispute handling":
        "Dispute, Exception & Problem Management",

    "dispute management":
        "Dispute, Exception & Problem Management",

    "Disagreement management":
        "Dispute, Exception & Problem Management",

    "disagreement resolution":
        "Dispute, Exception & Problem Management",

    "exception handling":
        "Dispute, Exception & Problem Management",

    "failure procedures":
        "Dispute, Exception & Problem Management",

    "problem resolution":
        "Dispute, Exception & Problem Management",


    # Interorganizational governance

    "Interorganizational agreement":
        "Interorganizational Governance Alignment & Trust",

    "interorganizational trust":
        "Interorganizational Governance Alignment & Trust",

    "governance alignment":
        "Interorganizational Governance Alignment & Trust"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_bgc_evidence[
            "Final_BGC_Theme"
        ]
        .astype(str)
        .str.strip()
        .eq(theme)
    )

    final_bgc_evidence.loc[
        mask,
        "BGC_Dimension"
    ] = dimension


# ================================================================
# 16. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_bgc_evidence
    .groupby(
        "BGC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_BGC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # ------------------------------------------------------------
    # Calculate experts mentioning dimension
    # ------------------------------------------------------------

    experts_mentioning = 0


    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        temp = participant_matrix.copy()

        first_col = temp.columns[0]

        temp[
            "_Theme"
        ] = (
            temp[
                first_col
            ]
            .astype(str)
            .str.strip()
        )


        theme_lower = {
            str(x).strip().lower()
            for x in themes
        }


        matched = temp[
            temp[
                "_Theme"
            ]
            .str.lower()
            .isin(
                theme_lower
            )
        ]


        available_participants = [
            p
            for p in participants
            if p in matched.columns
        ]


        if (
            len(matched) > 0
            and len(available_participants) > 0
        ):

            vals = (
                matched[
                    available_participants
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce"
                )
                .fillna(0)
            )


            experts_mentioning = int(
                (
                    vals.sum(axis=0) > 0
                ).sum()
            )


    # ------------------------------------------------------------
    # Fallback using final evidence
    # ------------------------------------------------------------

    if experts_mentioning == 0:

        if (
            "Experts_Mentioning"
            in group.columns
        ):

            experts_mentioning = int(
                group[
                    "Experts_Mentioning"
                ]
                .max()
            )


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0
    )


    dimension_rows.append({

        "BGC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_BGC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 17. SORT DIMENSION SUMMARY
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(columns=["Rank"])
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 18. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "BGC_Dimension",

    "Final_BGC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [
    c
    for c in dimension_theme_columns
    if c in final_bgc_evidence.columns
]


dimension_themes = final_bgc_evidence[
    dimension_theme_columns
].copy()


dimension_themes = (
    dimension_themes
    .sort_values(
        [
            "BGC_Dimension",
            "Experts_Mentioning"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ================================================================
# 19. REVIEW-REQUIRED THEMES
# ================================================================

unmapped_check = final_bgc_evidence[
    final_bgc_evidence[
        "BGC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    unmapped_check = unmapped_check[
        [
            "Final_BGC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ]
    ].copy()

    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )

else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All BGC themes mapped to a dimension."
        ]

    })


# ================================================================
# 20. DECISION CHECK
# ================================================================

coding_audit = None

if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


if coding_audit is not None:

    decision_columns = [
        c
        for c in coding_audit.columns
        if "Decision" in str(c)
    ]


    if len(decision_columns) > 0:

        decision_col = decision_columns[0]

        missing = coding_audit[
            coding_audit[
                decision_col
            ].isna()
        ]


        if len(missing) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All BGC coding decisions are present."
                ]

            })

        else:

            decision_check = missing.copy()


    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })

else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit unavailable."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final BGC evidence available",

    "Result":
        "PASS"
        if len(final_bgc_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_bgc_evidence)} final BGC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "BGC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative dimensions"

})


quality_rows.append({

    "Quality_Check":
        "Themes requiring review",

    "Result":
        (
            len(unmapped_check)
            if "Final_BGC_Theme"
            in unmapped_check.columns
            else 0
        ),

    "Details":
        "Themes assigned to Review Required"

})


quality_rows.append({

    "Quality_Check":
        "Duplicate theme labels",

    "Result":
        int(
            final_bgc_evidence[
                "Final_BGC_Theme"
            ]
            .str.lower()
            .duplicated()
            .sum()
        ),

    "Details":
        "Case-insensitive duplicate labels detected"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:
        return sheets[sheet].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. WRITE OUTPUT WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"BGC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )

    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )

    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    # ------------------------------------------------------------
    # FINAL BGC EVIDENCE
    # ------------------------------------------------------------

    final_bgc_evidence.to_excel(
        writer,
        sheet_name="13_Final_BGC_Evidence",
        index=False
    )

    # ------------------------------------------------------------
    # BGC DIMENSION SUMMARY
    # ------------------------------------------------------------

    dimension_summary.to_excel(
        writer,
        sheet_name="14_BGC_Dimension_Summary",
        index=False
    )

    # ------------------------------------------------------------
    # BGC DIMENSION × THEME
    # ------------------------------------------------------------

    dimension_themes.to_excel(
        writer,
        sheet_name="15_BGC_Dimension_Themes",
        index=False
    )

    # ------------------------------------------------------------
    # CONSTRUCT STATISTICS
    # ------------------------------------------------------------

    construct_statistics.to_excel(
        writer,
        sheet_name="16_Construct_Statistics",
        index=False
    )


# ================================================================
# 24. PRINT RESULTS
# ================================================================

print("\n")
print("=" * 80)
print("BGC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 80)

print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)


print("\n")
print("-" * 80)
print("FINAL BGC EVIDENCE")
print("-" * 80)

print(
    final_bgc_evidence.to_string(
        index=False
    )
)


print("\n")
print("-" * 80)
print("BGC DIMENSION SUMMARY")
print("-" * 80)

print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("OUTPUT SHEETS")
print("=" * 80)

print("""
01_Original_Responses
02_Raw_Themes
03_Cleaned_Themes
04_Coding_Audit
05_Participant_Matrix
06_Theme_Summary
07_Normalization_Summary
08_Decision_Summary
09_Participant_Coverage
10_Unmapped_Check
11_Decision_Check
12_Quality_Checks
13_Final_BGC_Evidence
14_BGC_Dimension_Summary
15_BGC_Dimension_Themes
16_Construct_Statistics
""")


print("\n")
print("=" * 80)
print("NEXT STAGE")
print("=" * 80)

print("""
For questionnaire development, focus on:

14_BGC_Dimension_Summary
15_BGC_Dimension_Themes

These sheets connect the expert-derived BGC themes
to broader qualitative dimensions.

IMPORTANT:
The dimensions are evidence-based qualitative groupings.
They are NOT yet statistically validated measurement
dimensions. They will be used to develop candidate
questionnaire items.
""")

BGC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\BGC_FINAL_QUALITATIVE_ANALYSIS_20260825_135606.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_BGC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Final BGC evidence sheet: 11_Final_BGC_Evidence

Participants used: 26


BGC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\BGC_FINAL_EVIDENCE_AND_DIMENSIONS_20260825_143904.xlsx


--------------------------------------------------------------------------------
FINAL BGC EVIDENCE
-------------------------------------------------------------------------

## RIC

In [5]:
# ================================================================
# RIC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT RIC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "RIC":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "RIC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No RIC responses were found.

        Check that RIC appears in the participant sheets.
        """
    )

print("\nRIC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "RIC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)

print("\nRIC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW RIC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("RIC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. RIC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# Add researcher-approved RIC mappings here after reviewing
# the actual RIC raw themes.
#
# Example format:
#
# "risk identification":
#     "Risk identification",
#
# "identifying emerging risks":
#     "Risk identification",
#
# ------------------------------------------------

RIC_NORMALIZATION = {

    # ADD RIC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in RIC_NORMALIZATION:

        normalized = RIC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × RIC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "RIC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "RIC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("RIC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique RIC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized RIC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

RIC original responses:
26

RIC cleaned theme observations:
98


RIC RAW THEMES
                           Raw_Theme                          Theme_Clean                            Theme_Key
                        alternatives                         alternatives                         alternatives
                        anticipation                         anticipation                         anticipation
             Business interpretation              Business interpretation              business interpretation
                  causal connections                   causal connections                   causal connections
                 combined indicators                  combined indicators              

In [19]:
# ================================================================
# COMPLETE RIC QUALITATIVE CODING ANALYSIS
# Input: RIC_Coding_20260821_092922.xlsx
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path("RIC_Coding_20260825_132313.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as your "
        "Python notebook."
    )

print("=" * 70)
print("RIC QUALITATIVE CODING ANALYSIS")
print("=" * 70)

print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 3. FLEXIBLE SHEET FINDER
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_RIC",
    "01_Original",
    "Original_RIC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_RIC",
    "02_Raw",
    "Raw_RIC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_RIC",
    "03_Cleaned",
    "Cleaned_RIC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 4. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(
        input_file,
        sheet_name=original_sheet
    )
    if original_sheet
    else pd.DataFrame()
)

raw_df = (
    pd.read_excel(
        input_file,
        sheet_name=raw_sheet
    )
    if raw_sheet
    else pd.DataFrame()
)

clean_df = (
    pd.read_excel(
        input_file,
        sheet_name=clean_sheet
    )
    if clean_sheet
    else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the RIC Coding/Normalized Coding sheet.\n"
        "Available sheets are:\n"
        + "\n".join(
            str(x)
            for x in excel.sheet_names
        )
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 6. COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

raw_theme_col = find_column(
    coding_df,
    [
        "Raw_Theme",
        "Raw Theme",
        "RawTheme"
    ]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    [
        "Decision"
    ]
)


if theme_key_col is None:
    raise ValueError(
        "\nTheme_Key column not found."
    )

if raw_theme_col is None:
    raise ValueError(
        "\nRaw_Theme column not found."
    )

if normalized_col is None:
    raise ValueError(
        "\nNormalized_Theme column not found."
    )

if decision_col is None:
    raise ValueError(
        "\nDecision column not found."
    )


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. FIND PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:

    raise ValueError(
        "\nParticipant column not found in Cleaned RIC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col:
            "Participant"
        }
    )


# ================================================================
# 8. FIND THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key not found in Cleaned RIC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key:
            "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        [
            "nan",
            "None",
            "",
            "NaN"
        ],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANT LIST
# ================================================================

participants = sorted(
    clean_df[
        "Participant"
    ]
    .replace("", np.nan)
    .dropna()
    .unique()
)

print("\nParticipants:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 12. MERGE CLEANED DATA WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. CHECK UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df[
        "Normalized_Theme"
    ].isna()
].copy()


print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)


if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED RIC THEME
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme[
        "Normalized_Theme"
    ],
    participant_theme[
        "Participant"
    ]
)


# ================================================================
# 16. ORDER PARTICIPANTS P01, P02, ...
# ================================================================

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()


participant_columns = (
    existing + other
)


# ================================================================
# 17. FREQUENCY
# ================================================================

matrix["Frequency"] = (
    matrix[
        participant_columns
    ]
    .sum(axis=1)
)


# ================================================================
# 18. PERCENTAGE OF PARTICIPANTS
# ================================================================

total_participants = len(participants)

if total_participants > 0:

    matrix["Percentage"] = (
        matrix["Frequency"]
        / total_participants
        * 100
    ).round(1)

else:

    matrix["Percentage"] = 0


# ================================================================
# 19. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 20. RIC THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_RIC_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 21. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 22. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby(
        "Normalized_Theme"
    )
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit[
    "Experts_Mentioning"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


if total_participants > 0:

    coding_audit[
        "Percentage_of_Experts"
    ] = (
        coding_audit[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    coding_audit[
        "Percentage_of_Experts"
    ] = 0


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 23. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


if len(coding_df) > 0:

    decision_summary[
        "Percentage"
    ] = (
        decision_summary[
            "Number_of_Raw_Themes"
        ]
        / len(coding_df)
        * 100
    ).round(1)

else:

    decision_summary[
        "Percentage"
    ] = 0


# ================================================================
# 24. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=[
            "Normalized_Theme"
        ]
    )
    .groupby(
        "Normalized_Theme"
    )
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


if total_participants > 0:

    normalization_summary[
        "Percentage_of_Experts"
    ] = (
        normalization_summary[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    normalization_summary[
        "Percentage_of_Experts"
    ] = 0


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 25. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_RIC_Themes"
]


# ================================================================
# 26. RIC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": [
        "RIC"
    ],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df[
            "Theme_Key"
        ].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 27. FINAL RIC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_RIC_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 28. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)


quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Final normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme records",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        total_participants,

        len(original_df),

        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum(),

        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"

    ]
})


# ================================================================
# 29. SAVE COMPLETE RIC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"RIC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


# Avoid overwrite / PermissionError

counter = 1

while output_file.exists():

    output_file = Path(
        f"RIC_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_RIC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 30. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("RIC ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nParticipants:",
    total_participants
)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df[
        "Theme_Key"
    ].nunique()
)

print(
    "Final normalized RIC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df[
            "Decision"
        ] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df[
            "Decision"
        ] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 31. DISPLAY FINAL RIC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL RIC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_RIC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 32. OUTPUT FILE
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(
    output_file.resolve()
)

print(
    "\nComplete RIC qualitative analysis workbook "
    "created successfully."
)

RIC QUALITATIVE CODING ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\RIC_Coding_20260825_132313.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Participants:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

Number of unmapped themes: 0


RIC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 98
Cleaned theme observations: 98
Uniq

In [35]:
# ================================================================
# RIC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# RIC_FINAL_QUALITATIVE_ANALYSIS_20260821_101129.xlsx
#
# OUTPUT:
# RIC_FINAL_EVIDENCE_AND_DIMENSIONS_YYYYMMDD_HHMMSS.xlsx
#
# PURPOSE:
# 1. Preserve the completed RIC qualitative coding
# 2. Extract final RIC evidence
# 3. Organize RIC themes into evidence-based dimensions
# 4. Calculate expert prevalence
# 5. Identify themes requiring review
# 6. Produce dimension × theme evidence tables
#
# IMPORTANT:
# The dimensions are qualitative groupings for questionnaire
# development. They are NOT statistically validated dimensions.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

TARGET = "RIC_FINAL_QUALITATIVE_ANALYSIS_20260825_135724.xlsx"

possible_files = []

search_locations = [
    Path("."),
    Path.home(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop",
]

for location in search_locations:

    if location.exists():

        for ext in [".xlsx", ".xlsm", ".xls"]:

            try:
                possible_files.extend(
                    location.rglob(TARGET + ext)
                )
            except Exception:
                pass


# Remove duplicates
possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)


if len(possible_files) == 0:

    raise FileNotFoundError(
        "\nRIC input file was not found.\n\n"
        f"Expected:\n{TARGET}.xlsx\n\n"
        "Put the Excel file in the same folder as your "
        "Python notebook or change TARGET to the exact "
        "filename."
    )


INPUT_FILE = possible_files[0]


print("=" * 80)
print("RIC FINAL QUALITATIVE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE)


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():

            return s

    return None


def clean_text(x):

    if pd.isna(x):

        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:

        x = float(x)

    except:

        return "Not available"

    if x >= 75:

        return "Very High"

    elif x >= 50:

        return "High"

    elif x >= 25:

        return "Moderate"

    else:

        return "Low"


# ================================================================
# 5. IDENTIFY SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_RIC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


if final_sheet is None:

    raise ValueError(
        "11_Final_RIC_Evidence was not found."
    )


print(
    "\nFinal RIC evidence sheet:",
    final_sheet
)


# ================================================================
# 6. READ FINAL RIC EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()

final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# ================================================================
# 7. IDENTIFY RIC THEME COLUMN
# ================================================================

theme_col = None


if "Final_RIC_Theme" in final_evidence.columns:

    theme_col = "Final_RIC_Theme"


else:

    for c in final_evidence.columns:

        if (
            "RIC" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify the Final RIC Theme column."
    )


final_evidence[
    "Final_RIC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE IMPORTANT COLUMNS
# ================================================================

if (
    "Experts_Mentioning"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if (
    "Expert_Prevalence_%"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_RIC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. PARTICIPANT COUNT
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []


if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(
                cstr
            )


if len(participants) > 0:

    n_participants = len(
        participants
    )

else:

    # Study participant count
    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if (
    "Expert_Prevalence_%"
    in final_evidence.columns
):

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 12. SORT FINAL EVIDENCE
# ================================================================

if (
    "Experts_Mentioning"
    in final_evidence.columns
):

    final_evidence = (
        final_evidence
        .sort_values(
            "Experts_Mentioning",
            ascending=False
        )
        .reset_index(drop=True)
    )


# ================================================================
# 13. FINAL RIC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_RIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]


evidence_columns = [

    c
    for c in preferred_columns
    if c in final_evidence.columns

]


final_ric_evidence = final_evidence[
    evidence_columns
].copy()


# Remove existing Rank if present
if "Rank" in final_ric_evidence.columns:

    final_ric_evidence = (
        final_ric_evidence
        .drop(columns=["Rank"])
    )


final_ric_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_ric_evidence) + 1
    )
)


# ================================================================
# 14. RIC DIMENSION MAPPING
# ================================================================
#
# These dimensions are derived from the 61 RIC themes present
# in the attached workbook.
#
# ================================================================


def map_ric_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. RISK DETECTION & EARLY IDENTIFICATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "early detection",
            "early identification",
            "warning signs",
            "external warnings",
            "connecting signals",
            "signal combination",
            "signal connection",
            "monitoring"

        ]
    ):

        return (
            "Risk Detection & Early Identification"
        )


    # ------------------------------------------------------------
    # 2. RISK EXPOSURE & VULNERABILITY ASSESSMENT
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "exposure",
            "vulnerability",
            "overall exposure",
            "systemic exposure",
            "internal exposure",
            "exposure assessment",
            "contextual exposure",
            "inventory vulnerability",
            "internal vulnerabilities",
            "likelihood",
            "probability",
            "severity",
            "significance"

        ]
    ):

        return (
            "Risk Exposure & Vulnerability Assessment"
        )


    # ------------------------------------------------------------
    # 3. CONTEXTUAL RISK INTERPRETATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "contextual assessment",
            "contextual interpretation",
            "contextual risk interpretation",
            "exposure interpretation",
            "business interpretation",
            "event interpretation",
            "interpretation",
            "conditions",
            "historical patterns",
            "external developments",
            "risk development"

        ]
    ):

        return (
            "Contextual Risk Interpretation"
        )


    # ------------------------------------------------------------
    # 4. RISK INTERCONNECTEDNESS & SYSTEMIC UNDERSTANDING
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "dependencies",
            "dependency",
            "interactions",
            "relationships",
            "knock-on effects",
            "event interactions",
            "causal connections",
            "interconnected",
            "reinforcing risks",
            "combined risks",
            "exposure combinations",
            "combined indicators"

        ]
    ):

        return (
            "Risk Interconnectedness & Systemic Understanding"
        )


    # ------------------------------------------------------------
    # 5. RISK AGGREGATION & PRIORITIZATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "combined-risk assessment",
            "risk aggregation",
            "prioritization",
            "ranking",
            "resource focus",
            "alternatives",
            "multiple consequences",
            "consequences",
            "impact"

        ]
    ):

        return (
            "Risk Aggregation & Prioritization"
        )


    # ------------------------------------------------------------
    # 6. RISK INFORMATION INTEGRATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "integrated assessment",
            "inventory",
            "critical materials",
            "external developments",
            "internal vulnerabilities",
            "combined indicators"

        ]
    ):

        return (
            "Risk Information Integration"
        )


    # ------------------------------------------------------------
    # 7. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_ric_evidence[
    "RIC_Dimension"
] = (
    final_ric_evidence[
        "Final_RIC_Theme"
    ]
    .apply(
        map_ric_dimension
    )
)


# ================================================================
# 15. EXPLICIT MAPPING FOR AMBIGUOUS / DUPLICATE LABELS
# ================================================================

manual_mapping = {

    # ------------------------------------------------------------
    # Detection
    # ------------------------------------------------------------

    "early detection":
        "Risk Detection & Early Identification",

    "early identification":
        "Risk Detection & Early Identification",

    "warning signs":
        "Risk Detection & Early Identification",

    "External warnings":
        "Risk Detection & Early Identification",

    "Monitoring":
        "Risk Detection & Early Identification",

    "Connecting signals":
        "Risk Detection & Early Identification",

    "Signal combination":
        "Risk Detection & Early Identification",

    "Signal connection":
        "Risk Detection & Early Identification",


    # ------------------------------------------------------------
    # Exposure / vulnerability
    # ------------------------------------------------------------

    "Exposure":
        "Risk Exposure & Vulnerability Assessment",

    "exposure":
        "Risk Exposure & Vulnerability Assessment",

    "Exposure interpretation":
        "Contextual Risk Interpretation",

    "Contextual exposure":
        "Risk Exposure & Vulnerability Assessment",

    "Systemic exposure":
        "Risk Exposure & Vulnerability Assessment",

    "internal exposure":
        "Risk Exposure & Vulnerability Assessment",

    "exposure assessment":
        "Risk Exposure & Vulnerability Assessment",

    "inventory vulnerability":
        "Risk Exposure & Vulnerability Assessment",

    "internal vulnerabilities":
        "Risk Exposure & Vulnerability Assessment",

    "vulnerability":
        "Risk Exposure & Vulnerability Assessment",

    "vulnerabilities":
        "Risk Exposure & Vulnerability Assessment",

    "likelihood":
        "Risk Exposure & Vulnerability Assessment",

    "Likelihood":
        "Risk Exposure & Vulnerability Assessment",

    "Probability":
        "Risk Exposure & Vulnerability Assessment",

    "severity":
        "Risk Exposure & Vulnerability Assessment",

    "significance":
        "Risk Exposure & Vulnerability Assessment",

    "Overall exposure":
        "Risk Exposure & Vulnerability Assessment",


    # ------------------------------------------------------------
    # Contextual interpretation
    # ------------------------------------------------------------

    "Contextual assessment":
        "Contextual Risk Interpretation",

    "contextual assessment":
        "Contextual Risk Interpretation",

    "Contextual interpretation":
        "Contextual Risk Interpretation",

    "contextual interpretation":
        "Contextual Risk Interpretation",

    "Contextual risk interpretation":
        "Contextual Risk Interpretation",

    "Business interpretation":
        "Contextual Risk Interpretation",

    "Event interpretation":
        "Contextual Risk Interpretation",

    "interpretation":
        "Contextual Risk Interpretation",

    "conditions":
        "Contextual Risk Interpretation",

    "historical patterns":
        "Contextual Risk Interpretation",

    "Risk development":
        "Contextual Risk Interpretation",


    # ------------------------------------------------------------
    # Interconnectedness
    # ------------------------------------------------------------

    "dependencies":
        "Risk Interconnectedness & Systemic Understanding",

    "dependency":
        "Risk Interconnectedness & Systemic Understanding",

    "interactions":
        "Risk Interconnectedness & Systemic Understanding",

    "relationships":
        "Risk Interconnectedness & Systemic Understanding",

    "knock-on effects":
        "Risk Interconnectedness & Systemic Understanding",

    "Event interactions":
        "Risk Interconnectedness & Systemic Understanding",

    "causal connections":
        "Risk Interconnectedness & Systemic Understanding",

    "Interconnected and reinforcing risks":
        "Risk Interconnectedness & Systemic Understanding",

    "combined risks":
        "Risk Interconnectedness & Systemic Understanding",

    "exposure combinations":
        "Risk Interconnectedness & Systemic Understanding",

    "combined indicators":
        "Risk Interconnectedness & Systemic Understanding",


    # ------------------------------------------------------------
    # Aggregation / prioritization
    # ------------------------------------------------------------

    "prioritization":
        "Risk Aggregation & Prioritization",

    "ranking":
        "Risk Aggregation & Prioritization",

    "resource focus":
        "Risk Aggregation & Prioritization",

    "alternatives":
        "Risk Aggregation & Prioritization",

    "Consequences":
        "Risk Aggregation & Prioritization",

    "consequences":
        "Risk Aggregation & Prioritization",

    "impact":
        "Risk Aggregation & Prioritization",

    "multiple consequences":
        "Risk Aggregation & Prioritization",

    "risk aggregation":
        "Risk Aggregation & Prioritization",

    "combined-risk assessment":
        "Risk Aggregation & Prioritization",


    # ------------------------------------------------------------
    # Information integration
    # ------------------------------------------------------------

    "integrated assessment":
        "Risk Information Integration",

    "inventory":
        "Risk Information Integration",

    "critical materials":
        "Risk Information Integration",

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_ric_evidence[
            "Final_RIC_Theme"
        ]
        .astype(str)
        .str.strip()
        .eq(theme)
    )

    final_ric_evidence.loc[
        mask,
        "RIC_Dimension"
    ] = dimension


# ================================================================
# 16. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_ric_evidence
    .groupby(
        "RIC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_RIC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # ------------------------------------------------------------
    # Calculate expert prevalence
    # ------------------------------------------------------------

    experts_mentioning = 0


    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        temp = participant_matrix.copy()

        # Identify theme column
        theme_candidates = [
            c
            for c in temp.columns
            if "theme" in str(c).lower()
        ]

        if len(theme_candidates) > 0:

            matrix_theme_col = (
                theme_candidates[0]
            )

        else:

            matrix_theme_col = (
                temp.columns[0]
            )


        temp["_Theme"] = (
            temp[
                matrix_theme_col
            ]
            .astype(str)
            .str.strip()
        )


        theme_lower = {
            str(x).strip().lower()
            for x in themes
        }


        matched = temp[
            temp[
                "_Theme"
            ]
            .str.lower()
            .isin(
                theme_lower
            )
        ]


        available_participants = [

            p

            for p in participants

            if p in matched.columns

        ]


        if (
            len(matched) > 0
            and len(available_participants) > 0
        ):

            vals = (
                matched[
                    available_participants
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce"
                )
                .fillna(0)
            )


            experts_mentioning = int(
                (
                    vals.sum(axis=0) > 0
                ).sum()
            )


    # ------------------------------------------------------------
    # Fallback to final evidence
    # ------------------------------------------------------------

    if experts_mentioning == 0:

        if (
            "Experts_Mentioning"
            in group.columns
        ):

            experts_mentioning = int(
                group[
                    "Experts_Mentioning"
                ]
                .max()
            )


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "RIC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_RIC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 17. SORT DIMENSION SUMMARY
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(columns=["Rank"])
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 18. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "RIC_Dimension",

    "Final_RIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [
    c
    for c in dimension_theme_columns
    if c in final_ric_evidence.columns
]


dimension_themes = final_ric_evidence[
    dimension_theme_columns
].copy()


dimension_themes = (
    dimension_themes
    .sort_values(
        [
            "RIC_Dimension",
            "Experts_Mentioning"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ================================================================
# 19. REVIEW-REQUIRED THEMES
# ================================================================

unmapped_check = final_ric_evidence[
    final_ric_evidence[
        "RIC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    unmapped_check = unmapped_check[
        [
            "Final_RIC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ]
    ].copy()


    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )


else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All RIC themes mapped to a dimension."
        ]

    })


# ================================================================
# 20. DECISION CHECK
# ================================================================

coding_audit = None

if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


if coding_audit is not None:

    decision_columns = [

        c

        for c in coding_audit.columns

        if "Decision" in str(c)

    ]


    if len(decision_columns) > 0:

        decision_col = (
            decision_columns[0]
        )


        missing = coding_audit[
            coding_audit[
                decision_col
            ].isna()
        ]


        if len(missing) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All RIC coding decisions are present."
                ]

            })

        else:

            decision_check = missing.copy()


    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })


else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit unavailable."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final RIC evidence available",

    "Result":
        "PASS"
        if len(final_ric_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_ric_evidence)} final RIC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "RIC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative dimensions"

})


review_count = (

    len(unmapped_check)

    if "Final_RIC_Theme"
    in unmapped_check.columns

    else 0

)


quality_rows.append({

    "Quality_Check":
        "Themes requiring review",

    "Result":
        review_count,

    "Details":
        "Themes assigned to Review Required"

})


quality_rows.append({

    "Quality_Check":
        "Duplicate theme labels",

    "Result":
        int(
            final_ric_evidence[
                "Final_RIC_Theme"
            ]
            .str.lower()
            .duplicated()
            .sum()
        ),

    "Details":
        "Case-insensitive duplicate labels detected"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:

        return sheets[
            sheet
        ].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. WRITE OUTPUT WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"RIC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    if coding_audit is not None:

        coding_audit.to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )


    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    # ------------------------------------------------------------
    # FINAL RIC EVIDENCE
    # ------------------------------------------------------------

    final_ric_evidence.to_excel(
        writer,
        sheet_name="13_Final_RIC_Evidence",
        index=False
    )


    # ------------------------------------------------------------
    # RIC DIMENSION SUMMARY
    # ------------------------------------------------------------

    dimension_summary.to_excel(
        writer,
        sheet_name="14_RIC_Dimension_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # RIC DIMENSION × THEME
    # ------------------------------------------------------------

    dimension_themes.to_excel(
        writer,
        sheet_name="15_RIC_Dimension_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # CONSTRUCT STATISTICS
    # ------------------------------------------------------------

    construct_statistics.to_excel(
        writer,
        sheet_name="16_Construct_Statistics",
        index=False
    )


# ================================================================
# 24. PRINT RESULTS
# ================================================================

print("\n")
print("=" * 80)
print("RIC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 80)


print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)


print("\n")
print("-" * 80)
print("FINAL RIC EVIDENCE")
print("-" * 80)

print(
    final_ric_evidence.to_string(
        index=False
    )
)


print("\n")
print("-" * 80)
print("RIC DIMENSION SUMMARY")
print("-" * 80)

print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("OUTPUT SHEETS")
print("=" * 80)

print("""
01_Original_Responses
02_Raw_Themes
03_Cleaned_Themes
04_Coding_Audit
05_Participant_Matrix
06_Theme_Summary
07_Normalization_Summary
08_Decision_Summary
09_Participant_Coverage
10_Unmapped_Check
11_Decision_Check
12_Quality_Checks
13_Final_RIC_Evidence
14_RIC_Dimension_Summary
15_RIC_Dimension_Themes
16_Construct_Statistics
""")


print("\n")
print("=" * 80)
print("NEXT STAGE")
print("=" * 80)

print("""
For questionnaire development, the two important sheets are:

14_RIC_Dimension_Summary
15_RIC_Dimension_Themes

These provide the bridge from:

Expert responses
      ↓
Raw themes
      ↓
Normalized themes
      ↓
Final RIC themes
      ↓
RIC qualitative dimensions
      ↓
Candidate questionnaire items

The dimensions are qualitative evidence-based groupings.
They are NOT yet statistically validated measurement
dimensions.
""")

RIC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\RIC_FINAL_QUALITATIVE_ANALYSIS_20260825_135724.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_RIC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Final RIC evidence sheet: 11_Final_RIC_Evidence

Participants used: 26


RIC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\RIC_FINAL_EVIDENCE_AND_DIMENSIONS_20260825_144545.xlsx


--------------------------------------------------------------------------------
FINAL RIC EVIDENCE
-------------------------------------------------------------------------

## ROC

In [6]:
# ================================================================
# ROC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT ROC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "ROC":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "ROC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No ROC responses were found.

        Check that ROC appears in the participant sheets.
        """
    )

print("\nROC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "ROC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)

print("\nROC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW ROC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("ROC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. ROC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# Add researcher-approved ROC mappings here after reviewing
# the actual ROC raw themes.
#
# Example format:
#
# "coordinated risk response":
#     "Coordinated risk response",
#
# "risk response coordination":
#     "Coordinated risk response",
#
# ------------------------------------------------

ROC_NORMALIZATION = {

    # ADD ROC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in ROC_NORMALIZATION:

        normalized = ROC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × ROC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "ROC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "ROC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("ROC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique ROC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized ROC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

ROC original responses:
26

ROC cleaned theme observations:
89


ROC RAW THEMES
                               Raw_Theme                              Theme_Clean                                Theme_Key
                              adaptation                               adaptation                               adaptation
                     Adaptive allocation                      Adaptive allocation                      adaptive allocation
                              adjustment                               adjustment                               adjustment
                               alignment                                alignment                                alignment
                           

In [20]:
# ================================================================
# COMPLETE ROC QUALITATIVE CODING ANALYSIS
# Input: ROC_Coding_20260825_132506.xlsx
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path("ROC_Coding_20260825_132506.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as your "
        "Python notebook."
    )

print("=" * 70)
print("ROC QUALITATIVE CODING ANALYSIS")
print("=" * 70)
print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 3. FLEXIBLE SHEET FINDER
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_ROC",
    "01_Original",
    "Original_ROC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_ROC",
    "02_Raw",
    "Raw_ROC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_ROC",
    "03_Cleaned",
    "Cleaned_ROC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 4. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(
        input_file,
        sheet_name=original_sheet
    )
    if original_sheet
    else pd.DataFrame()
)

raw_df = (
    pd.read_excel(
        input_file,
        sheet_name=raw_sheet
    )
    if raw_sheet
    else pd.DataFrame()
)

clean_df = (
    pd.read_excel(
        input_file,
        sheet_name=clean_sheet
    )
    if clean_sheet
    else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the ROC Coding/Normalized Coding sheet.\n"
        "Available sheets are:\n"
        + "\n".join(
            str(x)
            for x in excel.sheet_names
        )
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 6. COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

raw_theme_col = find_column(
    coding_df,
    [
        "Raw_Theme",
        "Raw Theme",
        "RawTheme"
    ]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    [
        "Decision"
    ]
)


if theme_key_col is None:
    raise ValueError(
        "\nTheme_Key column not found."
    )

if raw_theme_col is None:
    raise ValueError(
        "\nRaw_Theme column not found."
    )

if normalized_col is None:
    raise ValueError(
        "\nNormalized_Theme column not found."
    )

if decision_col is None:
    raise ValueError(
        "\nDecision column not found."
    )


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. FIND PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:

    raise ValueError(
        "\nParticipant column not found in Cleaned ROC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col:
            "Participant"
        }
    )


# ================================================================
# 8. FIND THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key not found in Cleaned ROC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key:
            "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        [
            "nan",
            "None",
            "",
            "NaN"
        ],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANT LIST
# ================================================================

participants = sorted(
    clean_df[
        "Participant"
    ]
    .replace("", np.nan)
    .dropna()
    .unique()
)

print("\nParticipants:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 12. MERGE CLEANED DATA WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. CHECK UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df[
        "Normalized_Theme"
    ].isna()
].copy()


print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)


if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED ROC THEME
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme[
        "Normalized_Theme"
    ],
    participant_theme[
        "Participant"
    ]
)


# ================================================================
# 16. ORDER PARTICIPANTS
# ================================================================

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()

participant_columns = (
    existing + other
)


# ================================================================
# 17. FREQUENCY
# ================================================================

matrix["Frequency"] = (
    matrix[
        participant_columns
    ]
    .sum(axis=1)
)


# ================================================================
# 18. PERCENTAGE OF PARTICIPANTS
# ================================================================

total_participants = len(participants)

if total_participants > 0:

    matrix["Percentage"] = (
        matrix["Frequency"]
        / total_participants
        * 100
    ).round(1)

else:

    matrix["Percentage"] = 0


# ================================================================
# 19. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 20. ROC THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_ROC_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 21. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 22. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby(
        "Normalized_Theme"
    )
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit[
    "Experts_Mentioning"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


if total_participants > 0:

    coding_audit[
        "Percentage_of_Experts"
    ] = (
        coding_audit[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    coding_audit[
        "Percentage_of_Experts"
    ] = 0


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 23. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


if len(coding_df) > 0:

    decision_summary[
        "Percentage"
    ] = (
        decision_summary[
            "Number_of_Raw_Themes"
        ]
        / len(coding_df)
        * 100
    ).round(1)

else:

    decision_summary[
        "Percentage"
    ] = 0


# ================================================================
# 24. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=[
            "Normalized_Theme"
        ]
    )
    .groupby(
        "Normalized_Theme"
    )
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


if total_participants > 0:

    normalization_summary[
        "Percentage_of_Experts"
    ] = (
        normalization_summary[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    normalization_summary[
        "Percentage_of_Experts"
    ] = 0


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 25. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_ROC_Themes"
]


# ================================================================
# 26. ROC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": [
        "ROC"
    ],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df[
            "Theme_Key"
        ].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 27. FINAL ROC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_ROC_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 28. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)


quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",
        "Original response rows",
        "Raw theme observations",
        "Unique raw themes",
        "Final normalized themes",
        "Unmapped themes",
        "Duplicate participant-theme records",
        "Keep decisions",
        "Merge decisions"

    ],

    "Result": [

        total_participants,
        len(original_df),
        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df[
                "Decision"
            ] == "Keep"
        ).sum(),

        (
            coding_df[
                "Decision"
            ] == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",
        "PASS",
        "PASS"

    ]
})


# ================================================================
# 29. SAVE COMPLETE ROC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"ROC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


# Avoid overwrite / PermissionError

counter = 1

while output_file.exists():

    output_file = Path(
        f"ROC_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_ROC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 30. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("ROC ANALYSIS COMPLETED")
print("=" * 70)

print("\nParticipants:", total_participants)

print("Original responses:", len(original_df))

print("Raw theme observations:", len(raw_df))

print("Cleaned theme observations:", len(clean_df))

print(
    "Unique raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Final normalized ROC themes:",
    coding_df["Normalized_Theme"].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df["Decision"] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df["Decision"] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 31. DISPLAY FINAL ROC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL ROC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_ROC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 32. OUTPUT FILE
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(output_file.resolve())

print(
    "\nComplete ROC qualitative analysis workbook "
    "created successfully."
)

ROC QUALITATIVE CODING ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\ROC_Coding_20260825_132506.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Participants:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

Number of unmapped themes: 0


ROC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 89
Cleaned theme observations: 89
Uniq

In [38]:
# ================================================================
# ROC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# ROC_FINAL_QUALITATIVE_ANALYSIS_20260821_101408.xlsx
#
# OUTPUT:
# ROC_FINAL_EVIDENCE_AND_DIMENSIONS_YYYYMMDD_HHMMSS.xlsx
#
# PURPOSE:
# Finalize ROC qualitative evidence and organize the themes
# into evidence-based dimensions for questionnaire development.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

TARGET = "ROC_FINAL_QUALITATIVE_ANALYSIS_20260825_135918"

possible_files = []

# First check the current working directory and common folders
search_locations = [
    Path("."),
    Path.home(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop",
]

for location in search_locations:
    if location.exists():
        for ext in [".xlsx", ".xlsm", ".xls"]:
            try:
                possible_files.extend(
                    location.rglob(TARGET + ext)
                )
            except Exception:
                pass

# Also check /mnt/data for uploaded/copied files
try:
    possible_files.extend(
        Path("/mnt/data").glob("*.xlsx")
    )
except Exception:
    pass

possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)

# Prefer an exact filename match
exact_files = [
    p for p in possible_files
    if p.stem == TARGET
]

if len(exact_files) > 0:
    INPUT_FILE = exact_files[0]

else:

    # If the original filename has been renamed by the upload system,
    # identify the workbook by its ROC-specific sheet.
    roc_candidates = []

    for p in possible_files:

        try:
            xl = pd.ExcelFile(
                p,
                engine="openpyxl"
            )

            if "11_Final_ROC_Evidence" in xl.sheet_names:
                roc_candidates.append(p)

        except Exception:
            pass

    if len(roc_candidates) > 0:
        INPUT_FILE = roc_candidates[0]

    else:
        raise FileNotFoundError(
            "\nROC input workbook could not be found.\n\n"
            "Expected filename:\n"
            f"{TARGET}.xlsx"
        )


print("=" * 80)
print("ROC FINAL QUALITATIVE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE)


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():
            return s

    return None


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:
        x = float(x)
    except:
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


# ================================================================
# 5. IDENTIFY SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_ROC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


if final_sheet is None:

    raise ValueError(
        "11_Final_ROC_Evidence was not found."
    )


# ================================================================
# 6. READ FINAL ROC EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()

final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# ================================================================
# 7. FIND ROC THEME COLUMN
# ================================================================

theme_col = None

if "Final_ROC_Theme" in final_evidence.columns:

    theme_col = "Final_ROC_Theme"

else:

    for c in final_evidence.columns:

        if (
            "ROC" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify the final ROC theme column."
    )


final_evidence[
    "Final_ROC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE EXPERT COLUMNS
# ================================================================

if "Experts_Mentioning" not in final_evidence.columns:

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if "Expert_Prevalence_%" not in final_evidence.columns:

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_ROC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. PARTICIPANT COUNT
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []

if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(cstr)


# Your qualitative dataset has 26 participants
if len(participants) > 0:

    n_participants = len(participants)

else:

    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(prevalence_category)
    )


# ================================================================
# 12. SORT FINAL EVIDENCE
# ================================================================

if "Experts_Mentioning" in final_evidence.columns:

    final_evidence = (
        final_evidence
        .sort_values(
            "Experts_Mentioning",
            ascending=False
        )
        .reset_index(drop=True)
    )


# ================================================================
# 13. FINAL ROC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_ROC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]

evidence_columns = [
    c for c in preferred_columns
    if c in final_evidence.columns
]

final_roc_evidence = final_evidence[
    evidence_columns
].copy()


# Prevent Rank duplication
if "Rank" in final_roc_evidence.columns:

    final_roc_evidence = (
        final_roc_evidence
        .drop(columns=["Rank"])
    )


final_roc_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_roc_evidence) + 1
    )
)


# ================================================================
# 14. ROC DIMENSION MAPPING
# ================================================================
#
# These dimensions are intended for questionnaire-development
# purposes. They organize the final ROC themes according to
# what the experts describe as risk orchestration activities.
#
# They are NOT yet statistically validated subdimensions.
# ================================================================

def map_roc_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. RISK RESPONSE COORDINATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "coordination",
            "coordinating",
            "cross-functional",
            "cross functional",
            "interdepartmental",
            "between departments",
            "between functions",
            "stakeholder coordination",
            "supplier coordination",
            "partner coordination"

        ]
    ):

        return (
            "Risk Response Coordination"
        )


    # ------------------------------------------------------------
    # 2. RISK RESPONSE PRIORITIZATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "prioritization",
            "prioritisation",
            "priority",
            "prioritize",
            "prioritise",
            "resource allocation",
            "resource allocation decisions",
            "critical risks",
            "most critical"

        ]
    ):

        return (
            "Risk Response Prioritization"
        )


    # ------------------------------------------------------------
    # 3. RISK RESPONSE INTEGRATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "integrated response",
            "integrated action",
            "integrating",
            "integration",
            "combined response",
            "coordinated response",
            "holistic response",
            "response across"

        ]
    ):

        return (
            "Integrated Risk Response"
        )


    # ------------------------------------------------------------
    # 4. RESOURCE & ACTION ORCHESTRATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "resource",
            "capacity",
            "reallocate",
            "reallocation",
            "deploy",
            "deployment",
            "mobilize",
            "mobilise",
            "action",
            "response action",
            "corrective action"

        ]
    ):

        return (
            "Resource & Action Orchestration"
        )


    # ------------------------------------------------------------
    # 5. ADAPTIVE RISK RESPONSE
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "adapt",
            "adaptive",
            "adjust",
            "adjustment",
            "flexibility",
            "flexible response",
            "changing conditions",
            "changing circumstances",
            "dynamic response",
            "respond to changes"

        ]
    ):

        return (
            "Adaptive Risk Response"
        )


    # ------------------------------------------------------------
    # 6. RISK RESPONSE MONITORING & FOLLOW-UP
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "monitor response",
            "monitoring response",
            "follow-up",
            "follow up",
            "tracking",
            "response effectiveness",
            "response performance",
            "evaluate response",
            "review response"

        ]
    ):

        return (
            "Risk Response Monitoring & Follow-up"
        )


    # ------------------------------------------------------------
    # 7. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_roc_evidence[
    "ROC_Dimension"
] = (
    final_roc_evidence[
        "Final_ROC_Theme"
    ]
    .apply(
        map_roc_dimension
    )
)


# ================================================================
# 15. ADDITIONAL MANUAL MAPPING
# ================================================================
#
# Only explicit labels are mapped here.
# Anything not confidently mapped remains Review Required.
# ================================================================

manual_mapping = {

    # Coordination
    "coordination":
        "Risk Response Coordination",

    "risk coordination":
        "Risk Response Coordination",

    "cross-functional coordination":
        "Risk Response Coordination",

    "cross functional coordination":
        "Risk Response Coordination",

    "stakeholder coordination":
        "Risk Response Coordination",

    "supplier coordination":
        "Risk Response Coordination",

    "partner coordination":
        "Risk Response Coordination",


    # Prioritization
    "prioritization":
        "Risk Response Prioritization",

    "prioritisation":
        "Risk Response Prioritization",

    "priority":
        "Risk Response Prioritization",

    "resource allocation":
        "Risk Response Prioritization",

    "critical risks":
        "Risk Response Prioritization",


    # Integration
    "integration":
        "Integrated Risk Response",

    "integrated response":
        "Integrated Risk Response",

    "integrated action":
        "Integrated Risk Response",

    "combined response":
        "Integrated Risk Response",

    "holistic response":
        "Integrated Risk Response",


    # Resource / action
    "resource":
        "Resource & Action Orchestration",

    "resource allocation decisions":
        "Resource & Action Orchestration",

    "reallocation":
        "Resource & Action Orchestration",

    "resource reallocation":
        "Resource & Action Orchestration",

    "deployment":
        "Resource & Action Orchestration",

    "mobilization":
        "Resource & Action Orchestration",

    "mobilisation":
        "Resource & Action Orchestration",


    # Adaptation
    "adaptation":
        "Adaptive Risk Response",

    "adaptive response":
        "Adaptive Risk Response",

    "flexibility":
        "Adaptive Risk Response",

    "flexible response":
        "Adaptive Risk Response",

    "dynamic response":
        "Adaptive Risk Response",


    # Monitoring
    "response monitoring":
        "Risk Response Monitoring & Follow-up",

    "monitoring response":
        "Risk Response Monitoring & Follow-up",

    "follow-up":
        "Risk Response Monitoring & Follow-up",

    "follow up":
        "Risk Response Monitoring & Follow-up",

    "response effectiveness":
        "Risk Response Monitoring & Follow-up",

    "response performance":
        "Risk Response Monitoring & Follow-up"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_roc_evidence[
            "Final_ROC_Theme"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            theme.lower()
        )
    )

    final_roc_evidence.loc[
        mask,
        "ROC_Dimension"
    ] = dimension


# ================================================================
# 16. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_roc_evidence
    .groupby(
        "ROC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_ROC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    experts_mentioning = 0


    # ------------------------------------------------------------
    # Use participant matrix when available
    # ------------------------------------------------------------

    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        temp = participant_matrix.copy()


        theme_candidates = [
            c
            for c in temp.columns
            if "theme" in str(c).lower()
        ]


        if len(theme_candidates) > 0:

            matrix_theme_col = (
                theme_candidates[0]
            )

        else:

            matrix_theme_col = (
                temp.columns[0]
            )


        temp["_Theme"] = (
            temp[
                matrix_theme_col
            ]
            .astype(str)
            .str.strip()
        )


        theme_lower = {
            str(x).strip().lower()
            for x in themes
        }


        matched = temp[
            temp[
                "_Theme"
            ]
            .str.lower()
            .isin(
                theme_lower
            )
        ]


        available_participants = [

            p

            for p in participants

            if p in matched.columns

        ]


        if (
            len(matched) > 0
            and len(available_participants) > 0
        ):

            vals = (
                matched[
                    available_participants
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce"
                )
                .fillna(0)
            )


            experts_mentioning = int(
                (
                    vals.sum(axis=0) > 0
                ).sum()
            )


    # ------------------------------------------------------------
    # Fallback
    # ------------------------------------------------------------

    if experts_mentioning == 0:

        if (
            "Experts_Mentioning"
            in group.columns
        ):

            try:

                experts_mentioning = int(
                    pd.to_numeric(
                        group[
                            "Experts_Mentioning"
                        ],
                        errors="coerce"
                    )
                    .max()
                )

            except:

                experts_mentioning = 0


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "ROC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_ROC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 17. SORT DIMENSIONS
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(columns=["Rank"])
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 18. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "ROC_Dimension",

    "Final_ROC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [
    c
    for c in dimension_theme_columns
    if c in final_roc_evidence.columns
]


dimension_themes = final_roc_evidence[
    dimension_theme_columns
].copy()


dimension_themes = (
    dimension_themes
    .sort_values(
        [
            "ROC_Dimension",
            "Experts_Mentioning"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ================================================================
# 19. REVIEW-REQUIRED THEMES
# ================================================================

unmapped_check = final_roc_evidence[
    final_roc_evidence[
        "ROC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    unmapped_check = unmapped_check[
        [
            "Final_ROC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ]
    ].copy()


    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )

else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All ROC themes mapped to a dimension."
        ]

    })


# ================================================================
# 20. CODING DECISION CHECK
# ================================================================

coding_audit = None

if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


if coding_audit is not None:

    decision_columns = [

        c
        for c in coding_audit.columns
        if "Decision" in str(c)

    ]


    if len(decision_columns) > 0:

        decision_col = (
            decision_columns[0]
        )


        missing = coding_audit[
            coding_audit[
                decision_col
            ].isna()
        ]


        if len(missing) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All ROC coding decisions are present."
                ]

            })

        else:

            decision_check = missing.copy()


    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })

else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit unavailable."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final ROC evidence available",

    "Result":
        "PASS"
        if len(final_roc_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_roc_evidence)} final ROC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "ROC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative evidence-based dimensions"

})


review_count = (

    len(unmapped_check)

    if "Final_ROC_Theme"
    in unmapped_check.columns

    else 0

)


quality_rows.append({

    "Quality_Check":
        "Themes requiring review",

    "Result":
        review_count,

    "Details":
        "Themes assigned to Review Required"

})


duplicate_count = int(
    final_roc_evidence[
        "Final_ROC_Theme"
    ]
    .str.lower()
    .duplicated()
    .sum()
)


quality_rows.append({

    "Quality_Check":
        "Duplicate theme labels",

    "Result":
        duplicate_count,

    "Details":
        "Case-insensitive duplicate labels"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:
        return sheets[sheet].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. WRITE OUTPUT
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"ROC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    if coding_audit is not None:

        coding_audit.to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )


    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    final_roc_evidence.to_excel(
        writer,
        sheet_name="13_Final_ROC_Evidence",
        index=False
    )


    dimension_summary.to_excel(
        writer,
        sheet_name="14_ROC_Dimension_Summary",
        index=False
    )


    dimension_themes.to_excel(
        writer,
        sheet_name="15_ROC_Dimension_Themes",
        index=False
    )


    construct_statistics.to_excel(
        writer,
        sheet_name="16_Construct_Statistics",
        index=False
    )


# ================================================================
# 24. FINAL REPORT
# ================================================================

print("\n")
print("=" * 80)
print("ROC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 80)

print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)


print("\nNumber of final ROC themes:")
print(
    len(final_roc_evidence)
)


print("\nNumber of ROC dimensions:")
print(
    len(dimension_summary)
)


print("\n")
print("-" * 80)
print("ROC DIMENSION SUMMARY")
print("-" * 80)

print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("-" * 80)
print("QUALITY CHECKS")
print("-" * 80)

print(
    quality_checks_new.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("IMPORTANT")
print("=" * 80)

print("""
The two sheets you should use for the next questionnaire stage are:

14_ROC_Dimension_Summary
15_ROC_Dimension_Themes

They connect your qualitative evidence to questionnaire-item
development.

The dimensions are NOT automatically treated as separate
statistical constructs. They are qualitative evidence-based
groupings that help us determine what aspects of ROC should
be represented in the questionnaire.

The next stage is to convert the strongest, non-overlapping
ROC themes into candidate reflective questionnaire items.
""")

ROC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\ROC_FINAL_QUALITATIVE_ANALYSIS_20260825_135918.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_ROC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Participants used: 26


ROC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\ROC_FINAL_EVIDENCE_AND_DIMENSIONS_20260825_145325.xlsx

Number of final ROC themes:
62

Number of ROC dimensions:
6


--------------------------------------------------------------------------------
ROC DIMENSION SUMMARY
----------------------------------------------------------

## RCC

In [7]:
# ================================================================
# RCC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT RCC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "RCC":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "RCC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No RCC responses were found.

        Check that RCC appears in the participant sheets.
        """
    )

print("\nRCC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "RCC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)

print("\nRCC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW RCC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("RCC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. RCC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# Add researcher-approved RCC mappings here after reviewing
# the actual RCC raw themes.
#
# Example format:
#
# "coordinated risk actions":
#     "Coordinated risk response",
#
# "joint risk response":
#     "Coordinated risk response",
#
# ------------------------------------------------

RCC_NORMALIZATION = {

    # ADD RCC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in RCC_NORMALIZATION:

        normalized = RCC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × RCC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "RCC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "RCC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("RCC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique RCC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized RCC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

RCC original responses:
26

RCC cleaned theme observations:
94


RCC RAW THEMES
                      Raw_Theme                     Theme_Clean                       Theme_Key
               Action agreement                Action agreement                action agreement
               action alignment                action alignment                action alignment
                 action clarity                  action clarity                  action clarity
                      alignment                       alignment                       alignment
                      authority                       authority                       authority
            changing priorities             changing priorities   

In [22]:
# ================================================================
# COMPLETE RCC QUALITATIVE CODING ANALYSIS
# Input: RCC_Coding_20260825_132706.xlsx
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path("RCC_Coding_20260825_132706.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as this notebook."
    )

print("=" * 70)
print("RCC QUALITATIVE CODING ANALYSIS")
print("=" * 70)

excel = pd.ExcelFile(input_file)

print("\nInput file:", input_file.resolve())
print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 2. FIND SHEETS FLEXIBLY
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_RCC",
    "01_Original",
    "Original_RCC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_RCC",
    "02_Raw",
    "Raw_RCC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_RCC",
    "03_Cleaned",
    "Cleaned_RCC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 3. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(input_file, sheet_name=original_sheet)
    if original_sheet else pd.DataFrame()
)

raw_df = (
    pd.read_excel(input_file, sheet_name=raw_sheet)
    if raw_sheet else pd.DataFrame()
)

clean_df = (
    pd.read_excel(input_file, sheet_name=clean_sheet)
    if clean_sheet else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCoding/Normalized Coding sheet could not be found.\n\n"
        "Available sheets:\n" +
        "\n".join(str(x) for x in excel.sheet_names)
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 4. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:
    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 5. FIND COLUMNS
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    ["Theme_Key", "Theme Key", "ThemeKey"]
)

raw_theme_col = find_column(
    coding_df,
    ["Raw_Theme", "Raw Theme", "RawTheme"]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    ["Decision"]
)


if theme_key_col is None:
    raise ValueError("Theme_Key column not found.")

if raw_theme_col is None:
    raise ValueError("Raw_Theme column not found.")

if normalized_col is None:
    raise ValueError("Normalized_Theme column not found.")

if decision_col is None:
    raise ValueError("Decision column not found.")


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 6. PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:

    raise ValueError(
        "\nParticipant column not found.\n\n"
        "Available columns:\n" +
        str(list(clean_df.columns))
    )

if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col: "Participant"
        }
    )


# ================================================================
# 7. THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key not found in Cleaned sheet.\n\n"
        "Available columns:\n" +
        str(list(clean_df.columns))
    )

if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key: "Theme_Key"
        }
    )


# ================================================================
# 8. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        ["nan", "None", "", "NaN"],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 9. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df["Participant"]
    .replace("", np.nan)
    .dropna()
    .unique()
)

print("\nNumber of participants:", len(participants))
print("Participants:", participants)


# ================================================================
# 11. MERGE CLEANED THEMES WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 12. UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df["Normalized_Theme"].isna()
].copy()

print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)

if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 13. PARTICIPANT × NORMALIZED RCC THEMES
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 14. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme["Normalized_Theme"],
    participant_theme["Participant"]
)


# ================================================================
# 15. ORDER PARTICIPANTS P01–P26
# ================================================================

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()

participant_columns = existing + other


# ================================================================
# 16. FREQUENCY
# ================================================================

matrix["Frequency"] = (
    matrix[participant_columns]
    .sum(axis=1)
)


# ================================================================
# 17. PERCENTAGE OF EXPERTS
# ================================================================

total_participants = len(participants)

if total_participants > 0:

    matrix["Percentage"] = (
        matrix["Frequency"]
        / total_participants
        * 100
    ).round(1)

else:

    matrix["Percentage"] = 0


# ================================================================
# 18. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)

matrix.insert(
    0,
    "Rank",
    range(1, len(matrix) + 1)
)


# ================================================================
# 19. RCC THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()

theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme": "Final_RCC_Theme",
        "Frequency": "Experts_Mentioning",
        "Percentage": "Percentage_of_Experts"
    }
)


# ================================================================
# 20. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary["Prevalence_Category"] = (
    theme_summary["Percentage_of_Experts"]
    .apply(prevalence_category)
)


# ================================================================
# 21. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby("Normalized_Theme")
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)

coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)

coding_audit["Experts_Mentioning"] = (
    coding_audit["Experts_Mentioning"]
    .fillna(0)
    .astype(int)
)

if total_participants > 0:

    coding_audit["Percentage_of_Experts"] = (
        coding_audit["Experts_Mentioning"]
        / total_participants
        * 100
    ).round(1)

else:

    coding_audit["Percentage_of_Experts"] = 0


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 22. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)

if len(coding_df) > 0:

    decision_summary["Percentage"] = (
        decision_summary["Number_of_Raw_Themes"]
        / len(coding_df)
        * 100
    ).round(1)

else:

    decision_summary["Percentage"] = 0


# ================================================================
# 23. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(subset=["Normalized_Theme"])
    .groupby("Normalized_Theme")
    .agg(
        Raw_Themes=("Raw_Theme", "count"),

        Keep_Count=(
            "Decision",
            lambda x: (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x: (x == "Merge").sum()
        )
    )
    .reset_index()
)

normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)

normalization_summary["Experts_Mentioning"] = (
    normalization_summary["Experts_Mentioning"]
    .fillna(0)
    .astype(int)
)

if total_participants > 0:

    normalization_summary["Percentage_of_Experts"] = (
        normalization_summary["Experts_Mentioning"]
        / total_participants
        * 100
    ).round(1)

else:

    normalization_summary["Percentage_of_Experts"] = 0


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 24. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_RCC_Themes"
]


# ================================================================
# 25. RCC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": ["RCC"],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df["Theme_Key"].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df["Normalized_Theme"].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df["Decision"] == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df["Decision"] == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 26. FINAL RCC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()

final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme": "Final_RCC_Theme",
        "Raw_Themes": "Number_of_Raw_Themes",
        "Percentage_of_Experts": "Expert_Prevalence_%",
        "Keep_Count": "Raw_Themes_Kept",
        "Merge_Count": "Raw_Themes_Merged"
    }
)

final_evidence["Prevalence_Category"] = (
    final_evidence["Expert_Prevalence_%"]
    .apply(prevalence_category)
)


# ================================================================
# 27. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)

quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",
        "Original response rows",
        "Raw theme observations",
        "Unique raw themes",
        "Final normalized themes",
        "Unmapped themes",
        "Duplicate participant-theme records",
        "Keep decisions",
        "Merge decisions"

    ],

    "Result": [

        total_participants,
        len(original_df),
        len(raw_df),

        coding_df["Theme_Key"].nunique(),

        coding_df["Normalized_Theme"].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df["Decision"] == "Keep"
        ).sum(),

        (
            coding_df["Decision"] == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS" if total_participants > 0 else "CHECK",

        "PASS" if len(original_df) > 0 else "CHECK",

        "PASS" if len(raw_df) > 0 else "CHECK",

        "PASS"
        if coding_df["Theme_Key"].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df["Normalized_Theme"].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",
        "PASS",
        "PASS"
    ]
})


# ================================================================
# 28. SAVE COMPLETE RCC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"RCC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)

counter = 1

while output_file.exists():

    output_file = Path(
        f"RCC_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_RCC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 29. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("RCC ANALYSIS COMPLETED")
print("=" * 70)

print("\nParticipants:", total_participants)

print("Original responses:", len(original_df))

print("Raw theme observations:", len(raw_df))

print("Cleaned theme observations:", len(clean_df))

print(
    "Unique raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Final normalized RCC themes:",
    coding_df["Normalized_Theme"].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df["Decision"] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df["Decision"] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 30. DISPLAY FINAL RCC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL RCC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_RCC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 31. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(output_file.resolve())

print(
    "\nComplete RCC qualitative analysis workbook "
    "created successfully."
)

RCC QUALITATIVE CODING ANALYSIS

Input file: C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\RCC_Coding_20260825_132706.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Number of participants: 26
Participants: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of unmapped themes: 0


RCC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 94
Cleaned theme observations: 94
Uniqu

In [39]:
# ================================================================
# RCC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# RCC_FINAL_QUALITATIVE_ANALYSIS_20260825_140150.xlsx
#
# OUTPUT:
# RCC_FINAL_EVIDENCE_AND_DIMENSIONS_YYYYMMDD_HHMMSS.xlsx
#
# PURPOSE:
# 1. Preserve the completed RCC qualitative coding
# 2. Extract final RCC evidence
# 3. Organize themes into evidence-based RCC dimensions
# 4. Produce dimension/theme tables for questionnaire development
# 5. Perform quality checks
#
# IMPORTANT:
# Dimensions generated here are qualitative groupings.
# They are NOT automatically treated as separate constructs.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. LOCATE INPUT FILE
# ================================================================

TARGET = "RCC_FINAL_QUALITATIVE_ANALYSIS_20260825_140150"

search_locations = [
    Path("/mnt/data"),
    Path("."),
    Path.home(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop"
]

possible_files = []

for location in search_locations:

    if not location.exists():
        continue

    try:

        for ext in [".xlsx", ".xlsm", ".xls"]:

            possible_files.extend(
                location.rglob(TARGET + ext)
            )

    except Exception:
        pass


# Remove duplicates
possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)


# ---------------------------------------------------------------
# Exact filename first
# ---------------------------------------------------------------

exact_files = [
    p for p in possible_files
    if p.stem == TARGET
]


if len(exact_files) > 0:

    INPUT_FILE = exact_files[0]


else:

    # -----------------------------------------------------------
    # Uploaded file may have been renamed by the system.
    # Identify it using the RCC-specific evidence sheet.
    # -----------------------------------------------------------

    rcc_candidates = []

    for p in possible_files:

        try:

            xl_test = pd.ExcelFile(
                p,
                engine="openpyxl"
            )

            if "11_Final_RCC_Evidence" in xl_test.sheet_names:

                rcc_candidates.append(p)

        except Exception:
            pass


    if len(rcc_candidates) > 0:

        INPUT_FILE = rcc_candidates[0]

    else:

        raise FileNotFoundError(
            "\nRCC input workbook could not be found.\n"
            "Expected:\n"
            f"{TARGET}.xlsx"
        )


print("=" * 85)
print("RCC FINAL QUALITATIVE ANALYSIS")
print("=" * 85)

print("\nInput file:")
print(INPUT_FILE)


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():

            return s

    return None


def clean_text(x):

    if pd.isna(x):

        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:

        x = float(x)

    except:

        return "Not available"


    if x >= 75:

        return "Very High"

    elif x >= 50:

        return "High"

    elif x >= 25:

        return "Moderate"

    else:

        return "Low"


# ================================================================
# 5. IDENTIFY ALL SUPPORTING SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_RCC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


if final_sheet is None:

    raise ValueError(
        "11_Final_RCC_Evidence was not found."
    )


# ================================================================
# 6. LOAD FINAL RCC EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()


final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# ================================================================
# 7. IDENTIFY RCC THEME COLUMN
# ================================================================

theme_col = None


if "Final_RCC_Theme" in final_evidence.columns:

    theme_col = "Final_RCC_Theme"


else:

    for c in final_evidence.columns:

        cstr = str(c)

        if (
            "RCC" in cstr
            and "Theme" in cstr
        ):

            theme_col = c

            break


# Fallback
if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c

            break


if theme_col is None:

    raise ValueError(
        "Could not identify the final RCC theme column."
    )


final_evidence[
    "Final_RCC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE EXPERT COLUMNS
# ================================================================

if "Experts_Mentioning" not in final_evidence.columns:

    for c in final_evidence.columns:

        cstr = str(c)

        if (
            "Experts" in cstr
            and "Mention" in cstr
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if "Expert_Prevalence_%" not in final_evidence.columns:

    for c in final_evidence.columns:

        cstr = str(c)

        if (
            "Prevalence" in cstr
            and "%" in cstr
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_RCC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. PARTICIPANT MATRIX
# ================================================================

participant_matrix = None


if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []


if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(
                cstr
            )


# If participant columns cannot be detected,
# use the known study participant count.
if len(participants) > 0:

    n_participants = len(
        participants
    )

else:

    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 12. SORT FINAL EVIDENCE
# ================================================================

if "Experts_Mentioning" in final_evidence.columns:

    final_evidence[
        "_Sort_Experts"
    ] = pd.to_numeric(
        final_evidence[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    final_evidence = (
        final_evidence
        .sort_values(
            "_Sort_Experts",
            ascending=False
        )
        .drop(
            columns=["_Sort_Experts"]
        )
        .reset_index(drop=True)
    )


# ================================================================
# 13. CREATE FINAL RCC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_RCC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]


evidence_columns = [

    c

    for c in preferred_columns

    if c in final_evidence.columns

]


final_rcc_evidence = final_evidence[
    evidence_columns
].copy()


# Avoid duplicate Rank errors
if "Rank" in final_rcc_evidence.columns:

    final_rcc_evidence = (
        final_rcc_evidence
        .drop(
            columns=["Rank"]
        )
    )


final_rcc_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_rcc_evidence) + 1
    )
)


# ================================================================
# 14. RCC DIMENSION MAPPING
# ================================================================
#
# The dimensions represent qualitative aspects of RCC.
#
# They are NOT automatically separate constructs.
#
# Their purpose is questionnaire-item development.
# ================================================================

def map_rcc_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. INFORMATION SHARING & COMMUNICATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "information sharing",
            "information exchange",
            "information flow",
            "information flows",
            "communication",
            "communicate",
            "sharing information",
            "timely information",
            "risk information"

        ]
    ):

        return (
            "Risk Information Sharing & Communication"
        )


    # ------------------------------------------------------------
    # 2. CROSS-FUNCTIONAL COORDINATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "cross-functional",
            "cross functional",
            "functional coordination",
            "department coordination",
            "interdepartmental",
            "between departments",
            "between functions",
            "internal coordination",
            "cross-unit",
            "cross unit"

        ]
    ):

        return (
            "Cross-Functional Risk Coordination"
        )


    # ------------------------------------------------------------
    # 3. SUPPLY-CHAIN PARTNER COORDINATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "supplier coordination",
            "supplier collaboration",
            "supplier cooperation",
            "customer coordination",
            "customer collaboration",
            "partner coordination",
            "partner collaboration",
            "external coordination",
            "supply chain partners",
            "supply-chain partners",
            "supplier",
            "customer",
            "partner"

        ]
    ):

        return (
            "Supply-Chain Partner Coordination"
        )


    # ------------------------------------------------------------
    # 4. JOINT RISK RESPONSE
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "joint response",
            "joint action",
            "collective response",
            "collective action",
            "coordinated response",
            "coordinated action",
            "joint risk",
            "shared response",
            "response coordination"

        ]
    ):

        return (
            "Joint Risk Response"
        )


    # ------------------------------------------------------------
    # 5. RESOURCE & RESPONSIBILITY COORDINATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "resource coordination",
            "resource allocation",
            "resource sharing",
            "resource deployment",
            "responsibility",
            "responsibilities",
            "role clarity",
            "roles",
            "task allocation",
            "allocation of resources",
            "resource"

        ]
    ):

        return (
            "Resource & Responsibility Coordination"
        )


    # ------------------------------------------------------------
    # 6. COORDINATED DECISION-MAKING
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "decision coordination",
            "decision making",
            "decision-making",
            "joint decision",
            "shared decision",
            "collective decision",
            "coordinated decisions",
            "decision alignment",
            "alignment of decisions"

        ]
    ):

        return (
            "Coordinated Risk Decision-Making"
        )


    # ------------------------------------------------------------
    # 7. COORDINATION MONITORING & ALIGNMENT
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "monitor coordination",
            "coordination monitoring",
            "monitoring",
            "tracking",
            "alignment",
            "synchronization",
            "synchronisation",
            "follow-up",
            "follow up",
            "coordination effectiveness"

        ]
    ):

        return (
            "Coordination Monitoring & Alignment"
        )


    # ------------------------------------------------------------
    # 8. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_rcc_evidence[
    "RCC_Dimension"
] = (
    final_rcc_evidence[
        "Final_RCC_Theme"
    ]
    .apply(
        map_rcc_dimension
    )
)


# ================================================================
# 15. EXACT THEME OVERRIDES
# ================================================================
#
# These only apply when a final theme exactly matches the
# specified wording.
# ================================================================

manual_mapping = {

    "information sharing":
        "Risk Information Sharing & Communication",

    "information exchange":
        "Risk Information Sharing & Communication",

    "information flow":
        "Risk Information Sharing & Communication",

    "communication":
        "Risk Information Sharing & Communication",

    "cross-functional coordination":
        "Cross-Functional Risk Coordination",

    "cross functional coordination":
        "Cross-Functional Risk Coordination",

    "supplier coordination":
        "Supply-Chain Partner Coordination",

    "supplier collaboration":
        "Supply-Chain Partner Coordination",

    "partner coordination":
        "Supply-Chain Partner Coordination",

    "joint response":
        "Joint Risk Response",

    "joint action":
        "Joint Risk Response",

    "collective response":
        "Joint Risk Response",

    "coordinated response":
        "Joint Risk Response",

    "resource coordination":
        "Resource & Responsibility Coordination",

    "resource allocation":
        "Resource & Responsibility Coordination",

    "role clarity":
        "Resource & Responsibility Coordination",

    "decision coordination":
        "Coordinated Risk Decision-Making",

    "joint decision":
        "Coordinated Risk Decision-Making",

    "shared decision":
        "Coordinated Risk Decision-Making",

    "coordination monitoring":
        "Coordination Monitoring & Alignment",

    "coordination effectiveness":
        "Coordination Monitoring & Alignment"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_rcc_evidence[
            "Final_RCC_Theme"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            theme.lower()
        )
    )


    final_rcc_evidence.loc[
        mask,
        "RCC_Dimension"
    ] = dimension


# ================================================================
# 16. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_rcc_evidence
    .groupby(
        "RCC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_RCC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    experts_mentioning = 0


    # ------------------------------------------------------------
    # Participant matrix method
    # ------------------------------------------------------------

    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        temp = participant_matrix.copy()


        theme_candidates = [

            c

            for c in temp.columns

            if "theme" in str(c).lower()

        ]


        if len(theme_candidates) > 0:

            matrix_theme_col = (
                theme_candidates[0]
            )

        else:

            matrix_theme_col = (
                temp.columns[0]
            )


        temp["_Theme"] = (
            temp[
                matrix_theme_col
            ]
            .astype(str)
            .str.strip()
        )


        theme_lower = {

            str(x)
            .strip()
            .lower()

            for x in themes

        }


        matched = temp[
            temp[
                "_Theme"
            ]
            .str.lower()
            .isin(
                theme_lower
            )
        ]


        available_participants = [

            p

            for p in participants

            if p in matched.columns

        ]


        if (
            len(matched) > 0
            and len(available_participants) > 0
        ):

            vals = (
                matched[
                    available_participants
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce"
                )
                .fillna(0)
            )


            experts_mentioning = int(
                (
                    vals.sum(axis=0) > 0
                ).sum()
            )


    # ------------------------------------------------------------
    # Fallback to final evidence
    # ------------------------------------------------------------

    if experts_mentioning == 0:

        if (
            "Experts_Mentioning"
            in group.columns
        ):

            try:

                experts_mentioning = int(
                    pd.to_numeric(
                        group[
                            "Experts_Mentioning"
                        ],
                        errors="coerce"
                    )
                    .max()
                )

            except:

                experts_mentioning = 0


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0

        else 0

    )


    dimension_rows.append({

        "RCC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_RCC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 17. SORT DIMENSION SUMMARY
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary[
        "_sort"
    ] = pd.to_numeric(
        dimension_summary[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_summary = (
        dimension_summary
        .sort_values(
            "_sort",
            ascending=False
        )
        .drop(
            columns=["_sort"]
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(
                columns=["Rank"]
            )
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 18. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "RCC_Dimension",

    "Final_RCC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [

    c

    for c in dimension_theme_columns

    if c in final_rcc_evidence.columns

]


dimension_themes = final_rcc_evidence[
    dimension_theme_columns
].copy()


if "Experts_Mentioning" in dimension_themes.columns:

    dimension_themes[
        "_sort"
    ] = pd.to_numeric(
        dimension_themes[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_themes = (
        dimension_themes
        .sort_values(
            [
                "RCC_Dimension",
                "_sort"
            ],
            ascending=[
                True,
                False
            ]
        )
        .drop(
            columns=["_sort"]
        )
    )


# ================================================================
# 19. THEMES REQUIRING MANUAL REVIEW
# ================================================================

unmapped_check = final_rcc_evidence[
    final_rcc_evidence[
        "RCC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    review_columns = [

        "Final_RCC_Theme",

        "Experts_Mentioning",

        "Expert_Prevalence_%",

        "RCC_Dimension"

    ]


    review_columns = [

        c

        for c in review_columns

        if c in unmapped_check.columns

    ]


    unmapped_check = unmapped_check[
        review_columns
    ].copy()


    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )


else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All RCC themes mapped to a dimension."
        ]

    })


# ================================================================
# 20. CODING AUDIT CHECK
# ================================================================

coding_audit = None


if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


if coding_audit is not None:

    decision_columns = [

        c

        for c in coding_audit.columns

        if "Decision" in str(c)

    ]


    if len(decision_columns) > 0:

        decision_col = (
            decision_columns[0]
        )


        missing_decisions = coding_audit[
            coding_audit[
                decision_col
            ].isna()
        ]


        if len(missing_decisions) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All RCC coding decisions are present."
                ]

            })

        else:

            decision_check = missing_decisions.copy()


    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })

else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit unavailable."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final RCC evidence available",

    "Result":
        "PASS"
        if len(final_rcc_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_rcc_evidence)} final RCC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "RCC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative evidence-based dimensions"

})


review_count = (

    len(unmapped_check)

    if "Final_RCC_Theme"
    in unmapped_check.columns

    else 0

)


quality_rows.append({

    "Quality_Check":
        "Themes requiring manual review",

    "Result":
        review_count,

    "Details":
        "Themes assigned to Review Required"

})


duplicate_count = int(
    final_rcc_evidence[
        "Final_RCC_Theme"
    ]
    .str.lower()
    .duplicated()
    .sum()
)


quality_rows.append({

    "Quality_Check":
        "Duplicate theme labels",

    "Result":
        duplicate_count,

    "Details":
        "Case-insensitive duplicate labels"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:

        return sheets[
            sheet
        ].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. WRITE FINAL OUTPUT
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"RCC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:


    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    if coding_audit is not None:

        coding_audit.to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )


    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    final_rcc_evidence.to_excel(
        writer,
        sheet_name="13_Final_RCC_Evidence",
        index=False
    )


    dimension_summary.to_excel(
        writer,
        sheet_name="14_RCC_Dimension_Summary",
        index=False
    )


    dimension_themes.to_excel(
        writer,
        sheet_name="15_RCC_Dimension_Themes",
        index=False
    )


    construct_statistics.to_excel(
        writer,
        sheet_name="16_Construct_Statistics",
        index=False
    )


# ================================================================
# 24. DISPLAY RESULTS
# ================================================================

print("\n")
print("=" * 85)
print("RCC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 85)

print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)


print("\nFinal RCC themes:")
print(
    len(final_rcc_evidence)
)


print("\nRCC dimensions:")
print(
    len(dimension_summary)
)


print("\n")
print("-" * 85)
print("RCC DIMENSION SUMMARY")
print("-" * 85)

if len(dimension_summary) > 0:

    print(
        dimension_summary.to_string(
            index=False
        )
    )

else:

    print(
        "No dimensions were generated."
    )


print("\n")
print("-" * 85)
print("QUALITY CHECKS")
print("-" * 85)

print(
    quality_checks_new.to_string(
        index=False
    )
)


print("\n")
print("=" * 85)
print("NEXT QUESTIONNAIRE STAGE")
print("=" * 85)

print("""
Use these two sheets for RCC questionnaire development:

14_RCC_Dimension_Summary
15_RCC_Dimension_Themes

The dimensions are qualitative evidence-based groupings.
They should NOT automatically be treated as separate
latent constructs.

The next step is to examine the strongest RCC themes,
remove conceptual overlap, and convert the retained themes
into candidate reflective questionnaire items.
""")

RCC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\RCC_FINAL_QUALITATIVE_ANALYSIS_20260825_140150.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_RCC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Participants used: 26


RCC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\RCC_FINAL_EVIDENCE_AND_DIMENSIONS_20260825_145902.xlsx

Final RCC themes:
64

RCC dimensions:
7


-------------------------------------------------------------------------------------
RCC DIMENSION SUMMARY
-------------------------------------------------------------------------

## SPC

In [8]:
# ================================================================
# SPC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:
        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT SPC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "SPC":
                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({
                "Participant": participant,
                "Construct": "SPC",
                "Original_Response": response,
                "Source_Sheet": sheet,
                "Source_Row": row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(raw_rows)

if original_df.empty:

    raise ValueError(
        """
        No SPC responses were found.

        Check that SPC appears in the participant sheets.
        """
    )

print("\nSPC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace("\n", ",")
    response = response.replace(";", ",")
    response = response.replace("•", ",")

    themes = response.split(",")

    for theme in themes:

        theme = str(theme).strip()

        if theme == "":
            continue

        theme_rows.append({
            "Participant": row["Participant"],
            "Construct": "SPC",
            "Raw_Theme": theme,
            "Source_Sheet": row["Source_Sheet"],
            "Source_Row": row["Source_Row"]
        })


raw_df = pd.DataFrame(theme_rows)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(drop=True)

print("\nSPC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW SPC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Theme_Key")
    .reset_index(drop=True)
)

print("\n")
print("=" * 60)
print("SPC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(index=False)
)


# ------------------------------------------------
# 8. SPC NORMALIZATION DICTIONARY
# ------------------------------------------------

SPC_NORMALIZATION = {

    # ADD YOUR SPC MAPPINGS HERE
    #
    # Example:
    #
    # "example raw theme":
    #     "Normalized theme",
    #
}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""
coding_df["Decision"] = ""
coding_df["Reason"] = ""

for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in SPC_NORMALIZATION:

        normalized = SPC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × SPC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(name="Count")
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "SPC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "SPC_Coding_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("SPC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique SPC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized SPC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(index=False)
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

SPC original responses:
26

SPC cleaned theme observations:
93


SPC RAW THEMES
                             Raw_Theme                            Theme_Clean                              Theme_Key
                  alternative sourcing                   alternative sourcing                   alternative sourcing
                 alternative suppliers                  alternative suppliers                  alternative suppliers
                          alternatives                           alternatives                           alternatives
                       backup capacity                        backup capacity                        backup capacity
                      backup suppliers                   

In [23]:
# ================================================================
# COMPLETE SPC QUALITATIVE CODING ANALYSIS
# Input: SPC_Coding_20260825_132742.xlsx
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path("SPC_Coding_20260825_132742.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as this notebook."
    )

CONSTRUCT = "SPC"

print("=" * 70)
print(f"{CONSTRUCT} QUALITATIVE CODING ANALYSIS")
print("=" * 70)

excel = pd.ExcelFile(input_file)

print("\nInput file:", input_file.resolve())
print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 2. FIND SHEETS FLEXIBLY
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    f"01_Original_{CONSTRUCT}",
    "01_Original",
    f"Original_{CONSTRUCT}",
    "Original"
])

raw_sheet = find_sheet([
    f"02_Raw_{CONSTRUCT}",
    "02_Raw",
    f"Raw_{CONSTRUCT}",
    "Raw"
])

clean_sheet = find_sheet([
    f"03_Cleaned_{CONSTRUCT}",
    "03_Cleaned",
    f"Cleaned_{CONSTRUCT}",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 3. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(input_file, sheet_name=original_sheet)
    if original_sheet else pd.DataFrame()
)

raw_df = (
    pd.read_excel(input_file, sheet_name=raw_sheet)
    if raw_sheet else pd.DataFrame()
)

clean_df = (
    pd.read_excel(input_file, sheet_name=clean_sheet)
    if clean_sheet else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCoding/Normalized Coding sheet could not be found.\n\n"
        "Available sheets:\n" +
        "\n".join(str(x) for x in excel.sheet_names)
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 4. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:
    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 5. FIND COLUMNS
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    ["Theme_Key", "Theme Key", "ThemeKey"]
)

raw_theme_col = find_column(
    coding_df,
    ["Raw_Theme", "Raw Theme", "RawTheme"]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    ["Decision"]
)


if theme_key_col is None:
    raise ValueError("Theme_Key column not found.")

if raw_theme_col is None:
    raise ValueError("Raw_Theme column not found.")

if normalized_col is None:
    raise ValueError("Normalized_Theme column not found.")

if decision_col is None:
    raise ValueError("Decision column not found.")


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 6. FIND PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:

    raise ValueError(
        "\nParticipant column not found.\n\n"
        "Available columns:\n" +
        str(list(clean_df.columns))
    )

if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col: "Participant"
        }
    )


# ================================================================
# 7. FIND THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key not found in Cleaned sheet.\n\n"
        "Available columns:\n" +
        str(list(clean_df.columns))
    )

if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key: "Theme_Key"
        }
    )


# ================================================================
# 8. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        ["nan", "None", "", "NaN"],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 9. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df["Participant"]
    .replace("", np.nan)
    .dropna()
    .unique()
)

print("\nNumber of participants:", len(participants))
print("Participants:", participants)


# ================================================================
# 11. MERGE CLEANED DATA WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 12. IDENTIFY UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df["Normalized_Theme"].isna()
].copy()

print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)

if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 13. PARTICIPANT × NORMALIZED SPC THEMES
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 14. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme["Normalized_Theme"],
    participant_theme["Participant"]
)


# ================================================================
# 15. ORDER PARTICIPANTS
# ================================================================

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()

participant_columns = existing + other


# ================================================================
# 16. FREQUENCY
# ================================================================

matrix["Frequency"] = (
    matrix[participant_columns]
    .sum(axis=1)
)


# ================================================================
# 17. EXPERT PREVALENCE
# ================================================================

total_participants = len(participants)

if total_participants > 0:

    matrix["Percentage"] = (
        matrix["Frequency"]
        / total_participants
        * 100
    ).round(1)

else:

    matrix["Percentage"] = 0


# ================================================================
# 18. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)

matrix.insert(
    0,
    "Rank",
    range(1, len(matrix) + 1)
)


# ================================================================
# 19. SPC THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()

theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme": "Final_SPC_Theme",
        "Frequency": "Experts_Mentioning",
        "Percentage": "Percentage_of_Experts"
    }
)


# ================================================================
# 20. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary["Prevalence_Category"] = (
    theme_summary["Percentage_of_Experts"]
    .apply(prevalence_category)
)


# ================================================================
# 21. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby("Normalized_Theme")
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)

coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)

coding_audit["Experts_Mentioning"] = (
    coding_audit["Experts_Mentioning"]
    .fillna(0)
    .astype(int)
)

if total_participants > 0:

    coding_audit["Percentage_of_Experts"] = (
        coding_audit["Experts_Mentioning"]
        / total_participants
        * 100
    ).round(1)

else:

    coding_audit["Percentage_of_Experts"] = 0


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 22. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)

if len(coding_df) > 0:

    decision_summary["Percentage"] = (
        decision_summary["Number_of_Raw_Themes"]
        / len(coding_df)
        * 100
    ).round(1)

else:

    decision_summary["Percentage"] = 0


# ================================================================
# 23. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(subset=["Normalized_Theme"])
    .groupby("Normalized_Theme")
    .agg(
        Raw_Themes=("Raw_Theme", "count"),

        Keep_Count=(
            "Decision",
            lambda x: (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x: (x == "Merge").sum()
        )
    )
    .reset_index()
)

normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)

normalization_summary["Experts_Mentioning"] = (
    normalization_summary["Experts_Mentioning"]
    .fillna(0)
    .astype(int)
)

if total_participants > 0:

    normalization_summary["Percentage_of_Experts"] = (
        normalization_summary["Experts_Mentioning"]
        / total_participants
        * 100
    ).round(1)

else:

    normalization_summary["Percentage_of_Experts"] = 0


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 24. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_SPC_Themes"
]


# ================================================================
# 25. SPC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": [CONSTRUCT],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df["Theme_Key"].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df["Normalized_Theme"].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df["Decision"] == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df["Decision"] == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 26. FINAL SPC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()

final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme": "Final_SPC_Theme",
        "Raw_Themes": "Number_of_Raw_Themes",
        "Percentage_of_Experts": "Expert_Prevalence_%",
        "Keep_Count": "Raw_Themes_Kept",
        "Merge_Count": "Raw_Themes_Merged"
    }
)

final_evidence["Prevalence_Category"] = (
    final_evidence["Expert_Prevalence_%"]
    .apply(prevalence_category)
)


# ================================================================
# 27. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)

quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",
        "Original response rows",
        "Raw theme observations",
        "Unique raw themes",
        "Final normalized themes",
        "Unmapped themes",
        "Duplicate participant-theme records",
        "Keep decisions",
        "Merge decisions"

    ],

    "Result": [

        total_participants,
        len(original_df),
        len(raw_df),

        coding_df["Theme_Key"].nunique(),

        coding_df["Normalized_Theme"].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df["Decision"] == "Keep"
        ).sum(),

        (
            coding_df["Decision"] == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS" if total_participants > 0 else "CHECK",

        "PASS" if len(original_df) > 0 else "CHECK",

        "PASS" if len(raw_df) > 0 else "CHECK",

        "PASS"
        if coding_df["Theme_Key"].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df["Normalized_Theme"].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",
        "PASS",
        "PASS"
    ]
})


# ================================================================
# 28. SAVE COMPLETE SPC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"SPC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)

counter = 1

while output_file.exists():

    output_file = Path(
        f"SPC_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_SPC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 29. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("SPC ANALYSIS COMPLETED")
print("=" * 70)

print("\nParticipants:", total_participants)

print("Original responses:", len(original_df))

print("Raw theme observations:", len(raw_df))

print("Cleaned theme observations:", len(clean_df))

print(
    "Unique raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Final normalized SPC themes:",
    coding_df["Normalized_Theme"].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df["Decision"] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df["Decision"] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 30. DISPLAY FINAL SPC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL SPC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_SPC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 31. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(output_file.resolve())

print(
    "\nComplete SPC qualitative analysis workbook "
    "created successfully."
)

SPC QUALITATIVE CODING ANALYSIS

Input file: C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\SPC_Coding_20260825_132742.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Number of participants: 26
Participants: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of unmapped themes: 0


SPC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 93
Cleaned theme observations: 93
Uniqu

In [40]:
# ================================================================
# SPC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# SPC_FINAL_QUALITATIVE_ANALYSIS_20260825_140437.xlsx
#
# OUTPUT:
# SPC_FINAL_EVIDENCE_AND_DIMENSIONS_YYYYMMDD_HHMMSS.xlsx
#
# PURPOSE:
# 1. Read the completed SPC qualitative analysis
# 2. Extract final SPC evidence
# 3. Remove accidental capitalization duplicates
# 4. Group themes into evidence-based SPC dimensions
# 5. Produce dimension/theme tables
# 6. Produce quality checks
#
# IMPORTANT:
# Dimensions are NOT automatically separate constructs.
# They are qualitative domains for questionnaire-item development.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. LOCATE INPUT FILE
# ================================================================

TARGET = "SPC_FINAL_QUALITATIVE_ANALYSIS_20260825_140437"

search_locations = [
    Path("/mnt/data"),
    Path("."),
    Path.home(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop"
]

possible_files = []

for location in search_locations:

    if not location.exists():
        continue

    try:

        for ext in [".xlsx", ".xlsm", ".xls"]:
            possible_files.extend(
                location.rglob(TARGET + ext)
            )

    except Exception:
        pass


possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)


# ---------------------------------------------------------------
# Exact filename
# ---------------------------------------------------------------

exact_files = [
    p for p in possible_files
    if p.stem == TARGET
]


if len(exact_files) > 0:

    INPUT_FILE = exact_files[0]

else:

    # -----------------------------------------------------------
    # System may rename uploaded files.
    # Find workbook containing 11_Final_SPC_Evidence.
    # -----------------------------------------------------------

    candidates = []

    # Search common locations broadly
    broad_locations = [
        Path("/mnt/data"),
        Path("."),
        Path.home() / "Downloads",
        Path.home() / "Documents",
        Path.home() / "Desktop"
    ]

    for location in broad_locations:

        if not location.exists():
            continue

        try:

            for p in location.rglob("*.xlsx"):

                if p in candidates:
                    continue

                try:

                    test_xls = pd.ExcelFile(
                        p,
                        engine="openpyxl"
                    )

                    if "11_Final_SPC_Evidence" in test_xls.sheet_names:

                        candidates.append(
                            p.resolve()
                        )

                except Exception:
                    pass

        except Exception:
            pass


    if len(candidates) == 0:

        raise FileNotFoundError(
            "\nSPC input workbook could not be found.\n"
            "Expected:\n"
            f"{TARGET}.xlsx"
        )

    INPUT_FILE = candidates[0]


print("=" * 90)
print("SPC FINAL QUALITATIVE ANALYSIS")
print("=" * 90)

print("\nInput file:")
print(INPUT_FILE)


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():
            return s

    return None


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:
        x = float(x)
    except:
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


# ================================================================
# 5. IDENTIFY SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_SPC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


if final_sheet is None:

    raise ValueError(
        "11_Final_SPC_Evidence was not found."
    )


# ================================================================
# 6. LOAD FINAL SPC EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()


final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# ================================================================
# 7. IDENTIFY THEME COLUMN
# ================================================================

theme_col = None


if "Final_SPC_Theme" in final_evidence.columns:

    theme_col = "Final_SPC_Theme"

else:

    for c in final_evidence.columns:

        if (
            "SPC" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify the final SPC theme column."
    )


final_evidence[
    "Final_SPC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE IMPORTANT COLUMNS
# ================================================================

if "Experts_Mentioning" not in final_evidence.columns:

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if "Expert_Prevalence_%" not in final_evidence.columns:

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_SPC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. IDENTIFY PARTICIPANTS
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []


if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(cstr)


# Your qualitative dataset contains 26 participants.
if len(participants) > 0:

    n_participants = len(participants)

else:

    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 12. NORMALIZE CAPITALIZATION DUPLICATES
# ================================================================
#
# Example:
# Flexibility
# flexibility
#
# These represent the same qualitative theme and should not be
# treated as two different questionnaire domains.
#
# We retain the most informative/highest-prevalence row.
# ================================================================

final_evidence[
    "_Theme_Normalized"
] = (
    final_evidence[
        "Final_SPC_Theme"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


# Convert expert prevalence to numeric
if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "_Prevalence_Numeric"
    ] = pd.to_numeric(
        final_evidence[
            "Expert_Prevalence_%"
        ],
        errors="coerce"
    ).fillna(0)

else:

    final_evidence[
        "_Prevalence_Numeric"
    ] = 0


# Sort strongest version first
final_evidence = (
    final_evidence
    .sort_values(
        "_Prevalence_Numeric",
        ascending=False
    )
)


# Keep one capitalization variant
final_evidence = (
    final_evidence
    .drop_duplicates(
        subset=[
            "_Theme_Normalized"
        ],
        keep="first"
    )
    .copy()
)


# Remove temporary columns
final_evidence = (
    final_evidence
    .drop(
        columns=[
            "_Theme_Normalized",
            "_Prevalence_Numeric"
        ],
        errors="ignore"
    )
)


# ================================================================
# 13. SORT FINAL EVIDENCE
# ================================================================

if "Experts_Mentioning" in final_evidence.columns:

    final_evidence[
        "_Sort"
    ] = pd.to_numeric(
        final_evidence[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    final_evidence = (
        final_evidence
        .sort_values(
            "_Sort",
            ascending=False
        )
        .drop(
            columns=["_Sort"]
        )
        .reset_index(drop=True)
    )


# ================================================================
# 14. FINAL SPC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_SPC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]


evidence_columns = [

    c
    for c in preferred_columns
    if c in final_evidence.columns

]


final_spc_evidence = final_evidence[
    evidence_columns
].copy()


# Avoid Rank duplication
if "Rank" in final_spc_evidence.columns:

    final_spc_evidence = (
        final_spc_evidence
        .drop(
            columns=["Rank"]
        )
    )


final_spc_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_spc_evidence) + 1
    )
)


# ================================================================
# 15. SPC DIMENSION MAPPING
# ================================================================
#
# Based specifically on the themes present in your SPC file:
#
# Alternatives
# Flexibility
# Diversification
# Long-term planning
# Dependency reduction
# Capacity
# Critical dependencies
# Vulnerability identification
# Weakness identification
# Proactive preparation
# Strategic planning
# Vulnerability reduction
# Alternative sourcing
# Backup suppliers
# Backup capacity
# Strategic resilience
# Contingency planning
# Inventory preparedness
# Scenario preparation
# Lessons learned
# Resilience investment
# etc.
#
# These are grouped into questionnaire-content domains.
# ================================================================

def map_spc_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. SUPPLY / SOURCE DIVERSIFICATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "diversification",
            "alternative sourcing",
            "alternative suppliers",
            "supplier alternatives",
            "backup suppliers",
            "alternatives",
            "alternative source",
            "multiple suppliers"

        ]
    ):

        return (
            "Supply Source Diversification & Alternatives"
        )


    # ------------------------------------------------------------
    # 2. DEPENDENCY & VULNERABILITY MANAGEMENT
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "dependency reduction",
            "dependencies",
            "dependency identification",
            "strategic dependency",
            "critical dependencies",
            "supplier criticality",
            "critical risks",
            "vulnerability identification",
            "vulnerability",
            "vulnerabilities",
            "vulnerability assessment",
            "vulnerability reduction",
            "proactive vulnerability",
            "weakness identification",
            "weakness"

        ]
    ):

        return (
            "Dependency & Vulnerability Management"
        )


    # ------------------------------------------------------------
    # 3. CAPACITY & INVENTORY FLEXIBILITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "capacity",
            "flexible capacity",
            "proactive capacity",
            "backup capacity",
            "spare capacity",
            "inventory",
            "inventory preparedness",
            "inventory policy",
            "capacity and inventory"

        ]
    ):

        return (
            "Capacity & Inventory Flexibility"
        )


    # ------------------------------------------------------------
    # 4. STRATEGIC FLEXIBILITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "flexibility",
            "strategic flexibility",
            "flexible"

        ]
    ):

        return (
            "Strategic Flexibility"
        )


    # ------------------------------------------------------------
    # 5. PROACTIVE PREPAREDNESS & CONTINGENCY PLANNING
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "proactive preparation",
            "proactive vulnerability preparation",
            "preparation",
            "preparedness",
            "embedded preparedness",
            "contingency planning",
            "contingencies",
            "scenario preparation",
            "backup",
            "long-term vulnerability"

        ]
    ):

        return (
            "Proactive Preparedness & Contingency Planning"
        )


    # ------------------------------------------------------------
    # 6. LONG-TERM STRATEGIC PLANNING
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "long-term planning",
            "long-term sourcing",
            "long-term consequences",
            "strategic planning",
            "strategic priorities",
            "strategic resilience",
            "planning"

        ]
    ):

        return (
            "Long-Term Strategic Planning"
        )


    # ------------------------------------------------------------
    # 7. RESILIENCE INVESTMENT & RESOURCE COMMITMENT
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "investment",
            "resilience investment",
            "resource commitment"

        ]
    ):

        return (
            "Resilience Investment & Resource Commitment"
        )


    # ------------------------------------------------------------
    # 8. LEARNING & PLAN IMPROVEMENT
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "lessons learned",
            "lesson learned",
            "review",
            "plan improvement",
            "improvement"

        ]
    ):

        return (
            "Learning & Strategic Plan Improvement"
        )


    # ------------------------------------------------------------
    # 9. GENERAL SPC / REVIEW
    # ------------------------------------------------------------

    return "Review Required"


final_spc_evidence = final_spc_evidence.copy()


final_spc_evidence[
    "SPC_Dimension"
] = (
    final_spc_evidence[
        "Final_SPC_Theme"
    ]
    .apply(
        map_spc_dimension
    )
)


# ================================================================
# 16. MANUAL EXACT-THEME OVERRIDES
# ================================================================

manual_mapping = {

    "alternatives":
        "Supply Source Diversification & Alternatives",

    "alternative sourcing":
        "Supply Source Diversification & Alternatives",

    "alternative suppliers":
        "Supply Source Diversification & Alternatives",

    "supplier alternatives":
        "Supply Source Diversification & Alternatives",

    "backup suppliers":
        "Supply Source Diversification & Alternatives",

    "diversification":
        "Supply Source Diversification & Alternatives",

    "flexibility":
        "Strategic Flexibility",

    "dependency reduction":
        "Dependency & Vulnerability Management",

    "dependencies":
        "Dependency & Vulnerability Management",

    "critical dependencies":
        "Dependency & Vulnerability Management",

    "strategic dependency":
        "Dependency & Vulnerability Management",

    "vulnerability":
        "Dependency & Vulnerability Management",

    "vulnerabilities":
        "Dependency & Vulnerability Management",

    "vulnerability assessment":
        "Dependency & Vulnerability Management",

    "vulnerability reduction":
        "Dependency & Vulnerability Management",

    "vulnerability identification":
        "Dependency & Vulnerability Management",

    "proactive vulnerability identification":
        "Dependency & Vulnerability Management",

    "weakness identification":
        "Dependency & Vulnerability Management",

    "capacity":
        "Capacity & Inventory Flexibility",

    "flexible capacity":
        "Capacity & Inventory Flexibility",

    "proactive capacity":
        "Capacity & Inventory Flexibility",

    "backup capacity":
        "Capacity & Inventory Flexibility",

    "spare capacity":
        "Capacity & Inventory Flexibility",

    "inventory preparedness":
        "Capacity & Inventory Flexibility",

    "inventory policy":
        "Capacity & Inventory Flexibility",

    "proactive preparation":
        "Proactive Preparedness & Contingency Planning",

    "preparation":
        "Proactive Preparedness & Contingency Planning",

    "embedded preparedness":
        "Proactive Preparedness & Contingency Planning",

    "scenario preparation":
        "Proactive Preparedness & Contingency Planning",

    "contingency planning":
        "Proactive Preparedness & Contingency Planning",

    "contingencies":
        "Proactive Preparedness & Contingency Planning",

    "long-term planning":
        "Long-Term Strategic Planning",

    "long-term sourcing/capacity planning":
        "Long-Term Strategic Planning",

    "long-term consequences":
        "Long-Term Strategic Planning",

    "strategic planning":
        "Long-Term Strategic Planning",

    "strategic priorities":
        "Long-Term Strategic Planning",

    "strategic resilience":
        "Long-Term Strategic Planning",

    "investment":
        "Resilience Investment & Resource Commitment",

    "resilience investment":
        "Resilience Investment & Resource Commitment",

    "lessons learned":
        "Learning & Strategic Plan Improvement",

    "plan improvement":
        "Learning & Strategic Plan Improvement",

    "review":
        "Learning & Strategic Plan Improvement"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_spc_evidence[
            "Final_SPC_Theme"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            theme.lower()
        )
    )


    final_spc_evidence.loc[
        mask,
        "SPC_Dimension"
    ] = dimension


# ================================================================
# 17. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_spc_evidence
    .groupby(
        "SPC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_SPC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # ------------------------------------------------------------
    # Determine experts mentioning dimension
    # ------------------------------------------------------------

    experts = 0


    if "Experts_Mentioning" in group.columns:

        try:

            experts = int(
                pd.to_numeric(
                    group[
                        "Experts_Mentioning"
                    ],
                    errors="coerce"
                )
                .max()
            )

        except:

            experts = 0


    prevalence = (

        experts
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "SPC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_SPC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 18. SORT DIMENSIONS
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary[
        "_sort"
    ] = pd.to_numeric(
        dimension_summary[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_summary = (
        dimension_summary
        .sort_values(
            "_sort",
            ascending=False
        )
        .drop(
            columns=["_sort"]
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(
                columns=["Rank"]
            )
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 19. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "SPC_Dimension",

    "Final_SPC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [

    c
    for c in dimension_theme_columns
    if c in final_spc_evidence.columns

]


dimension_themes = final_spc_evidence[
    dimension_theme_columns
].copy()


if "Experts_Mentioning" in dimension_themes.columns:

    dimension_themes[
        "_sort"
    ] = pd.to_numeric(
        dimension_themes[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_themes = (
        dimension_themes
        .sort_values(
            [
                "SPC_Dimension",
                "_sort"
            ],
            ascending=[
                True,
                False
            ]
        )
        .drop(
            columns=["_sort"]
        )
    )


# ================================================================
# 20. THEMES REQUIRING REVIEW
# ================================================================

unmapped_check = final_spc_evidence[
    final_spc_evidence[
        "SPC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    review_columns = [

        "Final_SPC_Theme",

        "Experts_Mentioning",

        "Expert_Prevalence_%",

        "SPC_Dimension"

    ]


    review_columns = [

        c
        for c in review_columns
        if c in unmapped_check.columns

    ]


    unmapped_check = unmapped_check[
        review_columns
    ].copy()


    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )


else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All SPC themes mapped to a dimension."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final SPC evidence available",

    "Result":
        "PASS"
        if len(final_spc_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_spc_evidence)} normalized SPC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used"

})


quality_rows.append({

    "Quality_Check":
        "SPC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative evidence-based domains"

})


review_count = (

    len(unmapped_check)

    if "Final_SPC_Theme"
    in unmapped_check.columns

    else 0

)


quality_rows.append({

    "Quality_Check":
        "Themes requiring manual review",

    "Result":
        review_count,

    "Details":
        "Themes assigned Review Required"

})


duplicate_count = int(
    final_spc_evidence[
        "Final_SPC_Theme"
    ]
    .astype(str)
    .str.lower()
    .duplicated()
    .sum()
)


quality_rows.append({

    "Quality_Check":
        "Remaining capitalization duplicates",

    "Result":
        duplicate_count,

    "Details":
        "Should normally be zero"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:
        return sheets[sheet].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)

coding_audit = get_sheet_or_empty(
    audit_sheet
)


# ================================================================
# 23. OUTPUT FILE
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"SPC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


# ================================================================
# 24. WRITE OUTPUT
# ================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    unmapped_check.to_excel(
        writer,
        sheet_name="11_Unmapped_Check",
        index=False
    )

    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    final_spc_evidence.to_excel(
        writer,
        sheet_name="13_Final_SPC_Evidence",
        index=False
    )

    dimension_summary.to_excel(
        writer,
        sheet_name="14_SPC_Dimension_Summary",
        index=False
    )

    dimension_themes.to_excel(
        writer,
        sheet_name="15_SPC_Dimension_Themes",
        index=False
    )


# ================================================================
# 25. FINAL REPORT
# ================================================================

print("\n")
print("=" * 90)
print("SPC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 90)

print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)

print("\nOriginal final-theme rows:")
print(
    len(df)
)

print("\nNormalized final SPC themes:")
print(
    len(final_spc_evidence)
)

print("\nSPC dimensions:")
print(
    len(dimension_summary)
)


print("\n")
print("-" * 90)
print("SPC DIMENSION SUMMARY")
print("-" * 90)

if len(dimension_summary) > 0:

    print(
        dimension_summary.to_string(
            index=False
        )
    )

else:

    print(
        "No dimensions generated."
    )


print("\n")
print("-" * 90)
print("QUALITY CHECKS")
print("-" * 90)

print(
    quality_checks_new.to_string(
        index=False
    )
)


print("\n")
print("=" * 90)
print("NEXT STAGE")
print("=" * 90)

print("""
Use these sheets for SPC questionnaire development:

13_Final_SPC_Evidence
14_SPC_Dimension_Summary
15_SPC_Dimension_Themes

The dimensions are qualitative content domains.
They are NOT automatically separate constructs.

The next stage is to use the strongest, conceptually
distinct SPC themes to generate candidate reflective
questionnaire items, followed by expert/content validation.
""")

SPC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\SPC_FINAL_QUALITATIVE_ANALYSIS_20260825_140437.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_SPC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Participants used: 26


SPC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\SPC_FINAL_EVIDENCE_AND_DIMENSIONS_20260825_150141.xlsx

Original final-theme rows:
16

Normalized final SPC themes:
51

SPC dimensions:
9


------------------------------------------------------------------------------------------
SPC DIMENSION SUMMARY
--------------------------

## DTG

In [9]:
# ================================================================
# DTG CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT FILE
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS P01-P26
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT DTG RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "DTG":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "DTG",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No DTG responses were found.

        Check that DTG appears in the participant sheets.
        """
    )

print("\nDTG original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT RESPONSES INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "DTG",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)

print("\nDTG cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW DTG THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("DTG RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. DTG NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# IMPORTANT:
# Do NOT put arbitrary categories here.
#
# After running the code, review the actual DTG RAW THEMES
# and create researcher-approved mappings.
#
# Example format:
#
# "data integrity":
#     "Data integrity",
#
# "accurate shared data":
#     "Data integrity",
#
# "data provenance":
#     "Data provenance",
#
# ------------------------------------------------

DTG_NORMALIZATION = {

    # ADD DTG MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in DTG_NORMALIZATION:

        normalized = DTG_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP NORMALIZED THEMES BACK TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × DTG THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "DTG_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EVERYTHING TO EXCEL
# ------------------------------------------------

output_file = Path(
    "DTG_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("DTG CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique DTG raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized DTG themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

DTG original responses:
26

DTG cleaned theme observations:
107


DTG RAW THEMES
                      Raw_Theme                     Theme_Clean                       Theme_Key
                         access                          access                          access
      access and change control       access and change control       access and change control
                 access control                  access control                  access control
               analytical trust                analytical trust                analytical trust
             Authoritative data              Authoritative data              authoritative data
                  authorization                   authorization  

In [24]:
# ================================================================
# COMPLETE DTG QUALITATIVE CODING ANALYSIS
# Input: DTG_Coding_20260825_132813.xlsx
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT
# ================================================================

input_file = Path("DTG_Coding_20260825_132813.xlsx")
CONSTRUCT = "DTG"

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as this notebook."
    )

print("=" * 70)
print("DTG QUALITATIVE CODING ANALYSIS")
print("=" * 70)

excel = pd.ExcelFile(input_file)

print("\nInput file:", input_file.resolve())
print("\nAvailable sheets:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 2. FLEXIBLE SHEET FINDER
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_DTG",
    "01_Original",
    "Original_DTG",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_DTG",
    "02_Raw",
    "Raw_DTG",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_DTG",
    "03_Cleaned",
    "Cleaned_DTG",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 3. READ DATA
# ================================================================

original_df = (
    pd.read_excel(input_file, sheet_name=original_sheet)
    if original_sheet else pd.DataFrame()
)

raw_df = (
    pd.read_excel(input_file, sheet_name=raw_sheet)
    if raw_sheet else pd.DataFrame()
)

clean_df = (
    pd.read_excel(input_file, sheet_name=clean_sheet)
    if clean_sheet else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the Coding/Normalized Coding sheet.\n\n"
        "Available sheets:\n" +
        "\n".join(str(x) for x in excel.sheet_names)
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 4. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:

        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 5. FLEXIBLE COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:

        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


# ================================================================
# 6. IDENTIFY CODING COLUMNS
# ================================================================

theme_key_col = find_column(
    coding_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

raw_theme_col = find_column(
    coding_df,
    [
        "Raw_Theme",
        "Raw Theme",
        "RawTheme"
    ]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    [
        "Decision"
    ]
)


if theme_key_col is None:
    raise ValueError(
        "Theme_Key column was not found."
    )

if raw_theme_col is None:
    raise ValueError(
        "Raw_Theme column was not found."
    )

if normalized_col is None:
    raise ValueError(
        "Normalized_Theme column was not found."
    )

if decision_col is None:
    raise ValueError(
        "Decision column was not found."
    )


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:

    raise ValueError(
        "\nParticipant column was not found.\n\n"
        "Available columns:\n" +
        str(list(clean_df.columns))
    )

if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col: "Participant"
        }
    )


# ================================================================
# 8. THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key was not found in the cleaned data.\n\n"
        "Available columns:\n" +
        str(list(clean_df.columns))
    )

if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key: "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        [
            "nan",
            "None",
            "",
            "NaN"
        ],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df["Participant"]
    .replace("", np.nan)
    .dropna()
    .unique()
)

print("\nNumber of participants:", len(participants))
print("Participants:", participants)


# ================================================================
# 12. MERGE CLEANED DATA + CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. CHECK UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df["Normalized_Theme"].isna()
].copy()

print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)

if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED DTG THEMES
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme["Normalized_Theme"],
    participant_theme["Participant"]
)


# ================================================================
# 16. ORDER PARTICIPANTS P01–P26
# ================================================================

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()

participant_columns = existing + other


# ================================================================
# 17. FREQUENCY
# ================================================================

matrix["Frequency"] = (
    matrix[participant_columns]
    .sum(axis=1)
)


# ================================================================
# 18. EXPERT PREVALENCE
# ================================================================

total_participants = len(participants)

if total_participants > 0:

    matrix["Percentage"] = (
        matrix["Frequency"]
        / total_participants
        * 100
    ).round(1)

else:

    matrix["Percentage"] = 0


# ================================================================
# 19. RANK DTG THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)

matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 20. DTG THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()

theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme": "Final_DTG_Theme",
        "Frequency": "Experts_Mentioning",
        "Percentage": "Percentage_of_Experts"
    }
)


# ================================================================
# 21. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary["Prevalence_Category"] = (
    theme_summary[
        "Percentage_of_Experts"
    ].apply(prevalence_category)
)


# ================================================================
# 22. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby("Normalized_Theme")
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)

coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)

coding_audit["Experts_Mentioning"] = (
    coding_audit["Experts_Mentioning"]
    .fillna(0)
    .astype(int)
)

if total_participants > 0:

    coding_audit["Percentage_of_Experts"] = (
        coding_audit["Experts_Mentioning"]
        / total_participants
        * 100
    ).round(1)

else:

    coding_audit["Percentage_of_Experts"] = 0


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 23. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)

if len(coding_df) > 0:

    decision_summary["Percentage"] = (
        decision_summary[
            "Number_of_Raw_Themes"
        ]
        / len(coding_df)
        * 100
    ).round(1)

else:

    decision_summary["Percentage"] = 0


# ================================================================
# 24. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=["Normalized_Theme"]
    )
    .groupby("Normalized_Theme")
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)

normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)

if total_participants > 0:

    normalization_summary[
        "Percentage_of_Experts"
    ] = (
        normalization_summary[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    normalization_summary[
        "Percentage_of_Experts"
    ] = 0


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 25. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_DTG_Themes"
]


# ================================================================
# 26. DTG CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": [CONSTRUCT],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df["Theme_Key"].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df["Normalized_Theme"].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df["Decision"] == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df["Decision"] == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 27. FINAL DTG EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()

final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme": "Final_DTG_Theme",
        "Raw_Themes": "Number_of_Raw_Themes",
        "Percentage_of_Experts": "Expert_Prevalence_%",
        "Keep_Count": "Raw_Themes_Kept",
        "Merge_Count": "Raw_Themes_Merged"
    }
)

final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ].apply(prevalence_category)
)


# ================================================================
# 28. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)

quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",
        "Original response rows",
        "Raw theme observations",
        "Unique raw themes",
        "Final normalized themes",
        "Unmapped themes",
        "Duplicate participant-theme records",
        "Keep decisions",
        "Merge decisions"

    ],

    "Result": [

        total_participants,
        len(original_df),
        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df["Decision"]
            == "Keep"
        ).sum(),

        (
            coding_df["Decision"]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",
        "PASS",
        "PASS"
    ]
})


# ================================================================
# 29. SAVE COMPLETE DTG WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"DTG_FINAL_QUALITATIVE_ANALYSIS_"
    f"{timestamp}.xlsx"
)

counter = 1

while output_file.exists():

    output_file = Path(
        f"DTG_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_DTG_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 30. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("DTG ANALYSIS COMPLETED")
print("=" * 70)

print("\nParticipants:", total_participants)

print("Original responses:", len(original_df))

print("Raw theme observations:", len(raw_df))

print("Cleaned theme observations:", len(clean_df))

print(
    "Unique raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Final normalized DTG themes:",
    coding_df["Normalized_Theme"].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df["Decision"] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df["Decision"] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 31. DISPLAY FINAL DTG THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL DTG THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_DTG_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 32. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(output_file.resolve())

print(
    "\nComplete DTG qualitative analysis workbook "
    "created successfully."
)

DTG QUALITATIVE CODING ANALYSIS

Input file: C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\DTG_Coding_20260825_132813.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Number of participants: 26
Participants: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of unmapped themes: 0


DTG ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 107
Cleaned theme observations: 107

In [41]:
# ================================================================
# DTG — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# DTG_FINAL_QUALITATIVE_ANALYSIS_20260825_140639.xlsx
#
# OUTPUT:
# DTG_FINAL_EVIDENCE_AND_DIMENSIONS_YYYYMMDD_HHMMSS.xlsx
#
# PURPOSE:
# 1. Read completed DTG qualitative analysis
# 2. Extract final DTG evidence
# 3. Remove capitalization duplicates
# 4. Group DTG themes into evidence-based content dimensions
# 5. Prepare the evidence for questionnaire-item development
#
# IMPORTANT:
# Dimensions are CONTENT DOMAINS, not additional constructs.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. INPUT FILE
# ================================================================

TARGET = "DTG_FINAL_QUALITATIVE_ANALYSIS_20260825_140639"

search_locations = [
    Path("/mnt/data"),
    Path("."),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop"
]

possible_files = []

for location in search_locations:

    if not location.exists():
        continue

    try:

        for ext in [".xlsx", ".xlsm", ".xls"]:

            possible_files.extend(
                location.rglob(TARGET + ext)
            )

    except Exception:
        pass


possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)


if len(possible_files) > 0:

    INPUT_FILE = possible_files[0]

else:

    # Search for workbook containing DTG final evidence
    candidates = []

    for location in search_locations:

        if not location.exists():
            continue

        try:

            for p in location.rglob("*.xlsx"):

                try:

                    test_xls = pd.ExcelFile(
                        p,
                        engine="openpyxl"
                    )

                    if "11_Final_DTG_Evidence" in test_xls.sheet_names:
                        candidates.append(
                            p.resolve()
                        )

                except Exception:
                    pass

        except Exception:
            pass


    if len(candidates) == 0:

        raise FileNotFoundError(
            "\nDTG input workbook was not found.\n"
            f"Expected: {TARGET}.xlsx"
        )

    INPUT_FILE = candidates[0]


print("=" * 90)
print("DTG FINAL QUALITATIVE ANALYSIS")
print("=" * 90)

print("\nInput file:")
print(INPUT_FILE)


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():
            return s

    return None


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:
        x = float(x)
    except:
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


# ================================================================
# 5. IDENTIFY SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_DTG_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


if final_sheet is None:

    raise ValueError(
        "11_Final_DTG_Evidence was not found."
    )


# ================================================================
# 6. LOAD FINAL DTG EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()


final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# ================================================================
# 7. IDENTIFY THEME COLUMN
# ================================================================

theme_col = None


if "Final_DTG_Theme" in final_evidence.columns:

    theme_col = "Final_DTG_Theme"

else:

    for c in final_evidence.columns:

        if (
            "DTG" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify the final DTG theme column."
    )


final_evidence[
    "Final_DTG_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE IMPORTANT COLUMNS
# ================================================================

if "Experts_Mentioning" not in final_evidence.columns:

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if "Expert_Prevalence_%" not in final_evidence.columns:

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_DTG_Theme"
    ] != ""
].copy()


# ================================================================
# 10. PARTICIPANT COUNT
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []

if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(cstr)


if len(participants) > 0:

    n_participants = len(participants)

else:

    # Your qualitative coding dataset
    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 12. NORMALIZE CAPITALIZATION DUPLICATES
# ================================================================

final_evidence[
    "_Theme_Normalized"
] = (
    final_evidence[
        "Final_DTG_Theme"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "_Prevalence_Numeric"
    ] = pd.to_numeric(
        final_evidence[
            "Expert_Prevalence_%"
        ],
        errors="coerce"
    ).fillna(0)

else:

    final_evidence[
        "_Prevalence_Numeric"
    ] = 0


final_evidence = (
    final_evidence
    .sort_values(
        "_Prevalence_Numeric",
        ascending=False
    )
)


final_evidence = (
    final_evidence
    .drop_duplicates(
        subset=[
            "_Theme_Normalized"
        ],
        keep="first"
    )
    .copy()
)


final_evidence = (
    final_evidence
    .drop(
        columns=[
            "_Theme_Normalized",
            "_Prevalence_Numeric"
        ],
        errors="ignore"
    )
)


# ================================================================
# 13. SORT FINAL EVIDENCE
# ================================================================

if "Experts_Mentioning" in final_evidence.columns:

    final_evidence[
        "_Sort"
    ] = pd.to_numeric(
        final_evidence[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    final_evidence = (
        final_evidence
        .sort_values(
            "_Sort",
            ascending=False
        )
        .drop(
            columns=["_Sort"]
        )
        .reset_index(drop=True)
    )


# ================================================================
# 14. FINAL DTG EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_DTG_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]


evidence_columns = [

    c
    for c in preferred_columns
    if c in final_evidence.columns

]


final_dtg_evidence = final_evidence[
    evidence_columns
].copy()


if "Rank" in final_dtg_evidence.columns:

    final_dtg_evidence = (
        final_dtg_evidence
        .drop(
            columns=["Rank"]
        )
    )


final_dtg_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_dtg_evidence) + 1
    )
)


# ================================================================
# 15. DTG DIMENSION MAPPING
# ================================================================
#
# These are qualitative CONTENT DOMAINS for item development.
# They are NOT separate constructs.
#
# DTG = Data Trust & Governance
#
# Core domains:
#   1. Data accuracy & quality
#   2. Data integrity & consistency
#   3. Data security & protection
#   4. Data access & availability
#   5. Data transparency & traceability
#   6. Data ownership & accountability
#   7. Governance policies & standards
#   8. Data sharing & controlled access
#   9. Data validation & monitoring
#  10. Compliance & responsible data governance
# ================================================================

def map_dtg_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. DATA ACCURACY & QUALITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "accuracy",
            "accurate data",
            "data quality",
            "quality of data",
            "high-quality data",
            "reliable data",
            "data reliability",
            "completeness",
            "complete data",
            "timely data",
            "data consistency",
            "consistency of data"

        ]
    ):

        return (
            "Data Accuracy & Quality"
        )


    # ------------------------------------------------------------
    # 2. DATA INTEGRITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "data integrity",
            "integrity",
            "tamper",
            "tamper-proof",
            "unaltered",
            "data authenticity",
            "authenticity",
            "data validation",
            "validation"

        ]
    ):

        return (
            "Data Integrity & Validation"
        )


    # ------------------------------------------------------------
    # 3. DATA SECURITY & PROTECTION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "security",
            "data security",
            "cybersecurity",
            "cyber security",
            "data protection",
            "protection of data",
            "privacy",
            "data privacy",
            "unauthorized access",
            "access protection",
            "confidentiality"

        ]
    ):

        return (
            "Data Security & Protection"
        )


    # ------------------------------------------------------------
    # 4. DATA ACCESS & AVAILABILITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "data access",
            "accessibility",
            "accessible data",
            "availability",
            "data availability",
            "real-time access",
            "timely access",
            "access to information",
            "information access"

        ]
    ):

        return (
            "Data Access & Availability"
        )


    # ------------------------------------------------------------
    # 5. TRANSPARENCY & TRACEABILITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "transparency",
            "transparent",
            "traceability",
            "traceable",
            "data provenance",
            "provenance",
            "audit trail",
            "visibility",
            "data visibility",
            "tracking"

        ]
    ):

        return (
            "Data Transparency & Traceability"
        )


    # ------------------------------------------------------------
    # 6. OWNERSHIP & ACCOUNTABILITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "ownership",
            "data owner",
            "accountability",
            "responsibility",
            "responsible data",
            "data stewardship",
            "stewardship",
            "custodian",
            "data custodian"

        ]
    ):

        return (
            "Data Ownership & Accountability"
        )


    # ------------------------------------------------------------
    # 7. GOVERNANCE POLICIES & STANDARDS
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "governance",
            "governance policy",
            "governance policies",
            "data policy",
            "data policies",
            "standards",
            "data standards",
            "rules",
            "governance framework",
            "governance structure",
            "procedures",
            "protocols"

        ]
    ):

        return (
            "Data Governance Policies & Standards"
        )


    # ------------------------------------------------------------
    # 8. DATA SHARING & COLLABORATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "data sharing",
            "information sharing",
            "sharing data",
            "controlled sharing",
            "secure sharing",
            "data exchange",
            "information exchange",
            "interorganizational data",
            "data collaboration",
            "collaboration"

        ]
    ):

        return (
            "Controlled Data Sharing & Collaboration"
        )


    # ------------------------------------------------------------
    # 9. MONITORING & AUDITING
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "monitoring",
            "data monitoring",
            "continuous monitoring",
            "audit",
            "auditing",
            "data audit",
            "quality monitoring",
            "governance monitoring",
            "checking",
            "review"

        ]
    ):

        return (
            "Data Monitoring & Auditing"
        )


    # ------------------------------------------------------------
    # 10. COMPLIANCE
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "compliance",
            "regulatory",
            "regulation",
            "legal",
            "law",
            "gdpr",
            "ethical",
            "ethics",
            "responsible use"

        ]
    ):

        return (
            "Data Compliance & Responsible Governance"
        )


    # ------------------------------------------------------------
    # 11. GENERAL DATA TRUST
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "trust",
            "trusted data",
            "data trust",
            "confidence in data",
            "confidence"

        ]
    ):

        return (
            "Data Trust & Confidence"
        )


    # ------------------------------------------------------------
    # 12. REVIEW
    # ------------------------------------------------------------

    return "Review Required"


final_dtg_evidence[
    "DTG_Dimension"
] = (
    final_dtg_evidence[
        "Final_DTG_Theme"
    ]
    .apply(
        map_dtg_dimension
    )
)


# ================================================================
# 16. MANUAL EXACT-THEME OVERRIDES
# ================================================================

manual_mapping = {

    "data accuracy":
        "Data Accuracy & Quality",

    "accuracy":
        "Data Accuracy & Quality",

    "data quality":
        "Data Accuracy & Quality",

    "data consistency":
        "Data Accuracy & Quality",

    "consistency":
        "Data Accuracy & Quality",

    "data integrity":
        "Data Integrity & Validation",

    "integrity":
        "Data Integrity & Validation",

    "data validation":
        "Data Integrity & Validation",

    "validation":
        "Data Integrity & Validation",

    "data security":
        "Data Security & Protection",

    "security":
        "Data Security & Protection",

    "data protection":
        "Data Security & Protection",

    "data privacy":
        "Data Security & Protection",

    "privacy":
        "Data Security & Protection",

    "confidentiality":
        "Data Security & Protection",

    "data access":
        "Data Access & Availability",

    "data availability":
        "Data Access & Availability",

    "availability":
        "Data Access & Availability",

    "accessibility":
        "Data Access & Availability",

    "transparency":
        "Data Transparency & Traceability",

    "traceability":
        "Data Transparency & Traceability",

    "data provenance":
        "Data Transparency & Traceability",

    "provenance":
        "Data Transparency & Traceability",

    "audit trail":
        "Data Transparency & Traceability",

    "data ownership":
        "Data Ownership & Accountability",

    "ownership":
        "Data Ownership & Accountability",

    "accountability":
        "Data Ownership & Accountability",

    "responsibility":
        "Data Ownership & Accountability",

    "data governance":
        "Data Governance Policies & Standards",

    "governance":
        "Data Governance Policies & Standards",

    "governance policies":
        "Data Governance Policies & Standards",

    "data policies":
        "Data Governance Policies & Standards",

    "data standards":
        "Data Governance Policies & Standards",

    "data sharing":
        "Controlled Data Sharing & Collaboration",

    "information sharing":
        "Controlled Data Sharing & Collaboration",

    "controlled data sharing":
        "Controlled Data Sharing & Collaboration",

    "data exchange":
        "Controlled Data Sharing & Collaboration",

    "monitoring":
        "Data Monitoring & Auditing",

    "data monitoring":
        "Data Monitoring & Auditing",

    "audit":
        "Data Monitoring & Auditing",

    "auditing":
        "Data Monitoring & Auditing",

    "compliance":
        "Data Compliance & Responsible Governance",

    "regulatory compliance":
        "Data Compliance & Responsible Governance",

    "ethical data use":
        "Data Compliance & Responsible Governance",

    "data trust":
        "Data Trust & Confidence",

    "trust":
        "Data Trust & Confidence",

    "trusted data":
        "Data Trust & Confidence"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_dtg_evidence[
            "Final_DTG_Theme"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            theme.lower()
        )
    )

    final_dtg_evidence.loc[
        mask,
        "DTG_Dimension"
    ] = dimension


# ================================================================
# 17. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_dtg_evidence
    .groupby(
        "DTG_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_DTG_Theme"
        ]
        .astype(str)
        .tolist()
    )


    experts = 0


    if "Experts_Mentioning" in group.columns:

        try:

            experts = int(
                pd.to_numeric(
                    group[
                        "Experts_Mentioning"
                    ],
                    errors="coerce"
                )
                .max()
            )

        except:

            experts = 0


    prevalence = (

        experts
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "DTG_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_DTG_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 18. SORT DIMENSIONS
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary[
        "_sort"
    ] = pd.to_numeric(
        dimension_summary[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_summary = (
        dimension_summary
        .sort_values(
            "_sort",
            ascending=False
        )
        .drop(
            columns=["_sort"]
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(
                columns=["Rank"]
            )
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 19. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "DTG_Dimension",

    "Final_DTG_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [

    c
    for c in dimension_theme_columns
    if c in final_dtg_evidence.columns

]


dimension_themes = final_dtg_evidence[
    dimension_theme_columns
].copy()


if "Experts_Mentioning" in dimension_themes.columns:

    dimension_themes[
        "_sort"
    ] = pd.to_numeric(
        dimension_themes[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_themes = (
        dimension_themes
        .sort_values(
            [
                "DTG_Dimension",
                "_sort"
            ],
            ascending=[
                True,
                False
            ]
        )
        .drop(
            columns=["_sort"]
        )
    )


# ================================================================
# 20. THEMES REQUIRING REVIEW
# ================================================================

unmapped_check = final_dtg_evidence[
    final_dtg_evidence[
        "DTG_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    review_columns = [

        "Final_DTG_Theme",

        "Experts_Mentioning",

        "Expert_Prevalence_%",

        "DTG_Dimension"

    ]


    review_columns = [

        c
        for c in review_columns
        if c in unmapped_check.columns

    ]


    unmapped_check = unmapped_check[
        review_columns
    ].copy()


    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )


else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All DTG themes mapped to a dimension."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final DTG evidence available",

    "Result":
        "PASS"
        if len(final_dtg_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_dtg_evidence)} normalized DTG themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used"

})


quality_rows.append({

    "Quality_Check":
        "DTG dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Evidence-based content domains"

})


review_count = (

    len(unmapped_check)

    if "Final_DTG_Theme"
    in unmapped_check.columns

    else 0

)


quality_rows.append({

    "Quality_Check":
        "Themes requiring manual review",

    "Result":
        review_count,

    "Details":
        "Themes assigned Review Required"

})


duplicate_count = int(
    final_dtg_evidence[
        "Final_DTG_Theme"
    ]
    .astype(str)
    .str.lower()
    .duplicated()
    .sum()
)


quality_rows.append({

    "Quality_Check":
        "Remaining capitalization duplicates",

    "Result":
        duplicate_count,

    "Details":
        "Should normally be zero"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:
        return sheets[sheet].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

coding_audit = get_sheet_or_empty(
    audit_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. OUTPUT FILE
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"DTG_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


# ================================================================
# 24. WRITE OUTPUT
# ================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    unmapped_check.to_excel(
        writer,
        sheet_name="11_Unmapped_Check",
        index=False
    )

    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    final_dtg_evidence.to_excel(
        writer,
        sheet_name="13_Final_DTG_Evidence",
        index=False
    )

    dimension_summary.to_excel(
        writer,
        sheet_name="14_DTG_Dimension_Summary",
        index=False
    )

    dimension_themes.to_excel(
        writer,
        sheet_name="15_DTG_Dimension_Themes",
        index=False
    )


# ================================================================
# 25. FINAL REPORT
# ================================================================

print("\n")
print("=" * 90)
print("DTG PROCESS COMPLETED SUCCESSFULLY")
print("=" * 90)

print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)

print("\nNormalized final DTG themes:")
print(
    len(final_dtg_evidence)
)

print("\nDTG content dimensions:")
print(
    len(dimension_summary)
)


print("\n")
print("-" * 90)
print("DTG DIMENSION SUMMARY")
print("-" * 90)

if len(dimension_summary) > 0:

    print(
        dimension_summary.to_string(
            index=False
        )
    )

else:

    print(
        "No dimensions generated."
    )


print("\n")
print("-" * 90)
print("QUALITY CHECKS")
print("-" * 90)

print(
    quality_checks_new.to_string(
        index=False
    )
)


print("\n")
print("=" * 90)
print("NEXT STAGE")
print("=" * 90)

print("""
Use these three sheets for DTG questionnaire development:

13_Final_DTG_Evidence
14_DTG_Dimension_Summary
15_DTG_Dimension_Themes

The DTG dimensions are qualitative content domains.
They are NOT separate constructs.

The next step is to convert the strongest and
conceptually distinct DTG themes into candidate
reflective questionnaire items, while avoiding
duplicate or overlapping items.
""")

DTG FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\DTG_FINAL_QUALITATIVE_ANALYSIS_20260825_140639.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_DTG_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Participants used: 26


DTG PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\DTG_FINAL_EVIDENCE_AND_DIMENSIONS_20260825_150329.xlsx

Normalized final DTG themes:
47

DTG content dimensions:
7


------------------------------------------------------------------------------------------
DTG DIMENSION SUMMARY
-------------------------------------------------

## DigIC

In [10]:
# ================================================================
# DigIC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT FILE
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS P01-P26
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT DigIC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().lower() == "digic":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "DigIC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No DigIC responses were found.

        Check that DigIC appears in the participant sheets.
        """
    )

print("\nDigIC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT RESPONSES INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "DigIC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)

print("\nDigIC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW DigIC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("DigIC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. DigIC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# DO NOT add categories before reviewing the actual
# DigIC themes produced by your experts.
#
# After seeing the RAW THEMES, add mappings such as:
#
# "raw theme":
#     "Normalized theme",
#
# ------------------------------------------------

DigIC_NORMALIZATION = {

    # ADD DigIC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in DigIC_NORMALIZATION:

        normalized = DigIC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP NORMALIZED THEMES BACK TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × DigIC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "DigIC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EVERYTHING TO EXCEL
# ------------------------------------------------

output_file = Path(
    "DigIC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("DigIC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique DigIC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized DigIC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

DigIC original responses:
26

DigIC cleaned theme observations:
86


DigIC RAW THEMES
                    Raw_Theme                   Theme_Clean                     Theme_Key
    actionable interpretation     actionable interpretation     actionable interpretation
           analytical support            analytical support            analytical support
                    analytics                     analytics                     analytics
            Anomaly detection             Anomaly detection             anomaly detection
            anomaly detection             anomaly detection             anomaly detection
        business implications         business implications         business implications
      

In [25]:
# ================================================================
# COMPLETE DigIC QUALITATIVE CODING ANALYSIS
# Input: DigIC_Coding_20260825_132853.xlsx
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT
# ================================================================

input_file = Path("DigIC_Coding_20260825_132853.xlsx")
CONSTRUCT = "DigIC"

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as this notebook."
    )

print("=" * 70)
print("DigIC QUALITATIVE CODING ANALYSIS")
print("=" * 70)

excel = pd.ExcelFile(input_file)

print("\nInput file:", input_file.resolve())
print("\nAvailable sheets:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 2. FIND SHEETS
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_DigIC",
    "01_Original",
    "Original_DigIC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_DigIC",
    "02_Raw",
    "Raw_DigIC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_DigIC",
    "03_Cleaned",
    "Cleaned_DigIC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])

print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 3. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(input_file, sheet_name=original_sheet)
    if original_sheet else pd.DataFrame()
)

raw_df = (
    pd.read_excel(input_file, sheet_name=raw_sheet)
    if raw_sheet else pd.DataFrame()
)

clean_df = (
    pd.read_excel(input_file, sheet_name=clean_sheet)
    if clean_sheet else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the Coding/Normalized Coding sheet.\n\n"
        "Available sheets:\n" +
        "\n".join(str(x) for x in excel.sheet_names)
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 4. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 5. FLEXIBLE COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:

        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


# ================================================================
# 6. IDENTIFY CODING COLUMNS
# ================================================================

theme_key_col = find_column(
    coding_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

raw_theme_col = find_column(
    coding_df,
    [
        "Raw_Theme",
        "Raw Theme",
        "RawTheme"
    ]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    [
        "Decision"
    ]
)


if theme_key_col is None:
    raise ValueError("Theme_Key column was not found.")

if raw_theme_col is None:
    raise ValueError("Raw_Theme column was not found.")

if normalized_col is None:
    raise ValueError("Normalized_Theme column was not found.")

if decision_col is None:
    raise ValueError("Decision column was not found.")


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:
    raise ValueError(
        "\nParticipant column was not found.\n\n"
        "Available columns:\n" +
        str(list(clean_df.columns))
    )

if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col: "Participant"
        }
    )


# ================================================================
# 8. THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:
    raise ValueError(
        "\nTheme_Key was not found in cleaned data.\n\n"
        "Available columns:\n" +
        str(list(clean_df.columns))
    )

if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key: "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        [
            "nan",
            "None",
            "",
            "NaN"
        ],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df["Participant"]
    .replace("", np.nan)
    .dropna()
    .unique()
)

print("\nNumber of participants:", len(participants))
print("Participants:", participants)


# ================================================================
# 12. MERGE CLEANED DATA WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. CHECK UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df["Normalized_Theme"].isna()
].copy()

print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)

if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED DigIC THEMES
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme["Normalized_Theme"],
    participant_theme["Participant"]
)


# ================================================================
# 16. ORDER PARTICIPANTS P01–P26
# ================================================================

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()

participant_columns = existing + other


# ================================================================
# 17. FREQUENCY
# ================================================================

matrix["Frequency"] = (
    matrix[participant_columns]
    .sum(axis=1)
)


# ================================================================
# 18. EXPERT PREVALENCE
# ================================================================

total_participants = len(participants)

if total_participants > 0:

    matrix["Percentage"] = (
        matrix["Frequency"]
        / total_participants
        * 100
    ).round(1)

else:

    matrix["Percentage"] = 0


# ================================================================
# 19. RANK DigIC THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)

matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 20. THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()

theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme": "Final_DigIC_Theme",
        "Frequency": "Experts_Mentioning",
        "Percentage": "Percentage_of_Experts"
    }
)


# ================================================================
# 21. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary["Prevalence_Category"] = (
    theme_summary[
        "Percentage_of_Experts"
    ].apply(prevalence_category)
)


# ================================================================
# 22. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby("Normalized_Theme")
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)

coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)

coding_audit["Experts_Mentioning"] = (
    coding_audit["Experts_Mentioning"]
    .fillna(0)
    .astype(int)
)

if total_participants > 0:

    coding_audit["Percentage_of_Experts"] = (
        coding_audit["Experts_Mentioning"]
        / total_participants
        * 100
    ).round(1)

else:

    coding_audit["Percentage_of_Experts"] = 0


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 23. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)

if len(coding_df) > 0:

    decision_summary["Percentage"] = (
        decision_summary[
            "Number_of_Raw_Themes"
        ]
        / len(coding_df)
        * 100
    ).round(1)

else:

    decision_summary["Percentage"] = 0


# ================================================================
# 24. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=["Normalized_Theme"]
    )
    .groupby("Normalized_Theme")
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)

normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)

normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)

if total_participants > 0:

    normalization_summary[
        "Percentage_of_Experts"
    ] = (
        normalization_summary[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    normalization_summary[
        "Percentage_of_Experts"
    ] = 0


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 25. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_DigIC_Themes"
]


# ================================================================
# 26. CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": [CONSTRUCT],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df["Theme_Key"].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df["Normalized_Theme"].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df["Decision"] == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df["Decision"] == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 27. FINAL DigIC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()

final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme": "Final_DigIC_Theme",
        "Raw_Themes": "Number_of_Raw_Themes",
        "Percentage_of_Experts": "Expert_Prevalence_%",
        "Keep_Count": "Raw_Themes_Kept",
        "Merge_Count": "Raw_Themes_Merged"
    }
)

final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ].apply(prevalence_category)
)


# ================================================================
# 28. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)

quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",
        "Original response rows",
        "Raw theme observations",
        "Unique raw themes",
        "Final normalized themes",
        "Unmapped themes",
        "Duplicate participant-theme records",
        "Keep decisions",
        "Merge decisions"

    ],

    "Result": [

        total_participants,
        len(original_df),
        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df["Decision"]
            == "Keep"
        ).sum(),

        (
            coding_df["Decision"]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",
        "PASS",
        "PASS"
    ]
})


# ================================================================
# 29. SAVE COMPLETE WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"DigIC_FINAL_QUALITATIVE_ANALYSIS_"
    f"{timestamp}.xlsx"
)

counter = 1

while output_file.exists():

    output_file = Path(
        f"DigIC_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_DigIC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 30. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("DigIC ANALYSIS COMPLETED")
print("=" * 70)

print("\nParticipants:", total_participants)

print("Original responses:", len(original_df))

print("Raw theme observations:", len(raw_df))

print("Cleaned theme observations:", len(clean_df))

print(
    "Unique raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Final normalized DigIC themes:",
    coding_df["Normalized_Theme"].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df["Decision"] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df["Decision"] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 31. DISPLAY FINAL DigIC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL DigIC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_DigIC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 32. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(output_file.resolve())

print(
    "\nComplete DigIC qualitative analysis workbook "
    "created successfully."
)

DigIC QUALITATIVE CODING ANALYSIS

Input file: C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\DigIC_Coding_20260825_132853.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Number of participants: 26
Participants: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of unmapped themes: 0


DigIC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 86
Cleaned theme observations

In [42]:
# ================================================================
# DigIC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# DigIC_FINAL_QUALITATIVE_ANALYSIS_20260825_140835.xlsx
#
# OUTPUT:
# DigIC_FINAL_EVIDENCE_AND_DIMENSIONS_YYYYMMDD_HHMMSS.xlsx
#
# PURPOSE:
# 1. Read completed DigIC qualitative analysis
# 2. Extract final DigIC evidence
# 3. Remove duplicate themes caused by capitalization/spacing
# 4. Organize themes into content domains
# 5. Produce evidence tables for questionnaire-item development
#
# NOTE:
# The dimensions generated here are CONTENT DOMAINS.
# They are NOT automatically treated as separate constructs.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os
import glob


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

TARGET = "DigIC_FINAL_QUALITATIVE_ANALYSIS_20260825_140835"

search_locations = [
    Path("/mnt/data"),
    Path("."),
    Path.cwd(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop"
]

possible_files = []

for location in search_locations:

    if not location.exists():
        continue

    try:

        for ext in [".xlsx", ".xlsm", ".xls"]:

            possible_files.extend(
                location.rglob(TARGET + ext)
            )

    except Exception:
        pass


possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)


# ================================================================
# 2. FALLBACK — SEARCH FOR ANY WORKBOOK CONTAINING DigIC EVIDENCE
# ================================================================

if len(possible_files) == 0:

    candidates = []

    for location in search_locations:

        if not location.exists():
            continue

        try:

            for p in location.rglob("*.xlsx"):

                try:

                    xls_test = pd.ExcelFile(
                        p,
                        engine="openpyxl"
                    )

                    if any(
                        "Final_DigIC_Evidence".lower()
                        in str(s).lower()
                        for s in xls_test.sheet_names
                    ):

                        candidates.append(
                            p.resolve()
                        )

                except Exception:
                    pass

        except Exception:
            pass


    if len(candidates) > 0:

        possible_files = candidates


# ================================================================
# 3. STOP IF FILE NOT FOUND
# ================================================================

if len(possible_files) == 0:

    raise FileNotFoundError(
        "\n\nINPUT FILE NOT FOUND.\n"
        "Expected file:\n"
        f"{TARGET}.xlsx\n\n"
        "Put the Excel file in the same folder as this notebook "
        "or in Downloads/Desktop and run the cell again."
    )


INPUT_FILE = possible_files[0]


print("=" * 90)
print("DigIC FINAL QUALITATIVE ANALYSIS")
print("=" * 90)

print("\nInput file found:")
print(INPUT_FILE)


# ================================================================
# 4. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 5. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 6. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    keyword = keyword.lower()

    for s in sheets.keys():

        if keyword in str(s).lower():

            return s

    return None


def clean_text(x):

    if pd.isna(x):

        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def numeric_value(x):

    try:

        return float(
            str(x)
            .replace("%", "")
            .strip()
        )

    except:

        return np.nan


def prevalence_category(x):

    x = numeric_value(x)

    if pd.isna(x):

        return "Not available"

    if x >= 75:

        return "Very High"

    elif x >= 50:

        return "High"

    elif x >= 25:

        return "Moderate"

    else:

        return "Low"


# ================================================================
# 7. IDENTIFY ORIGINAL SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_DigIC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


# ================================================================
# 8. IDENTIFY FINAL EVIDENCE SHEET
# ================================================================

if final_sheet is None:

    # Try more flexible search

    for s in sheets.keys():

        if (
            "final" in str(s).lower()
            and "digic" in str(s).lower()
            and "evidence" in str(s).lower()
        ):

            final_sheet = s
            break


if final_sheet is None:

    raise ValueError(
        "\nCould not find the DigIC final evidence sheet.\n"
        "Expected something similar to:\n"
        "11_Final_DigIC_Evidence"
    )


print(
    "\nFinal evidence sheet:",
    final_sheet
)


# ================================================================
# 9. LOAD FINAL DigIC EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()


final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


print(
    "\nFinal evidence columns:"
)

for i, c in enumerate(
    final_evidence.columns,
    start=1
):

    print(
        f"{i}. {c}"
    )


# ================================================================
# 10. FIND DigIC THEME COLUMN
# ================================================================

theme_col = None


possible_theme_columns = [

    "Final_DigIC_Theme",

    "Final_DigIC_Themes",

    "Final_Digital_Intelligence_Theme",

    "Final_Theme",

    "Theme",

    "Cleaned_Theme",

    "Normalized_Theme"

]


for c in possible_theme_columns:

    if c in final_evidence.columns:

        theme_col = c

        break


if theme_col is None:

    for c in final_evidence.columns:

        c_lower = str(c).lower()

        if (
            "theme" in c_lower
            and (
                "digic" in c_lower
                or "digital" in c_lower
            )
        ):

            theme_col = c

            break


if theme_col is None:

    for c in final_evidence.columns:

        if "theme" in str(c).lower():

            theme_col = c

            break


if theme_col is None:

    raise ValueError(
        "\nCould not identify the DigIC theme column."
    )


print(
    "\nTheme column detected:",
    theme_col
)


# ================================================================
# 11. STANDARDIZE THEME COLUMN
# ================================================================

final_evidence[
    "Final_DigIC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 12. IDENTIFY IMPORTANT COLUMNS
# ================================================================

# Experts mentioning

if "Experts_Mentioning" not in final_evidence.columns:

    for c in final_evidence.columns:

        cl = str(c).lower()

        if (
            "expert" in cl
            and "mention" in cl
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


# Prevalence

if "Expert_Prevalence_%" not in final_evidence.columns:

    for c in final_evidence.columns:

        cl = str(c).lower()

        if (
            "prevalence" in cl
            and (
                "%" in str(c)
                or "percent" in cl
            )
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 13. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_DigIC_Theme"
    ].astype(str).str.strip() != ""
].copy()


# ================================================================
# 14. DETERMINE NUMBER OF PARTICIPANTS
# ================================================================

n_participants = None


if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()

    participant_columns = []

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.fullmatch(
            r"P\d+",
            cstr,
            flags=re.IGNORECASE
        ):

            participant_columns.append(c)


    if len(participant_columns) > 0:

        n_participants = len(
            participant_columns
        )


if n_participants is None:

    # Try participant coverage

    if coverage_sheet is not None:

        coverage = sheets[
            coverage_sheet
        ]

        for c in coverage.columns:

            cl = str(c).lower()

            if (
                "participant" in cl
                and (
                    "id" in cl
                    or "code" in cl
                )
            ):

                vals = (
                    coverage[c]
                    .dropna()
                    .astype(str)
                    .str.strip()
                )

                vals = vals[
                    vals.str.match(
                        r"^P\d+$",
                        case=False
                    )
                ]

                if len(vals) > 0:

                    n_participants = (
                        vals.nunique()
                    )

                    break


if n_participants is None:

    # Last-resort value used in the existing
    # qualitative-analysis workflow.

    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 15. CREATE PREVALENCE CATEGORY
# ================================================================

if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 16. REMOVE DUPLICATES CAUSED BY CAPITALIZATION / SPACING
# ================================================================

final_evidence[
    "_normalized_theme"
] = (
    final_evidence[
        "Final_DigIC_Theme"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)


if "Experts_Mentioning" in final_evidence.columns:

    final_evidence[
        "_expert_sort"
    ] = pd.to_numeric(
        final_evidence[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)

else:

    final_evidence[
        "_expert_sort"
    ] = 0


final_evidence = (
    final_evidence
    .sort_values(
        "_expert_sort",
        ascending=False
    )
    .drop_duplicates(
        subset=[
            "_normalized_theme"
        ],
        keep="first"
    )
    .copy()
)


final_evidence = (
    final_evidence
    .drop(
        columns=[
            "_normalized_theme",
            "_expert_sort"
        ],
        errors="ignore"
    )
)


final_evidence = (
    final_evidence
    .reset_index(drop=True)
)


# ================================================================
# 17. SELECT FINAL EVIDENCE COLUMNS
# ================================================================

preferred_columns = [

    "Final_DigIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]


evidence_columns = [

    c
    for c in preferred_columns
    if c in final_evidence.columns

]


final_digic_evidence = final_evidence[
    evidence_columns
].copy()


# ================================================================
# 18. ADD RANK SAFELY
# ================================================================

# Avoid the previous:
# ValueError: cannot insert Rank, already exists

if "Rank" in final_digic_evidence.columns:

    final_digic_evidence = (
        final_digic_evidence
        .drop(
            columns=["Rank"]
        )
    )


final_digic_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_digic_evidence) + 1
    )
)


# ================================================================
# 19. DigIC CONTENT-DOMAIN MAPPING
# ================================================================
#
# These are CONTENT DOMAINS for questionnaire development.
#
# They do NOT automatically become separate dimensions
# of the measurement model.
#
# DigIC = Digital Intelligence Capability
#
# Main qualitative domains:
#
# 1. Digital Data Acquisition & Integration
# 2. Digital Data Processing & Analysis
# 3. Digital Information Interpretation
# 4. Digital Insight Generation
# 5. Real-Time Digital Monitoring
# 6. Predictive / Forward-Looking Intelligence
# 7. Digital Decision Support
# 8. Digital Information Sharing & Communication
# 9. Digital Learning & Adaptation
# 10. Digital Technology Integration
# 11. Digital Intelligence Responsiveness
# ================================================================

def map_digic_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. DIGITAL DATA ACQUISITION & INTEGRATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "data collection",
            "collect data",
            "data acquisition",
            "data gathering",
            "gather data",
            "data integration",
            "integrate data",
            "integrated data",
            "data sources",
            "multiple data sources",
            "combine data",
            "data aggregation",
            "data connectivity",
            "connected data"

        ]
    ):

        return (
            "Digital Data Acquisition & Integration"
        )


    # ------------------------------------------------------------
    # 2. DIGITAL DATA PROCESSING & ANALYSIS
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "data processing",
            "process data",
            "data analysis",
            "analyze data",
            "data analytics",
            "analytics",
            "analytical capability",
            "computational analysis",
            "data mining",
            "pattern analysis",
            "large datasets",
            "big data"

        ]
    ):

        return (
            "Digital Data Processing & Analysis"
        )


    # ------------------------------------------------------------
    # 3. DIGITAL INFORMATION INTERPRETATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "interpret data",
            "data interpretation",
            "interpret information",
            "information interpretation",
            "understand data",
            "understanding information",
            "meaning from data",
            "contextualize information",
            "contextual understanding",
            "information understanding"

        ]
    ):

        return (
            "Digital Information Interpretation"
        )


    # ------------------------------------------------------------
    # 4. DIGITAL INSIGHT GENERATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "insight",
            "insights",
            "generate insight",
            "generate insights",
            "actionable insight",
            "actionable insights",
            "intelligence generation",
            "knowledge generation",
            "derive knowledge",
            "extract knowledge",
            "business insight",
            "strategic insight"

        ]
    ):

        return (
            "Digital Insight Generation"
        )


    # ------------------------------------------------------------
    # 5. REAL-TIME DIGITAL MONITORING
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "real-time",
            "real time",
            "real time monitoring",
            "real-time monitoring",
            "continuous monitoring",
            "continuous data",
            "live data",
            "real-time information",
            "real-time visibility",
            "continuous visibility",
            "digital monitoring",
            "monitoring"

        ]
    ):

        return (
            "Real-Time Digital Monitoring"
        )


    # ------------------------------------------------------------
    # 6. PREDICTIVE / FORWARD-LOOKING INTELLIGENCE
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "predictive",
            "prediction",
            "forecast",
            "forecasting",
            "anticipate",
            "anticipation",
            "future trends",
            "future risk",
            "emerging trends",
            "early warning",
            "early detection",
            "proactive intelligence",
            "forward-looking"

        ]
    ):

        return (
            "Predictive & Forward-Looking Intelligence"
        )


    # ------------------------------------------------------------
    # 7. DIGITAL DECISION SUPPORT
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "decision support",
            "support decisions",
            "decision making",
            "decision-making",
            "better decisions",
            "informed decisions",
            "decision information",
            "decision intelligence",
            "digital decisions",
            "decision support systems"

        ]
    ):

        return (
            "Digital Decision Support"
        )


    # ------------------------------------------------------------
    # 8. DIGITAL INFORMATION SHARING & COMMUNICATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "information sharing",
            "data sharing",
            "share information",
            "share data",
            "information exchange",
            "data exchange",
            "digital communication",
            "communication",
            "collaboration",
            "information flow",
            "information exchange"

        ]
    ):

        return (
            "Digital Information Sharing & Communication"
        )


    # ------------------------------------------------------------
    # 9. DIGITAL LEARNING & ADAPTATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "learning",
            "organizational learning",
            "learn from data",
            "learning from data",
            "adapt",
            "adaptation",
            "adaptive",
            "continuous improvement",
            "improvement",
            "feedback",
            "learn from experience"

        ]
    ):

        return (
            "Digital Learning & Adaptation"
        )


    # ------------------------------------------------------------
    # 10. DIGITAL TECHNOLOGY INTEGRATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "digital technology",
            "technology integration",
            "technology adoption",
            "digital tools",
            "digital systems",
            "digital platform",
            "digital platforms",
            "technology infrastructure",
            "digital infrastructure",
            "integrated systems",
            "system integration",
            "ai",
            "artificial intelligence",
            "machine learning",
            "blockchain",
            "internet of things",
            "iot",
            "cloud"

        ]
    ):

        return (
            "Digital Technology Integration"
        )


    # ------------------------------------------------------------
    # 11. DIGITAL INTELLIGENCE RESPONSIVENESS
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "rapid response",
            "fast response",
            "quick response",
            "responsiveness",
            "respond quickly",
            "timely response",
            "timely action",
            "speed",
            "decision speed",
            "response capability",
            "rapid action"

        ]
    ):

        return (
            "Digital Intelligence Responsiveness"
        )


    # ------------------------------------------------------------
    # 12. GENERAL DIGITAL INTELLIGENCE
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "digital intelligence",
            "digital intelligent",
            "digital capability",
            "digital capability"

        ]
    ):

        return (
            "Digital Intelligence Capability"
        )


    # ------------------------------------------------------------
    # 13. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_digic_evidence[
    "DigIC_Dimension"
] = (
    final_digic_evidence[
        "Final_DigIC_Theme"
    ]
    .apply(
        map_digic_dimension
    )
)


# ================================================================
# 20. EXACT-THEME OVERRIDES
# ================================================================

manual_mapping = {

    "data collection":
        "Digital Data Acquisition & Integration",

    "data acquisition":
        "Digital Data Acquisition & Integration",

    "data integration":
        "Digital Data Acquisition & Integration",

    "data processing":
        "Digital Data Processing & Analysis",

    "data analysis":
        "Digital Data Processing & Analysis",

    "data analytics":
        "Digital Data Processing & Analysis",

    "data interpretation":
        "Digital Information Interpretation",

    "information interpretation":
        "Digital Information Interpretation",

    "insight generation":
        "Digital Insight Generation",

    "actionable insights":
        "Digital Insight Generation",

    "real-time monitoring":
        "Real-Time Digital Monitoring",

    "real time monitoring":
        "Real-Time Digital Monitoring",

    "continuous monitoring":
        "Real-Time Digital Monitoring",

    "predictive analytics":
        "Predictive & Forward-Looking Intelligence",

    "forecasting":
        "Predictive & Forward-Looking Intelligence",

    "prediction":
        "Predictive & Forward-Looking Intelligence",

    "decision support":
        "Digital Decision Support",

    "informed decision making":
        "Digital Decision Support",

    "data sharing":
        "Digital Information Sharing & Communication",

    "information sharing":
        "Digital Information Sharing & Communication",

    "digital communication":
        "Digital Information Sharing & Communication",

    "organizational learning":
        "Digital Learning & Adaptation",

    "adaptation":
        "Digital Learning & Adaptation",

    "continuous improvement":
        "Digital Learning & Adaptation",

    "digital technology integration":
        "Digital Technology Integration",

    "technology integration":
        "Digital Technology Integration",

    "digital infrastructure":
        "Digital Technology Integration",

    "responsiveness":
        "Digital Intelligence Responsiveness",

    "timely response":
        "Digital Intelligence Responsiveness",

    "digital intelligence":
        "Digital Intelligence Capability"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_digic_evidence[
            "Final_DigIC_Theme"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            theme.lower()
        )
    )

    final_digic_evidence.loc[
        mask,
        "DigIC_Dimension"
    ] = dimension


# ================================================================
# 21. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_digic_evidence
    .groupby(
        "DigIC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_DigIC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # Use maximum number of experts mentioning
    # a theme within the dimension rather than
    # summing them, because the same participant
    # can mention multiple themes.

    if "Experts_Mentioning" in group.columns:

        experts_numeric = pd.to_numeric(
            group[
                "Experts_Mentioning"
            ],
            errors="coerce"
        ).dropna()

        if len(experts_numeric) > 0:

            experts = int(
                experts_numeric.max()
            )

        else:

            experts = 0

    else:

        experts = 0


    prevalence = (

        experts
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "DigIC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_DigIC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 22. SAFE RANKING
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary[
        "_sort"
    ] = pd.to_numeric(
        dimension_summary[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_summary = (
        dimension_summary
        .sort_values(
            "_sort",
            ascending=False
        )
        .drop(
            columns=["_sort"]
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(
                columns=["Rank"]
            )
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 23. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "DigIC_Dimension",

    "Final_DigIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [

    c
    for c in dimension_theme_columns
    if c in final_digic_evidence.columns

]


dimension_themes = final_digic_evidence[
    dimension_theme_columns
].copy()


if "Experts_Mentioning" in dimension_themes.columns:

    dimension_themes[
        "_sort"
    ] = pd.to_numeric(
        dimension_themes[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_themes = (
        dimension_themes
        .sort_values(
            [
                "DigIC_Dimension",
                "_sort"
            ],
            ascending=[
                True,
                False
            ]
        )
        .drop(
            columns=["_sort"]
        )
        .reset_index(drop=True)
    )


# ================================================================
# 24. REVIEW-REQUIRED THEMES
# ================================================================

review_rows = final_digic_evidence[
    final_digic_evidence[
        "DigIC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(review_rows) > 0:

    review_columns = [

        "Final_DigIC_Theme",

        "Experts_Mentioning",

        "Expert_Prevalence_%",

        "DigIC_Dimension"

    ]


    review_columns = [

        c
        for c in review_columns
        if c in review_rows.columns

    ]


    unmapped_check = review_rows[
        review_columns
    ].copy()


    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )


else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All DigIC themes mapped to a content domain."
        ]

    })


# ================================================================
# 25. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final DigIC evidence available",

    "Result":
        "PASS"
        if len(final_digic_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_digic_evidence)} normalized DigIC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participant count",

    "Result":
        n_participants,

    "Details":
        "Number used for prevalence calculations"

})


quality_rows.append({

    "Quality_Check":
        "DigIC content domains generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative content domains"

})


review_count = (

    len(review_rows)

    if len(review_rows) > 0

    else 0

)


quality_rows.append({

    "Quality_Check":
        "Themes requiring manual review",

    "Result":
        review_count,

    "Details":
        "Themes assigned Review Required"

})


duplicate_count = int(
    final_digic_evidence[
        "Final_DigIC_Theme"
    ]
    .astype(str)
    .str.lower()
    .duplicated()
    .sum()
)


quality_rows.append({

    "Quality_Check":
        "Remaining duplicate themes",

    "Result":
        duplicate_count,

    "Details":
        "Should normally be zero"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 26. COPY ORIGINAL SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet_name):

    if sheet_name is not None:

        return sheets[
            sheet_name
        ].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

coding_audit = get_sheet_or_empty(
    audit_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 27. OUTPUT FILE
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"DigIC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


# ================================================================
# 28. WRITE OUTPUT
# ================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    if matrix_sheet is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_digic_evidence.to_excel(
        writer,
        sheet_name="11_Final_DigIC_Evidence",
        index=False
    )

    dimension_summary.to_excel(
        writer,
        sheet_name="12_DigIC_Dimension_Summary",
        index=False
    )

    dimension_themes.to_excel(
        writer,
        sheet_name="13_DigIC_Dimension_Themes",
        index=False
    )

    unmapped_check.to_excel(
        writer,
        sheet_name="14_Review_Required",
        index=False
    )

    quality_checks_new.to_excel(
        writer,
        sheet_name="15_Quality_Checks",
        index=False
    )


# ================================================================
# 29. FINAL REPORT
# ================================================================

print("\n")
print("=" * 90)
print("DigIC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 90)

print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)

print("\nNormalized DigIC themes:")
print(
    len(final_digic_evidence)
)

print("\nDigIC content domains:")
print(
    len(dimension_summary)
)


print("\n")
print("-" * 90)
print("DigIC DIMENSION SUMMARY")
print("-" * 90)

if len(dimension_summary) > 0:

    print(
        dimension_summary.to_string(
            index=False
        )
    )

else:

    print(
        "No dimensions generated."
    )


print("\n")
print("-" * 90)
print("QUALITY CHECKS")
print("-" * 90)

print(
    quality_checks_new.to_string(
        index=False
    )
)


print("\n")
print("=" * 90)
print("FILES / SHEETS CREATED")
print("=" * 90)

print("""
11_Final_DigIC_Evidence
    → Cleaned final DigIC qualitative themes

12_DigIC_Dimension_Summary
    → Higher-level content domains

13_DigIC_Dimension_Themes
    → Theme-to-domain mapping

14_Review_Required
    → Themes that need manual inspection

15_Quality_Checks
    → Automated quality-control results
""")


print("\n")
print("=" * 90)
print("IMPORTANT")
print("=" * 90)

print("""
The DigIC dimensions produced here are CONTENT DOMAINS,
not automatically separate measurement dimensions.

Their purpose is to ensure that questionnaire-item
development covers the full conceptual content of
Digital Intelligence Capability without simply
turning every qualitative theme into a separate factor.

The next research step is to use the strongest,
non-overlapping qualitative themes to draft candidate
reflective DigIC questionnaire items.
""")

DigIC FINAL QUALITATIVE ANALYSIS

Input file found:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\DigIC_FINAL_QUALITATIVE_ANALYSIS_20260825_140835.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_DigIC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Final evidence sheet: 11_Final_DigIC_Evidence

Final evidence columns:
1. Final_DigIC_Theme
2. Number_of_Raw_Themes
3. Experts_Mentioning
4. Expert_Prevalence_%
5. Raw_Themes_Kept
6. Raw_Themes_Merged
7. Prevalence_Category

Theme column detected: Final_DigIC_Theme

Participants used: 26


DigIC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Paper

## BRI

In [43]:
# ================================================================
# COMPLETE BRI QUALITATIVE CODING PIPELINE
# INPUT FILE: Themes.xlsx
# PARTICIPANTS: P01-P26
# ================================================================

import pandas as pd
from pathlib import Path

# ================================================================
# 1. FIND INPUT FILE AUTOMATICALLY
# ================================================================

possible_files = [
    Path("Themes.xlsx"),
    Path("Themes.xls"),
    Path("Themes.xlsm")
]

input_file = None

for f in possible_files:
    if f.exists():
        input_file = f
        break

if input_file is None:
    raise FileNotFoundError(
        "Themes file was not found. Put 'Themes.xlsx' in the same "
        "folder as your Jupyter notebook/Python script."
    )

print("=" * 70)
print("INPUT FILE FOUND")
print("=" * 70)
print(input_file.resolve())


# ================================================================
# 2. READ EXCEL WORKBOOK
# ================================================================

excel_file = pd.ExcelFile(input_file)

print("\nSheets found:")
print(excel_file.sheet_names)


# ================================================================
# 3. EXTRACT BRI FROM P01-P26
# ================================================================

original_rows = []
raw_rows = []

for sheet in excel_file.sheet_names:

    sheet_name = str(sheet).strip().upper()

    # ------------------------------------------------------------
    # Convert sheet name into participant number
    # Accepts:
    # P01, P1, 01, 1
    # Ignores:
    # Sheet1, Coding Dictionary, etc.
    # ------------------------------------------------------------

    if sheet_name.startswith("P"):
        number_part = sheet_name[1:]
    else:
        number_part = sheet_name

    try:
        participant_number = int(number_part)
    except (ValueError, TypeError):
        continue

    # Only process P01-P26
    if not 1 <= participant_number <= 26:
        continue

    participant = f"P{participant_number:02d}"

    print(f"Processing {sheet} -> {participant}")

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    # ------------------------------------------------------------
    # Find BRI row
    # Assumes:
    # column 2 = construct
    # column 3 = themes
    # ------------------------------------------------------------

    found_bri = False

    for _, row in df.iterrows():

        if len(row) < 3:
            continue

        construct = str(row.iloc[1]).strip().upper()

        if construct == "BRI":

            found_bri = True

            original_text = str(row.iloc[2]).strip()

            # Original participant response
            original_rows.append({
                "Participant": participant,
                "Construct": "BRI",
                "Original_BRI_Themes": original_text
            })

            # Split comma-separated themes
            themes = original_text.split(",")

            for theme in themes:

                theme = str(theme).strip()

                if theme and theme.lower() != "nan":

                    raw_rows.append({
                        "Participant": participant,
                        "Construct": "BRI",
                        "Raw_Theme": theme
                    })

            break

    if not found_bri:
        print(f"   WARNING: BRI not found in {participant}")


# ================================================================
# 4. CREATE RAW BRI DATAFRAME
# ================================================================

original_bri = pd.DataFrame(original_rows)

raw_bri = pd.DataFrame(raw_rows)

print("\n" + "=" * 70)
print("STAGE 1 — RAW BRI EXTRACTION")
print("=" * 70)

print(
    "Participants found:",
    original_bri["Participant"].nunique()
)

print(
    "Raw BRI observations:",
    len(raw_bri)
)


# ================================================================
# 5. CLEAN RAW THEMES
# ================================================================

clean_bri = raw_bri.copy()

clean_bri["Theme_Key"] = (
    clean_bri["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# Remove empty values
clean_bri = clean_bri[
    clean_bri["Theme_Key"].notna()
    & (clean_bri["Theme_Key"] != "")
    & (clean_bri["Theme_Key"] != "nan")
].copy()

# Remove duplicate mention of the same theme by same participant
clean_bri = clean_bri.drop_duplicates(
    subset=["Participant", "Theme_Key"]
)

print("\n" + "=" * 70)
print("STAGE 2 — CLEANING")
print("=" * 70)

print(
    "Unique participant × raw-theme observations:",
    len(clean_bri)
)


# ================================================================
# 6. BRI NORMALIZATION DICTIONARY
# ================================================================

bri_map = {

    # ---------------- SHARED RECORDS ----------------
    "common records": "Shared records",
    "shared records": "Shared records",
    "cross-party records": "Shared records",

    # ---------------- INFORMATION CONSISTENCY ----------------
    "information consistency": "Information consistency",
    "consistency": "Information consistency",
    "version consistency": "Information consistency",

    # ---------------- INFORMATION INTEGRITY ----------------
    "information integrity": "Information integrity",
    "trusted records": "Information integrity",

    # ---------------- TRACEABILITY ----------------
    "traceability": "End-to-end traceability",
    "cross-organizational traceability": "End-to-end traceability",
    "end-to-end traceability": "End-to-end traceability",

    # ---------------- TRANSACTION HISTORY ----------------
    "transaction history": "Transaction history",
    "shared transaction history": "Transaction history",
    "transaction sequence": "Transaction history",

    # ---------------- PRODUCT / PROVENANCE ----------------
    "product history": "Product/provenance history",
    "provenance": "Product/provenance history",
    "recall traceability": "Product/provenance history",

    # ---------------- HISTORICAL RECORD CONTINUITY ----------------
    "shared history": "Historical record continuity",
    "common history": "Historical record continuity",
    "continuous history": "Historical record continuity",

    # ---------------- EVENT / DISRUPTION ----------------
    "event tracing": "Event/disruption tracing",
    "disruption tracing": "Event/disruption tracing",
    "disruption identification": "Event/disruption tracing",
    "event sequence": "Event/disruption tracing",
    "shipment reconstruction": "Event/disruption tracing",

    # ---------------- INFORMATION TRAIL ----------------
    "information trail": "Information trail",
    "information history": "Information trail",

    # ---------------- INTERORGANIZATIONAL VISIBILITY ----------------
    "common visibility": "Interorganizational visibility",
    "interorganizational visibility": "Interorganizational visibility",
    "multi-party visibility": "Interorganizational visibility",
    "shared visibility": "Interorganizational visibility",
    "cross-party visibility": "Interorganizational visibility",
    "partner visibility": "Interorganizational visibility",

    # ---------------- PRODUCT / BATCH / LOT ----------------
    "batch tracking": "Product/batch/lot tracking",
    "batch visibility": "Product/batch/lot tracking",
    "lot visibility": "Product/batch/lot tracking",
    "product tracking": "Product/batch/lot tracking",

    # ---------------- SHIPMENT / MOVEMENT ----------------
    "shipment tracking": "Shipment/movement tracking",
    "movement tracking": "Shipment/movement tracking",
    "status and movement tracking": "Shipment/movement tracking",

    # ---------------- STATUS / LOCATION ----------------
    "status tracking": "Status/location tracking",
    "ownership/location tracking": "Status/location tracking",
    "status history": "Status/location tracking",

    # ---------------- TRANSACTION VISIBILITY ----------------
    "transaction visibility": "Transaction visibility",

    # ---------------- SHARED INFORMATION ----------------
    "shared information": "Shared information",

    # ---------------- VERIFICATION ----------------
    "verification": "Verification",
    "shared verification": "Verification",
    "shipment verification": "Verification",

    # ---------------- DISTINCT THEMES ----------------
    "immutability": "Immutability",
    "integration": "Integration",
    "interorganizational trust": "Interorganizational trust",
    "accountability": "Accountability",
    "reduced disputes": "Reduced disputes",
    "transparency": "Transparency",
    "visibility": "Visibility"
}


# ================================================================
# 7. APPLY NORMALIZATION
# ================================================================

clean_bri["Normalized_Theme"] = (
    clean_bri["Theme_Key"].map(bri_map)
)

unmapped = clean_bri[
    clean_bri["Normalized_Theme"].isna()
].copy()

print("\n" + "=" * 70)
print("STAGE 3 — NORMALIZATION")
print("=" * 70)

print(
    "Raw/cleaned themes:",
    len(clean_bri)
)

print(
    "Normalized themes:",
    clean_bri["Normalized_Theme"].nunique()
)

print(
    "Unmapped themes:",
    len(unmapped)
)

if len(unmapped) > 0:

    print("\nWARNING — THESE THEMES NEED CODING:")
    print(
        unmapped[
            ["Raw_Theme", "Theme_Key"]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 8. MERGE / KEEP DECISION
# ================================================================

decision_map = {

    "common records": "Merge",
    "shared records": "Keep",
    "cross-party records": "Merge",

    "information consistency": "Keep",
    "consistency": "Merge",
    "version consistency": "Merge",

    "information integrity": "Keep",
    "trusted records": "Merge",

    "traceability": "Merge",
    "cross-organizational traceability": "Merge",
    "end-to-end traceability": "Keep",

    "transaction history": "Keep",
    "shared transaction history": "Merge",
    "transaction sequence": "Merge",

    "product history": "Merge",
    "provenance": "Keep",
    "recall traceability": "Merge",

    "shared history": "Merge",
    "common history": "Merge",
    "continuous history": "Merge",

    "event tracing": "Keep",
    "disruption tracing": "Merge",
    "disruption identification": "Merge",
    "event sequence": "Merge",
    "shipment reconstruction": "Merge",

    "information trail": "Keep",
    "information history": "Merge",

    "common visibility": "Merge",
    "interorganizational visibility": "Keep",
    "multi-party visibility": "Merge",
    "shared visibility": "Merge",
    "cross-party visibility": "Merge",
    "partner visibility": "Merge",

    "batch tracking": "Keep",
    "batch visibility": "Merge",
    "lot visibility": "Merge",
    "product tracking": "Merge",

    "shipment tracking": "Keep",
    "movement tracking": "Merge",
    "status and movement tracking": "Merge",

    "status tracking": "Keep",
    "ownership/location tracking": "Merge",
    "status history": "Merge",

    "transaction visibility": "Keep",

    "shared information": "Keep",

    "verification": "Keep",
    "shared verification": "Merge",
    "shipment verification": "Merge",

    "immutability": "Keep",
    "integration": "Keep",
    "interorganizational trust": "Keep",
    "accountability": "Keep",
    "reduced disputes": "Keep",
    "transparency": "Keep",
    "visibility": "Keep"
}

clean_bri["Decision"] = (
    clean_bri["Theme_Key"].map(decision_map)
)

uncoded_decisions = clean_bri[
    clean_bri["Decision"].isna()
].copy()

clean_bri["Reason"] = clean_bri["Decision"].map({

    "Merge":
        "Conceptually overlaps with other raw expressions representing the same underlying theme.",

    "Keep":
        "Retained as a distinct conceptual aspect of BRI."
})


# ================================================================
# 9. PARTICIPANT × NORMALIZED THEME MATRIX
# ================================================================

coded_bri = clean_bri[
    clean_bri["Normalized_Theme"].notna()
].copy()

coded_bri = coded_bri.drop_duplicates(
    subset=[
        "Participant",
        "Normalized_Theme"
    ]
)

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix = pd.crosstab(
    coded_bri["Normalized_Theme"],
    coded_bri["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)


# ================================================================
# 10. FREQUENCY AND PERCENTAGE
# ================================================================

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"] /
    len(participants) *
    100
).round(1)

summary = (
    matrix[
        ["Frequency", "Percentage"]
    ]
    .sort_values(
        "Frequency",
        ascending=False
    )
    .reset_index()
)

summary = summary.rename(
    columns={
        "Normalized_Theme": "Theme"
    }
)


# ================================================================
# 11. NORMALIZED CODING DICTIONARY
# ================================================================

coding_dictionary = (
    clean_bri[
        [
            "Raw_Theme",
            "Theme_Key",
            "Normalized_Theme",
            "Decision",
            "Reason"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["Normalized_Theme", "Raw_Theme"]
    )
)


# ================================================================
# 12. CREATE OUTPUT EXCEL
# ================================================================

output_file = Path(
    "BRI_Complete_Qualitative_Coding.xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    # Original participant responses
    original_bri.to_excel(
        writer,
        sheet_name="01_Original_BRI",
        index=False
    )

    # Raw extracted themes
    raw_bri.to_excel(
        writer,
        sheet_name="02_Raw_BRI",
        index=False
    )

    # Cleaned themes
    clean_bri[
        [
            "Participant",
            "Construct",
            "Raw_Theme",
            "Theme_Key"
        ]
    ].to_excel(
        writer,
        sheet_name="03_Cleaned_BRI",
        index=False
    )

    # Normalization/coding dictionary
    coding_dictionary.to_excel(
        writer,
        sheet_name="04_Normalized_Coding",
        index=False
    )

    # Participant × theme matrix
    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix"
    )

    # Frequency and percentage
    summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    # Unmapped themes
    if len(unmapped) > 0:

        unmapped[
            [
                "Participant",
                "Raw_Theme",
                "Theme_Key"
            ]
        ].drop_duplicates().to_excel(
            writer,
            sheet_name="07_Unmapped_Check",
            index=False
        )

    else:

        pd.DataFrame({
            "Status": [
                "All BRI themes were successfully normalized."
            ]
        }).to_excel(
            writer,
            sheet_name="07_Unmapped_Check",
            index=False
        )

    # Decision check
    if len(uncoded_decisions) > 0:

        uncoded_decisions[
            [
                "Participant",
                "Raw_Theme",
                "Theme_Key",
                "Normalized_Theme"
            ]
        ].drop_duplicates().to_excel(
            writer,
            sheet_name="08_Decision_Check",
            index=False
        )

    else:

        pd.DataFrame({
            "Status": [
                "All BRI themes have a Merge/Keep decision."
            ]
        }).to_excel(
            writer,
            sheet_name="08_Decision_Check",
            index=False
        )


# ================================================================
# 13. FINAL REPORT
# ================================================================

print("\n" + "=" * 70)
print("BRI QUALITATIVE CODING COMPLETED")
print("=" * 70)

print("Participants detected:",
      original_bri["Participant"].nunique())

print("Raw BRI observations:",
      len(raw_bri))

print("Cleaned observations:",
      len(clean_bri))

print("Normalized themes:",
      clean_bri["Normalized_Theme"].nunique())

print("Merge decisions:",
      (clean_bri["Decision"] == "Merge").sum())

print("Keep decisions:",
      (clean_bri["Decision"] == "Keep").sum())

print("Unmapped themes:",
      len(unmapped))

print("Themes without decision:",
      len(uncoded_decisions))

print("\nTOP BRI THEMES")
print(
    summary.to_string(index=False)
)

print("\n" + "=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(output_file.resolve())

INPUT FILE FOUND
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\Themes.xlsx

Sheets found:
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26']
Processing 1 -> P01
Processing 2 -> P02
Processing 3 -> P03
Processing 4 -> P04
Processing 5 -> P05
Processing 6 -> P06
Processing 7 -> P07
Processing 8 -> P08
Processing 9 -> P09
Processing 10 -> P10
Processing 11 -> P11
Processing 12 -> P12
Processing 13 -> P13
Processing 14 -> P14
Processing 15 -> P15
Processing 16 -> P16
Processing 17 -> P17
Processing 18 -> P18
Processing 19 -> P19
Processing 20 -> P20
Processing 21 -> P21
Processing 22 -> P22
Processing 23 -> P23
Processing 24 -> P24
Processing 25 -> P25
Processing 26 -> P26

STAGE 1 — RAW BRI EXTRACTION
Participants found: 26
Raw BRI observations: 87

STAGE 2 — CLEANING
Unique participant × raw-theme observations: 87

STAGE 3 

In [46]:
# ================================================================
# COMPLETE BRI QUALITATIVE ANALYSIS
# ================================================================
#
# INPUT:
#     BRI_Complete_Qualitative_Coding.xlsx
#
# OUTPUT:
#     BRI_FINAL_QUALITATIVE_ANALYSIS.xlsx
#
# This code:
#   1. Reads the completed BRI coding workbook
#   2. Preserves the original responses
#   3. Preserves raw themes
#   4. Preserves cleaned themes
#   5. Preserves normalized coding
#   6. Rebuilds the participant × theme matrix
#   7. Recalculates theme frequency and percentage
#   8. Creates a ranked theme summary
#   9. Creates a coding audit
#  10. Creates participant coverage
#  11. Creates construct-level statistics
#  12. Creates a final BRI evidence table
#  13. Saves EVERYTHING into one Excel workbook
#
# ================================================================


import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

input_file = Path("BRI_Complete_Qualitative_Coding.xlsx")

# If exact filename is not found, search for an Excel file
# containing "BRI" in its name.

if not input_file.exists():

    possible_files = list(
        Path(".").glob("*BRI*.xlsx")
    )

    if len(possible_files) == 0:

        raise FileNotFoundError(
            "\nNo BRI Excel file was found.\n"
            "Please make sure the file is in the same folder "
            "as your Python notebook/script and is named:\n\n"
            "BRI_Complete_Qualitative_Coding.xlsx"
        )

    elif len(possible_files) == 1:

        input_file = possible_files[0]

    else:

        print("Multiple BRI files were found:")

        for i, f in enumerate(possible_files):
            print(f"{i}: {f.name}")

        raise ValueError(
            "\nMore than one BRI Excel file was found. "
            "Keep only the correct one in the folder."
        )


print("=" * 70)
print("BRI QUALITATIVE ANALYSIS")
print("=" * 70)

print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")

for sheet in excel.sheet_names:
    print(" -", sheet)


# ================================================================
# 3. READ EXISTING BRI SHEETS
# ================================================================

# ------------------------------------------------
# Original responses
# ------------------------------------------------

original_df = pd.read_excel(
    input_file,
    sheet_name="01_Original_BRI"
)


# ------------------------------------------------
# Raw themes
# ------------------------------------------------

raw_df = pd.read_excel(
    input_file,
    sheet_name="02_Raw_BRI"
)


# ------------------------------------------------
# Cleaned themes
# ------------------------------------------------

clean_df = pd.read_excel(
    input_file,
    sheet_name="03_Cleaned_BRI"
)


# ------------------------------------------------
# Normalized coding
# ------------------------------------------------

coding_df = pd.read_excel(
    input_file,
    sheet_name="04_Normalized_Coding"
)


# ================================================================
# 4. STANDARDIZE COLUMN NAMES
# ================================================================

# Make sure the important columns are correctly named.

original_df.columns = [
    str(c).strip()
    for c in original_df.columns
]

raw_df.columns = [
    str(c).strip()
    for c in raw_df.columns
]

clean_df.columns = [
    str(c).strip()
    for c in clean_df.columns
]

coding_df.columns = [
    str(c).strip()
    for c in coding_df.columns
]


# ================================================================
# 5. BASIC DATA CLEANING
# ================================================================

# Clean theme keys

if "Theme_Key" in clean_df.columns:

    clean_df["Theme_Key"] = (
        clean_df["Theme_Key"]
        .astype(str)
        .str.strip()
        .str.lower()
    )


coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .astype(str)
    .str.strip()
    .str.lower()
)


coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .astype(str)
    .str.strip()
)


coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype(str)
    .str.strip()
)


coding_df["Decision"] = (
    coding_df["Decision"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 6. CHECK PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df["Participant"]
    .dropna()
    .astype(str)
    .unique()
)

print("\nParticipants identified:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 7. CHECK FOR UNMAPPED THEMES
# ================================================================

unmapped = coding_df[
    coding_df["Normalized_Theme"].isin(
        ["", "nan", "None"]
    )
].copy()


# Also check for missing values

unmapped = coding_df[
    coding_df["Normalized_Theme"].isna()
    |
    (
        coding_df["Normalized_Theme"]
        .astype(str)
        .str.strip()
        .isin(["", "nan", "None"])
    )
].copy()


# ================================================================
# 8. REBUILD PARTICIPANT-LEVEL NORMALIZED DATA
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 9. CHECK FOR MISSING MAPPINGS
# ================================================================

missing_mapping = coded_df[
    coded_df["Normalized_Theme"].isna()
    |
    (
        coded_df["Normalized_Theme"]
        .astype(str)
        .str.strip()
        .isin(["", "nan", "None"])
    )
].copy()


if len(missing_mapping) > 0:

    print("\nWARNING:")
    print(
        "Some cleaned themes do not have a normalized theme."
    )

    print(
        missing_mapping[
            [
                "Participant",
                "Raw_Theme",
                "Theme_Key"
            ]
        ].to_string(index=False)
    )

else:

    print(
        "\nAll cleaned BRI themes have a normalized theme."
    )


# ================================================================
# 10. REMOVE DUPLICATE PARTICIPANT-THEME COMBINATIONS
# ================================================================

# A participant mentioning the same normalized theme several
# times should count ONCE for participant-level prevalence.

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .drop_duplicates()
    .copy()
)


# ================================================================
# 11. CREATE PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme["Normalized_Theme"],
    participant_theme["Participant"]
)


# Ensure P01-P26 appear in correct order

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]


matrix = matrix.reindex(
    columns=expected_participants,
    fill_value=0
)


matrix = matrix.reset_index()


# ================================================================
# 12. FREQUENCY AND PERCENTAGE
# ================================================================

matrix["Frequency"] = matrix[
    expected_participants
].sum(axis=1)


matrix["Percentage"] = (
    matrix["Frequency"]
    / len(expected_participants)
    * 100
).round(1)


# ================================================================
# 13. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    by=[
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


matrix.insert(
    0,
    "Rank",
    range(1, len(matrix) + 1)
)


# ================================================================
# 14. FINAL THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_BRI_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 15. ADD INTERPRETIVE PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = theme_summary[
    "Percentage_of_Experts"
].apply(
    prevalence_category
)


# ================================================================
# 16. CODING AUDIT
# ================================================================

coding_audit = coding_df.copy()


coding_audit["Normalized_Theme"] = (
    coding_audit["Normalized_Theme"]
    .astype(str)
    .str.strip()
)


# Count how many participants mentioned each normalized theme

theme_participant_counts = (
    participant_theme
    .groupby("Normalized_Theme")
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_audit.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit["Experts_Mentioning"] = (
    coding_audit["Experts_Mentioning"]
    .fillna(0)
    .astype(int)
)


coding_audit["Percentage_of_Experts"] = (
    coding_audit["Experts_Mentioning"]
    / len(expected_participants)
    * 100
).round(1)


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 17. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


decision_summary[
    "Percentage"
] = (
    decision_summary[
        "Number_of_Raw_Themes"
    ]
    / len(coding_df)
    * 100
).round(1)


# ================================================================
# 18. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .groupby("Normalized_Theme")
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),
        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),
        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = normalization_summary.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


normalization_summary[
    "Percentage_of_Experts"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    / len(expected_participants)
    * 100
).round(1)


normalization_summary = normalization_summary.sort_values(
    "Experts_Mentioning",
    ascending=False
).reset_index(drop=True)


# ================================================================
# 19. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby("Participant")
    .size()
    .reindex(
        expected_participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_Themes"
]


# ================================================================
# 20. CONSTRUCT-LEVEL STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": ["BRI"],

    "Participants": [
        len(expected_participants)
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df["Theme_Key"].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 21. FINAL BRI EVIDENCE TABLE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_BRI_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Experts_Mentioning":
            "Experts_Mentioning",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence["Prevalence_Category"] = (
    final_evidence[
        "Expert_Prevalence_%"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 22. CREATE QUALITY-CHECK TABLE
# ================================================================

quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme combinations",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        len(expected_participants),

        len(original_df),

        len(raw_df),

        coding_df["Theme_Key"].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        (
            len(coded_df)
            -
            len(participant_theme)
        ),

        (
            coding_df["Decision"]
            == "Keep"
        ).sum(),

        (
            coding_df["Decision"]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if len(expected_participants) == 26
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"

    ]

})


# ================================================================
# 23. CREATE OUTPUT FILE
# ================================================================

output_file = Path(
    "BRI_FINAL_QUALITATIVE_ANALYSIS_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)


# ================================================================
# 24. WRITE COMPLETE EXCEL WORKBOOK
# ================================================================

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    # ------------------------------------------------
    # Original evidence
    # ------------------------------------------------

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    # ------------------------------------------------
    # Raw coding
    # ------------------------------------------------

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    # ------------------------------------------------
    # Cleaned coding
    # ------------------------------------------------

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    # ------------------------------------------------
    # Complete coding dictionary
    # ------------------------------------------------

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )


    # ------------------------------------------------
    # Participant × theme matrix
    # ------------------------------------------------

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )


    # ------------------------------------------------
    # Theme summary
    # ------------------------------------------------

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    # ------------------------------------------------
    # Normalization summary
    # ------------------------------------------------

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    # ------------------------------------------------
    # Decision summary
    # ------------------------------------------------

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    # ------------------------------------------------
    # Participant coverage
    # ------------------------------------------------

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    # ------------------------------------------------
    # Construct statistics
    # ------------------------------------------------

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )


    # ------------------------------------------------
    # Final evidence table
    # ------------------------------------------------

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_BRI_Evidence",
        index=False
    )


    # ------------------------------------------------
    # Quality checks
    # ------------------------------------------------

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    # ------------------------------------------------
    # Unmapped themes
    # ------------------------------------------------

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 25. FINAL CONSOLE REPORT
# ================================================================

print("\n")
print("=" * 70)
print("BRI ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nParticipants:",
    len(expected_participants)
)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Final normalized BRI themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df["Decision"]
        == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df["Decision"]
        == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 26. SHOW FINAL BRI THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL BRI THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_BRI_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 27. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(
    output_file.resolve()
)

print("\nThe complete BRI analysis workbook has been created.")

BRI QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\BRI_Complete_Qualitative_Coding.xlsx

Sheets found:
 - 01_Original_BRI
 - 02_Raw_BRI
 - 03_Cleaned_BRI
 - 04_Normalized_Coding
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Unmapped_Check
 - 08_Decision_Check

Participants identified:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

All cleaned BRI themes have a normalized theme.


BRI ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 87
Cleaned theme observations: 87
Unique raw themes: 55
Final normalized BRI themes: 23
Keep decisions: 25
Merge decisions: 35
Unmapped themes: 0


FINAL BRI THEMES
               Final_BRI_Theme  Experts_Mentioning  Expert_Prevalence_% Prevalence_Category

In [47]:
# =====================================================================
# BRI — FINAL QUALITATIVE EVIDENCE + CONTENT-DOMAIN ANALYSIS
# =====================================================================
#
# INPUT:
# BRI_FINAL_QUALITATIVE_ANALYSIS_20260821_094624.xlsx
#
# OUTPUT:
# BRI_FINAL_EVIDENCE_AND_DIMENSIONS_YYYYMMDD_HHMMSS.xlsx
#
# PURPOSE:
# 1. Read completed BRI qualitative analysis
# 2. Extract final BRI evidence
# 3. Remove duplicate themes caused by capitalization/spacing
# 4. Organize themes into BRI content domains
# 5. Calculate expert prevalence
# 6. Produce theme-to-domain evidence tables
# 7. Produce quality-control checks
#
# IMPORTANT:
# The six BRI domains generated here are CONTENT DOMAINS.
# They are NOT automatically treated as six statistical dimensions
# in the later PLS-SEM measurement model.
# =====================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# =====================================================================
# 1. INPUT FILE
# =====================================================================

TARGET = "BRI_FINAL_QUALITATIVE_ANALYSIS_20260825_151306"


# Search locations
search_locations = [
    Path("/mnt/data"),
    Path("."),
    Path.cwd(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop"
]


possible_files = []


for location in search_locations:

    if not location.exists():
        continue

    try:

        for ext in [".xlsx", ".xlsm", ".xls"]:

            possible_files.extend(
                location.rglob(TARGET + ext)
            )

    except Exception:
        pass


possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)


# =====================================================================
# 2. FALLBACK SEARCH
# =====================================================================

if len(possible_files) == 0:

    print("\nExact filename not found.")
    print("Searching for another BRI qualitative workbook...\n")

    candidates = []

    for location in search_locations:

        if not location.exists():
            continue

        try:

            for p in location.rglob("*.xlsx"):

                try:

                    xls_test = pd.ExcelFile(
                        p,
                        engine="openpyxl"
                    )

                    sheet_names_lower = [
                        str(s).lower()
                        for s in xls_test.sheet_names
                    ]

                    if any(
                        (
                            "final_bri_evidence" in s
                            or
                            ("final" in s and "bri" in s and "evidence" in s)
                        )
                        for s in sheet_names_lower
                    ):

                        candidates.append(
                            p.resolve()
                        )

                except Exception:
                    pass

        except Exception:
            pass


    if len(candidates) > 0:

        possible_files = candidates


# =====================================================================
# 3. STOP IF INPUT NOT FOUND
# =====================================================================

if len(possible_files) == 0:

    raise FileNotFoundError(
        "\n\nINPUT FILE NOT FOUND.\n\n"
        f"Expected:\n{TARGET}.xlsx\n\n"
        "Please put the Excel workbook in the same folder as "
        "your notebook or in Downloads/Desktop.\n"
    )


INPUT_FILE = possible_files[0]


print("=" * 90)
print("BRI FINAL QUALITATIVE EVIDENCE + CONTENT-DOMAIN ANALYSIS")
print("=" * 90)

print("\nInput file found:")
print(INPUT_FILE)


# =====================================================================
# 4. READ WORKBOOK
# =====================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)


print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# =====================================================================
# 5. LOAD ALL SHEETS
# =====================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# =====================================================================
# 6. HELPER FUNCTIONS
# =====================================================================

def find_sheet(keyword):

    keyword = keyword.lower()

    for s in sheets.keys():

        if keyword in str(s).lower():

            return s

    return None


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def numeric_value(x):

    try:

        return float(
            str(x)
            .replace("%", "")
            .strip()
        )

    except:

        return np.nan


def prevalence_category(x):

    x = numeric_value(x)

    if pd.isna(x):
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


# =====================================================================
# 7. IDENTIFY ORIGINAL SHEETS
# =====================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_BRI_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


# =====================================================================
# 8. FIND FINAL BRI EVIDENCE SHEET
# =====================================================================

if final_sheet is None:

    for s in sheets.keys():

        sl = str(s).lower()

        if (
            "final" in sl
            and "bri" in sl
            and "evidence" in sl
        ):

            final_sheet = s
            break


if final_sheet is None:

    raise ValueError(
        "\nCould not find the BRI final evidence sheet.\n"
        "Expected something similar to:\n"
        "11_Final_BRI_Evidence"
    )


print(
    "\nFinal evidence sheet:",
    final_sheet
)


# =====================================================================
# 9. LOAD FINAL BRI EVIDENCE
# =====================================================================

final_evidence = sheets[
    final_sheet
].copy()


final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


print("\nFinal evidence columns:")

for i, c in enumerate(
    final_evidence.columns,
    start=1
):

    print(
        f"{i}. {c}"
    )


# =====================================================================
# 10. FIND BRI THEME COLUMN
# =====================================================================

theme_col = None


possible_theme_columns = [

    "Final_BRI_Theme",

    "Final_BRI_Themes",

    "Final_Theme",

    "Theme",

    "Cleaned_Theme",

    "Normalized_Theme"

]


for c in possible_theme_columns:

    if c in final_evidence.columns:

        theme_col = c
        break


if theme_col is None:

    for c in final_evidence.columns:

        cl = str(c).lower()

        if (
            "theme" in cl
            and "bri" in cl
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "theme" in str(c).lower():

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "\nCould not identify the BRI theme column."
    )


print(
    "\nTheme column detected:",
    theme_col
)


# =====================================================================
# 11. STANDARDIZE THEME COLUMN
# =====================================================================

final_evidence[
    "Final_BRI_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# =====================================================================
# 12. IDENTIFY EXPERT / PREVALENCE COLUMNS
# =====================================================================

if "Experts_Mentioning" not in final_evidence.columns:

    for c in final_evidence.columns:

        cl = str(c).lower()

        if (
            "expert" in cl
            and "mention" in cl
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if "Expert_Prevalence_%" not in final_evidence.columns:

    for c in final_evidence.columns:

        cl = str(c).lower()

        if (
            "prevalence" in cl
            and (
                "%" in str(c)
                or "percent" in cl
            )
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# =====================================================================
# 13. REMOVE EMPTY THEMES
# =====================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_BRI_Theme"
    ]
    .astype(str)
    .str.strip()
    != ""
].copy()


# =====================================================================
# 14. DETERMINE PARTICIPANT COUNT AUTOMATICALLY
# =====================================================================

n_participants = None


# First: participant matrix
if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()

    participant_columns = []

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.fullmatch(
            r"P\d+",
            cstr,
            flags=re.IGNORECASE
        ):

            participant_columns.append(c)


    if len(participant_columns) > 0:

        n_participants = len(
            participant_columns
        )


# Second: participant coverage
if n_participants is None and coverage_sheet is not None:

    coverage = sheets[
        coverage_sheet
    ].copy()

    for c in coverage.columns:

        cl = str(c).lower()

        if (
            "participant" in cl
            and (
                "id" in cl
                or "code" in cl
            )
        ):

            vals = (
                coverage[c]
                .dropna()
                .astype(str)
                .str.strip()
            )

            vals = vals[
                vals.str.match(
                    r"^P\d+$",
                    case=False
                )
            ]

            if len(vals) > 0:

                n_participants = (
                    vals.nunique()
                )

                break


# Third: use maximum observed expert count
if n_participants is None:

    if "Experts_Mentioning" in final_evidence.columns:

        observed = pd.to_numeric(
            final_evidence[
                "Experts_Mentioning"
            ],
            errors="coerce"
        )

        observed_max = observed.max()

        if not pd.isna(observed_max):

            # This is only a fallback.
            # It should NOT normally be used because
            # participant matrix should determine N.
            n_participants = int(observed_max)


# Final safeguard
if n_participants is None:

    raise ValueError(
        "\nCould not determine the number of participants automatically.\n"
        "Please check the Participant Matrix or Participant Coverage sheet."
    )


print(
    "\nParticipants detected:",
    n_participants
)


# =====================================================================
# 15. PREVALENCE CATEGORY
# =====================================================================

if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(prevalence_category)
    )


# =====================================================================
# 16. REMOVE DUPLICATES
# =====================================================================

final_evidence[
    "_normalized_theme"
] = (
    final_evidence[
        "Final_BRI_Theme"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)


if "Experts_Mentioning" in final_evidence.columns:

    final_evidence[
        "_expert_sort"
    ] = pd.to_numeric(
        final_evidence[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)

else:

    final_evidence[
        "_expert_sort"
    ] = 0


final_evidence = (
    final_evidence
    .sort_values(
        "_expert_sort",
        ascending=False
    )
    .drop_duplicates(
        subset=[
            "_normalized_theme"
        ],
        keep="first"
    )
    .copy()
)


final_evidence = (
    final_evidence
    .drop(
        columns=[
            "_normalized_theme",
            "_expert_sort"
        ],
        errors="ignore"
    )
    .reset_index(drop=True)
)


# =====================================================================
# 17. SELECT FINAL EVIDENCE COLUMNS
# =====================================================================

preferred_columns = [

    "Final_BRI_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]


evidence_columns = [
    c
    for c in preferred_columns
    if c in final_evidence.columns
]


final_bri_evidence = final_evidence[
    evidence_columns
].copy()


# =====================================================================
# 18. ADD RANK SAFELY
# =====================================================================

if "Rank" in final_bri_evidence.columns:

    final_bri_evidence = (
        final_bri_evidence
        .drop(
            columns=["Rank"]
        )
    )


final_bri_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_bri_evidence) + 1
    )
)


# =====================================================================
# 19. BRI CONTENT-DOMAIN MAPPING
# =====================================================================
#
# IMPORTANT:
# These are CONTENT DOMAINS derived from qualitative evidence.
#
# They are NOT automatically six measurement dimensions.
#
# The BRI qualitative analysis identified:
#
# 1. Traceability & Provenance
# 2. Interorganizational Information Sharing
# 3. Data Integrity & Verification
# 4. Transaction Transparency & History
# 5. Governance, Trust & Accountability
# 6. Blockchain-System Integration
#
# =====================================================================

def map_bri_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. TRACEABILITY & PROVENANCE
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "traceability",
            "trace",
            "provenance",
            "product history",
            "historical record continuity",
            "batch",
            "lot tracking",
            "product tracking",
            "shipment tracking",
            "movement tracking",
            "status tracking",
            "location tracking",
            "event tracing",
            "disruption tracing"

        ]
    ):

        return "Traceability & Provenance"


    # ------------------------------------------------------------
    # 2. INTERORGANIZATIONAL INFORMATION SHARING
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "shared records",
            "shared record",
            "shared information",
            "information sharing",
            "interorganizational visibility",
            "inter-organizational visibility",
            "interorganizational information",
            "information sharing across",
            "transaction visibility",
            "visibility across",
            "partner visibility"

        ]
    ):

        return "Interorganizational Information Sharing"


    # ------------------------------------------------------------
    # 3. DATA INTEGRITY & VERIFICATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "verification",
            "verify",
            "verified",
            "information consistency",
            "information integrity",
            "data integrity",
            "integrity",
            "immutability",
            "immutable",
            "data verification"

        ]
    ):

        return "Data Integrity & Verification"


    # ------------------------------------------------------------
    # 4. TRANSACTION TRANSPARENCY & HISTORY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "transaction history",
            "transaction transparency",
            "transaction transparency",
            "information trail",
            "transaction trail",
            "transparency",
            "transparent transactions",
            "transaction record"

        ]
    ):

        return "Transaction Transparency & History"


    # ------------------------------------------------------------
    # 5. GOVERNANCE, TRUST & ACCOUNTABILITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "accountability",
            "accountable",
            "interorganizational trust",
            "inter-organizational trust",
            "trust",
            "reduced disputes",
            "dispute reduction",
            "governance",
            "governance mechanism"

        ]
    ):

        return "Governance, Trust & Accountability"


    # ------------------------------------------------------------
    # 6. BLOCKCHAIN-SYSTEM INTEGRATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "integration",
            "system integration",
            "blockchain integration",
            "blockchain-system integration",
            "blockchain system integration",
            "system interoperability",
            "interoperability"

        ]
    ):

        return "Blockchain-System Integration"


    # ------------------------------------------------------------
    # REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_bri_evidence[
    "BRI_Dimension"
] = (
    final_bri_evidence[
        "Final_BRI_Theme"
    ]
    .apply(
        map_bri_dimension
    )
)


# =====================================================================
# 20. EXACT THEME OVERRIDES
# =====================================================================
#
# These override the keyword mapper for common BRI themes.
# =====================================================================

manual_mapping = {

    "end-to-end traceability":
        "Traceability & Provenance",

    "event/disruption tracing":
        "Traceability & Provenance",

    "product/batch/lot tracking":
        "Traceability & Provenance",

    "shipment/movement tracking":
        "Traceability & Provenance",

    "product/provenance history":
        "Traceability & Provenance",

    "historical record continuity":
        "Traceability & Provenance",

    "status/location tracking":
        "Traceability & Provenance",

    "shared records":
        "Interorganizational Information Sharing",

    "interorganizational visibility":
        "Interorganizational Information Sharing",

    "visibility":
        "Interorganizational Information Sharing",

    "transaction visibility":
        "Interorganizational Information Sharing",

    "shared information":
        "Interorganizational Information Sharing",

    "verification":
        "Data Integrity & Verification",

    "information consistency":
        "Data Integrity & Verification",

    "information integrity":
        "Data Integrity & Verification",

    "immutability":
        "Data Integrity & Verification",

    "transaction history":
        "Transaction Transparency & History",

    "information trail":
        "Transaction Transparency & History",

    "transparency":
        "Transaction Transparency & History",

    "accountability":
        "Governance, Trust & Accountability",

    "interorganizational trust":
        "Governance, Trust & Accountability",

    "reduced disputes":
        "Governance, Trust & Accountability",

    "integration":
        "Blockchain-System Integration"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_bri_evidence[
            "Final_BRI_Theme"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            theme.lower()
        )
    )

    final_bri_evidence.loc[
        mask,
        "BRI_Dimension"
    ] = dimension


# =====================================================================
# 21. DIMENSION SUMMARY
# =====================================================================

dimension_rows = []


for dimension, group in (
    final_bri_evidence
    .groupby(
        "BRI_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_BRI_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # IMPORTANT:
    # Do NOT sum Experts_Mentioning across themes.
    # One expert can mention multiple themes.
    #
    # Therefore, dimension-level prevalence is calculated
    # from the participant matrix where possible.


    experts = 0


    if matrix_sheet is not None:

        pm = participant_matrix.copy()

        participant_cols = []

        for c in pm.columns:

            cstr = str(c).strip()

            if re.fullmatch(
                r"P\d+",
                cstr,
                flags=re.IGNORECASE
            ):

                participant_cols.append(c)


        # Find theme column in participant matrix

        pm_theme_col = None

        for c in pm.columns:

            cl = str(c).lower()

            if "theme" in cl:

                pm_theme_col = c
                break


        if (
            pm_theme_col is not None
            and len(participant_cols) > 0
        ):

            relevant_rows = pm[
                pm[
                    pm_theme_col
                ]
                .astype(str)
                .str.strip()
                .isin(themes)
            ]


            if len(relevant_rows) > 0:

                # Any mention of any theme in this domain
                # counts the participant once.

                participant_values = (
                    relevant_rows[
                        participant_cols
                    ]
                    .apply(
                        pd.to_numeric,
                        errors="coerce"
                    )
                    .fillna(0)
                )


                participant_domain_totals = (
                    participant_values
                    .sum(axis=0)
                )


                experts = int(
                    (
                        participant_domain_totals
                        > 0
                    )
                    .sum()
                )


    # Fallback if participant matrix is unavailable
    if experts == 0:

        if "Experts_Mentioning" in group.columns:

            expert_values = pd.to_numeric(
                group[
                    "Experts_Mentioning"
                ],
                errors="coerce"
            ).dropna()

            if len(expert_values) > 0:

                experts = int(
                    expert_values.max()
                )


    prevalence = (

        experts
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "BRI_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_BRI_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# =====================================================================
# 22. SORT + RANK
# =====================================================================

if len(dimension_summary) > 0:

    dimension_summary[
        "_sort"
    ] = pd.to_numeric(
        dimension_summary[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_summary = (
        dimension_summary
        .sort_values(
            "_sort",
            ascending=False
        )
        .drop(
            columns=["_sort"]
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(
                columns=["Rank"]
            )
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# =====================================================================
# 23. DIMENSION × THEME TABLE
# =====================================================================

dimension_theme_columns = [

    "BRI_Dimension",

    "Final_BRI_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [
    c
    for c in dimension_theme_columns
    if c in final_bri_evidence.columns
]


dimension_themes = final_bri_evidence[
    dimension_theme_columns
].copy()


if "Experts_Mentioning" in dimension_themes.columns:

    dimension_themes[
        "_sort"
    ] = pd.to_numeric(
        dimension_themes[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_themes = (
        dimension_themes
        .sort_values(
            [
                "BRI_Dimension",
                "_sort"
            ],
            ascending=[
                True,
                False
            ]
        )
        .drop(
            columns=["_sort"]
        )
        .reset_index(drop=True)
    )


# =====================================================================
# 24. REVIEW-REQUIRED THEMES
# =====================================================================

review_rows = final_bri_evidence[
    final_bri_evidence[
        "BRI_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(review_rows) > 0:

    review_columns = [

        "Final_BRI_Theme",

        "Experts_Mentioning",

        "Expert_Prevalence_%",

        "BRI_Dimension"

    ]


    review_columns = [
        c
        for c in review_columns
        if c in review_rows.columns
    ]


    review_required = review_rows[
        review_columns
    ].copy()


    review_required.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )

else:

    review_required = pd.DataFrame({

        "Status": [
            "PASS — All BRI themes mapped to a content domain."
        ]

    })


# =====================================================================
# 25. QUALITY CHECKS
# =====================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final BRI evidence available",

    "Result":
        "PASS"
        if len(final_bri_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_bri_evidence)} normalized BRI themes"

})


quality_rows.append({

    "Quality_Check":
        "Participant count",

    "Result":
        n_participants,

    "Details":
        "Detected automatically from participant matrix / coverage"

})


quality_rows.append({

    "Quality_Check":
        "BRI content domains generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative content domains"

})


review_count = len(review_rows)


quality_rows.append({

    "Quality_Check":
        "Themes requiring manual review",

    "Result":
        review_count,

    "Details":
        "Themes assigned Review Required"

})


duplicate_count = int(
    final_bri_evidence[
        "Final_BRI_Theme"
    ]
    .astype(str)
    .str.lower()
    .duplicated()
    .sum()
)


quality_rows.append({

    "Quality_Check":
        "Remaining duplicate themes",

    "Result":
        duplicate_count,

    "Details":
        "Should normally be zero"

})


# Check all six expected BRI domains
expected_domains = {

    "Traceability & Provenance",

    "Interorganizational Information Sharing",

    "Data Integrity & Verification",

    "Transaction Transparency & History",

    "Governance, Trust & Accountability",

    "Blockchain-System Integration"

}


actual_domains = set(
    dimension_summary[
        "BRI_Dimension"
    ]
    .dropna()
    .astype(str)
)


missing_domains = (
    expected_domains
    - actual_domains
)


quality_rows.append({

    "Quality_Check":
        "Expected BRI content domains present",

    "Result":
        "PASS"
        if len(missing_domains) == 0
        else "CHECK",

    "Details":
        (
            "All six expected domains present"
            if len(missing_domains) == 0
            else
            "Missing: "
            + "; ".join(
                sorted(missing_domains)
            )
        )

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# =====================================================================
# 26. COPY ORIGINAL SUPPORTING SHEETS
# =====================================================================

def get_sheet_or_empty(sheet_name):

    if sheet_name is not None:

        return sheets[
            sheet_name
        ].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

coding_audit = get_sheet_or_empty(
    audit_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# =====================================================================
# 27. OUTPUT FILE
# =====================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"BRI_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


# =====================================================================
# 28. WRITE OUTPUT
# =====================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )


    if matrix_sheet is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_bri_evidence.to_excel(
        writer,
        sheet_name="11_Final_BRI_Evidence",
        index=False
    )

    dimension_summary.to_excel(
        writer,
        sheet_name="12_BRI_Dimension_Summary",
        index=False
    )

    dimension_themes.to_excel(
        writer,
        sheet_name="13_BRI_Dimension_Themes",
        index=False
    )

    review_required.to_excel(
        writer,
        sheet_name="14_Review_Required",
        index=False
    )

    quality_checks_new.to_excel(
        writer,
        sheet_name="15_Quality_Checks",
        index=False
    )


# =====================================================================
# 29. FINAL REPORT
# =====================================================================

print("\n")
print("=" * 90)
print("BRI PROCESS COMPLETED SUCCESSFULLY")
print("=" * 90)

print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)

print("\nNormalized BRI themes:")
print(
    len(final_bri_evidence)
)

print("\nBRI content domains:")
print(
    len(dimension_summary)
)


print("\n")
print("-" * 90)
print("BRI CONTENT-DOMAIN SUMMARY")
print("-" * 90)

if len(dimension_summary) > 0:

    print(
        dimension_summary.to_string(
            index=False
        )
    )

else:

    print(
        "No content domains generated."
    )


print("\n")
print("-" * 90)
print("QUALITY CHECKS")
print("-" * 90)

print(
    quality_checks_new.to_string(
        index=False
    )
)


print("\n")
print("=" * 90)
print("OUTPUT SHEETS CREATED")
print("=" * 90)

print("""
11_Final_BRI_Evidence
    → Cleaned final BRI qualitative themes

12_BRI_Dimension_Summary
    → Six BRI content domains

13_BRI_Dimension_Themes
    → Theme-to-domain mapping

14_Review_Required
    → Themes requiring manual inspection

15_Quality_Checks
    → Automated quality-control results
""")


print("\n")
print("=" * 90)
print("IMPORTANT METHODOLOGICAL NOTE")
print("=" * 90)

print("""
The BRI domains generated here are CONTENT DOMAINS derived
from expert qualitative evidence.

They are NOT automatically treated as six reflective
measurement dimensions.

Their purpose is to support systematic questionnaire-item
development by ensuring that the qualitative evidence
representing BRI is adequately covered.

The next stage is candidate BRI questionnaire-item
development, followed by expert content validation and
then quantitative measurement validation using PLS-SEM.
""")

BRI FINAL QUALITATIVE EVIDENCE + CONTENT-DOMAIN ANALYSIS

Input file found:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_analytics_AI\Papers\BRI_FINAL_QUALITATIVE_ANALYSIS_20260825_151306.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_BRI_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Final evidence sheet: 11_Final_BRI_Evidence

Final evidence columns:
1. Final_BRI_Theme
2. Number_of_Raw_Themes
3. Experts_Mentioning
4. Expert_Prevalence_%
5. Raw_Themes_Kept
6. Raw_Themes_Merged
7. Prevalence_Category

Theme column detected: Final_BRI_Theme

Participants detected: 26


BRI PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Engineering_data_an